# Llama 3.1 8B Bradford Two-Stage DAPT + Grounded SFT

This Colab notebook ports the two-stage Bradford training flow from Qwen to `meta-llama/Llama-3.1-8B-Instruct`.

Notes before you run it:
1. You must have accepted the Llama 3.1 model license on Hugging Face for the base model to download successfully.
2. This variant uses the tokenizer's native chat template instead of Qwen-specific `<|im_start|>` formatting.
3. It keeps the metric-fix evaluation flow so `eval_grounded_score` is visible to early stopping and best-checkpoint selection.

Pipeline:
1. Light domain adaptation (DAPT) on cleaned Born in Bradford PDF text
2. Retention mixing during DAPT using rendered grounded QA conversations
3. Grounded supervised fine-tuning (SFT) on your JSONL train/dev files with assistant-only loss
4. Checkpoint selection for Stage 2 using a grounded QA metric, not just LM loss
5. Push adapter, merged model, and optional 4-bit merged model to Hugging Face

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob
import os
import pathlib
import site
import subprocess
import sys

HF_DEP_SENTINEL = pathlib.Path('/content/.bib_hf_deps_transformers4513_bnb0492_cuda13_20260521')
if not pathlib.Path('/content').exists():
    HF_DEP_SENTINEL = pathlib.Path('.bib_hf_deps_transformers4513_bnb0492_cuda13_20260521')

HF_PACKAGES = [
    'transformers==4.51.3',
    'tokenizers==0.21.1',
    'huggingface_hub==0.30.2',
    'accelerate==1.6.0',
    'peft==0.15.2',
    'datasets>=2.20.0,<4.0.0',
    'bitsandbytes==0.49.2',
    'nvidia-nvjitlink==13.0.88',
    'nvidia-cuda-runtime==13.0.96',
    'sentencepiece>=0.2.0',
    'pymupdf>=1.24.0',
    'pypdf>=4.2.0',
]

# These packages are not needed for text-only Llama fine-tuning. In Colab they can be
# preinstalled against a different torch build and make transformers fail while importing
# optional vision modules, e.g. "operator torchvision::nms does not exist".
TEXT_ONLY_CONFLICT_PACKAGES = ['torchvision', 'torchao']


def package_is_installed(package_name: str) -> bool:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'show', package_name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    return result.returncode == 0


def configure_cuda_library_path() -> None:
    site_roots = []
    try:
        site_roots.extend(site.getsitepackages())
    except Exception:
        pass
    try:
        site_roots.append(site.getusersitepackages())
    except Exception:
        pass

    candidate_paths = []
    for root in site_roots:
        candidate_paths.extend(glob.glob(os.path.join(root, 'nvidia', '*', 'lib')))
    candidate_paths.extend(glob.glob('/usr/local/cuda*/lib64'))

    existing_paths = []
    for path in candidate_paths:
        if os.path.isdir(path) and path not in existing_paths:
            existing_paths.append(path)

    current_paths = [path for path in os.environ.get('LD_LIBRARY_PATH', '').split(':') if path]
    merged_paths = existing_paths + [path for path in current_paths if path not in existing_paths]
    if merged_paths:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(merged_paths)
        print('CUDA library paths configured for this Python process:')
        for path in existing_paths[:12]:
            print('  ', path)


needs_restart = False

if not HF_DEP_SENTINEL.exists():
    print('Installing a coherent Hugging Face stack with CUDA-13-capable bitsandbytes.')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--force-reinstall',
        '--no-cache-dir',
        *HF_PACKAGES,
    ])
    HF_DEP_SENTINEL.write_text('installed\n', encoding='utf-8')
    needs_restart = True
else:
    print('Hugging Face dependency stack already installed for this runtime image.')

removed_conflicts = []
for package_name in TEXT_ONLY_CONFLICT_PACKAGES:
    if package_is_installed(package_name):
        print(f'Removing optional text-only conflict package: {package_name}')
        subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', package_name])
        removed_conflicts.append(package_name)
        needs_restart = True

configure_cuda_library_path()

if needs_restart:
    if removed_conflicts:
        print('Removed optional packages:', ', '.join(removed_conflicts))
    print('Dependency repair complete. Restarting the runtime now; after it reconnects, rerun the notebook from this cell.')
    os.kill(os.getpid(), 9)

print('Dependency setup complete; continue to the imports cell.')


In [ ]:
import glob
import importlib.metadata as importlib_metadata
import json
import math
import os
import random
import re
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Optional

os.environ.setdefault('TRANSFORMERS_NO_TORCHVISION', '1')

import torch
from datasets import Dataset
from huggingface_hub import login
from peft import AutoPeftModelForCausalLM, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    default_data_collator,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

try:
    from transformers.models.llama.configuration_llama import LlamaConfig
except ImportError as exc:
    raise RuntimeError(
        'Transformers is installed inconsistently. Rerun the dependency setup cell, let it restart the runtime, '
        'then rerun the notebook from the imports cell.'
    ) from exc

print('Transformers:', importlib_metadata.version('transformers'))
print('Tokenizers:', importlib_metadata.version('tokenizers'))
print('PEFT:', importlib_metadata.version('peft'))
print('BitsAndBytes:', importlib_metadata.version('bitsandbytes'))
print('Transformers Llama config import OK.')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import userdata

HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged in from HF_TOKEN env var')
else:
    hf_token_secret = userdata.get('HF_TOKEN')
    if hf_token_secret:
        login(token=hf_token_secret)
        print('Logged in from HF_TOKEN secret var')
    else:
        print('HF_TOKEN not found. Run login() manually.')

In [ ]:
@dataclass
class Config:
    base_model: str = 'meta-llama/Llama-3.1-8B-Instruct'

    pdf_glob: str = '/content/drive/MyDrive/eval/papers/*.pdf'
    sft_train_path: str = '/content/drive/MyDrive/triples/train_dev_val/sft_train.jsonl'
    sft_dev_path: str = '/content/drive/MyDrive/triples/train_dev_val/sft_dev.jsonl'

    stage1_output_dir: str = './llama31_bradford_stage1_dapt'
    stage2_output_dir: str = './llama31_bradford_stage2_sft'

    push_repo_id_adapter: str = 'dizza01/llama-3.1-8b-bib-grounded-sft-lora'
    push_repo_id_merged: str = 'dizza01/llama-3.1-8b-bib-grounded-sft-merged'
    push_repo_id_merged_4bit: str = 'dizza01/llama-3.1-8b-bib-grounded-sft-merged-4bit'

    max_chars_per_doc: int = 500000
    stage1_test_size: float = 0.15

    stage1_chunk_target_tokens: int = 768
    stage1_chunk_overlap_tokens: int = 64
    stage1_min_chunk_tokens: int = 128

    max_seq_len: int = 4096
    generation_max_new_tokens: int = 64
    generation_repetition_penalty: float = 1.0
    generation_no_repeat_ngram_size: int = 0
    numeric_sensitive_oversample_copies: int = 2
    targeted_stage2_augmentation: bool = True
    targeted_stage2_augmentation_max_auto_examples: int = 120
    targeted_stage2_augmentation_max_context_chars: int = 3000
    targeted_stage2_augmentation_include_drills: bool = True
    model_context_limit: int = 131072
    production_system_prompt: str = (
        'You are an expert research assistant for the Born in Bradford (BiB) longitudinal cohort study. '
        'You help researchers understand the dataset, find relevant variables, plan analyses, '
        'and understand what has already been published. Use only the provided context. '
        'If the context contains the requested value, relationship, or finding, answer it directly; do not say it is absent. '
        'Only say the answer is not present after checking the paper excerpts, labels, headings, rows, and adjacent context for an exact match. '
        'For table-like or multi-finding context, choose the answer whose row label, column label, outcome, subgroup, timepoint, model, and measurement all match the question; ignore nearby rows where any anchor differs. '
        'Preserve all numbers, ranges, units, p-values, confidence intervals, dates, distances, '
        'comparison groups, and categories exactly as written in the context. '
        'Do not add page numbers, figure numbers, table positions, p-values, methods, ranges, '
        'or qualifiers unless they appear in the provided context. '
        'If the context contains a direct answer, restate it concisely. '
        'Follow the response rule after the question, because it identifies the exact extraction style needed. '
        'Answer only the requested value, relationship, or finding; do not include adjacent table values, '
        'extra categories, or explanatory detail unless the question asks for them.'
    )
    generation_eval_examples: int = 0
    sanity_check_examples: int = 4
    use_evaluation_sanity_records: bool = True
    evaluation_sanity_examples: int = 10
    run_hf_api_base_sanity_check: bool = True
    hf_api_base_model: str = 'meta-llama/Llama-3.1-8B-Instruct'
    hf_api_sanity_timeout_seconds: int = 120

    run_stage1: bool = False
    stage1_retention_fraction: float = 0.30

    stage1_epochs: float = 0.5
    stage1_learning_rate: float = 2e-6
    stage1_train_batch_size: int = 2
    stage1_eval_batch_size: int = 2
    stage1_grad_accum_steps: int = 8

    stage2_epochs: float = 1.0
    stage2_learning_rate: float = 1e-6
    stage2_train_batch_size: int = 1
    stage2_eval_batch_size: int = 1
    stage2_grad_accum_steps: int = 16

    gradient_checkpointing: bool = True
    gradient_checkpointing_use_reentrant: bool = False

    warmup_ratio: float = 0.03
    weight_decay: float = 0.01

    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    use_wandb: bool = False
    wandb_project: str = 'bib-llama-grounded-sft'
    wandb_run_name: str = 'llama31-8b-bib-two-stage'

    early_stopping_patience: int = 3
    use_4bit: bool = True
    allow_4bit_fallback: bool = True
    push_to_hub: bool = True
    export_4bit: bool = False

cfg = Config()
cfg


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Pad token:', tokenizer.pad_token)
print('EOS token:', tokenizer.eos_token)


In [ ]:
def load_jsonl(path: str) -> List[Dict]:
    rows = []
    with open(path, 'r', encoding='utf-8') as file_obj:
        for line in file_obj:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def normalize_text(text: str) -> str:
    text = text.replace('\x00', ' ')
    text = text.replace('\r', '\n')
    text = re.sub(r'-\n(?=\w)', '', text)
    text = re.sub(r'\t+', ' ', text)
    text = re.sub(r'\u00a0', ' ', text)
    text = re.sub(r' +', ' ', text)
    return text

def sanitize_tokenizer_config_dir(save_dir: str) -> None:
    tok_cfg_path = os.path.join(save_dir, 'tokenizer_config.json')
    if not os.path.exists(tok_cfg_path):
        return

    with open(tok_cfg_path, 'r', encoding='utf-8') as file_obj:
        tok_cfg = json.load(file_obj)

    extra = tok_cfg.get('extra_special_tokens')
    if isinstance(extra, list):
        tok_cfg['extra_special_tokens'] = {
            name: token
            for name, token in zip(
                [
                    'im_start',
                    'im_end',
                    'object_ref_start',
                    'object_ref_end',
                    'box_start',
                    'box_end',
                    'quad_start',
                    'quad_end',
                    'vision_start',
                    'vision_end',
                    'vision_pad',
                    'image_pad',
                    'video_pad',
                ],
                [token for token in extra if isinstance(token, str)],
            )
        }
        with open(tok_cfg_path, 'w', encoding='utf-8') as file_obj:
            json.dump(tok_cfg, file_obj, ensure_ascii=False, indent=2)
        print('[INFO] Sanitized tokenizer_config.extra_special_tokens from list to dict.')

def remove_reference_tail(text: str) -> str:
    patterns = [
        r'\nreferences\n',
        r'\nbibliography\n',
        r'\nacknowledg(?:e)?ments\n',
        r'\nfunding\n',
    ]
    lowered = text.lower()
    cut_positions = [lowered.find(pattern.replace('\\n', '\n')) for pattern in patterns]
    cut_positions = [pos for pos in cut_positions if pos > 0]
    if cut_positions:
        text = text[: min(cut_positions)]
    return text

def collapse_repeated_lines(text: str) -> str:
    raw_lines = [line.strip() for line in text.split('\n')]
    raw_lines = [line for line in raw_lines if line]
    counts = Counter(raw_lines)
    cleaned_lines = []
    for line in raw_lines:
        if re.fullmatch(r'\d+', line):
            continue
        if counts[line] >= 4 and len(line) < 120:
            continue
        cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)

def clean_pdf_text(text: str) -> str:
    text = normalize_text(text)
    text = collapse_repeated_lines(text)
    text = remove_reference_tail(text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_pdf_text(path: str) -> str:
    text_parts: List[str] = []
    try:
        import fitz

        doc = fitz.open(path)
        for page in doc:
            text_parts.append(page.get_text('text'))
        doc.close()
    except Exception:
        from pypdf import PdfReader

        reader = PdfReader(path)
        for page in reader.pages:
            text_parts.append(page.extract_text() or '')
    return clean_pdf_text('\n'.join(text_parts))

def paragraph_pack_chunks(text: str, chunk_target_tokens: int, overlap_tokens: int, min_chunk_tokens: int) -> List[str]:
    paragraphs = [paragraph.strip() for paragraph in re.split(r'\n\s*\n', text) if paragraph.strip()]
    if not paragraphs:
        return []

    chunks = []
    current_parts = []
    current_ids = []

    for paragraph in paragraphs:
        paragraph_ids = tokenizer(paragraph, add_special_tokens=False)['input_ids']
        if len(paragraph_ids) < 8:
            continue

        if current_ids and len(current_ids) + len(paragraph_ids) > chunk_target_tokens:
            if len(current_ids) >= min_chunk_tokens:
                chunks.append('\n\n'.join(current_parts))

            overlap_ids = current_ids[-overlap_tokens:] if overlap_tokens > 0 else []
            overlap_text = tokenizer.decode(overlap_ids, skip_special_tokens=True).strip()
            current_parts = [overlap_text] if overlap_text else []
            current_ids = overlap_ids[:] if overlap_ids else []

        current_parts.append(paragraph)
        current_ids.extend(paragraph_ids)

    if len(current_ids) >= min_chunk_tokens and current_parts:
        chunks.append('\n\n'.join(current_parts))

    return [chunk.strip() for chunk in chunks if chunk.strip()]

def render_chat_transcript(messages: List[Dict[str, str]]) -> str:
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

def render_prompt_and_answer(messages: List[Dict[str, str]]):
    if not messages or messages[-1]['role'] != 'assistant':
        raise ValueError('Expected final message to be assistant.')

    prompt_messages = messages[:-1]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return prompt_text, full_text

def extract_context_from_messages(messages: List[Dict[str, str]]) -> str:
    user_messages = [message['content'] for message in messages if message['role'] == 'user']
    if not user_messages:
        return ''
    content = user_messages[-1]
    if 'Question:' in content:
        return content.split('Question:', 1)[0].replace('Context:', '').replace('Instruction:', '').strip()
    if 'Researcher question:' in content:
        return content.split('Researcher question:', 1)[0].replace('Retrieved knowledge base context:', '').replace('Instruction:', '').strip(' -\n')
    return content.strip()


def extract_question_from_messages(messages: List[Dict[str, str]]) -> str:
    user_messages = [message['content'] for message in messages if message['role'] == 'user']
    if not user_messages:
        return ''
    content = user_messages[-1]
    if 'Researcher question:' in content:
        return content.rsplit('Researcher question:', 1)[1].strip()
    if 'Question:' in content:
        return content.rsplit('Question:', 1)[1].strip()
    return ''



NUMERIC_RESPONSE_RULE_RE = re.compile(
    r'\b(?:odds ratio|or|percentage|percent|proportion|count|number|value|values|cutoff|cut-off|coefficient|confidence interval|ci|p-value|sample size|mean|median|bmi|z-score|quintile|buffer|years?|months?|dates?|coded|code|classified|units?)\b',
    flags=re.IGNORECASE,
)

RELATIONSHIP_RESPONSE_RULE_RE = re.compile(
    r'\b(?:relationship|association|associated|correlation|linked|impact|effect|finding|found according to|according to the study)\b',
    flags=re.IGNORECASE,
)

DIRECT_NUMERIC_REQUEST_RE = re.compile(
    r'\b(?:odds ratio|\bor\b|percentage|percent|proportion|count|number|value|values|cutoff|cut-off|coefficient|confidence interval|ci|p-value|sample size|mean|median|coded|code|classified|units?)\b',
    flags=re.IGNORECASE,
)


PUBLISHED_PAPER_QUESTION_RE = re.compile(
    r'\b(?:paper|study|article|published|according to the study|according to the paper|systematic review|meta-analysis)\b',
    flags=re.IGNORECASE,
)

VARIABLE_METADATA_QUESTION_RE = re.compile(
    r'\b(?:variable|table|dataset|data table|field|column|label|coded|code|non-missing|records|entities|rows)\b',
    flags=re.IGNORECASE,
)


def build_response_rule(question: str, context: str = '') -> str:
    question_text = str(question or '')
    context_text = str(context or '')
    asks_for_relationship = bool(RELATIONSHIP_RESPONSE_RULE_RE.search(question_text))
    asks_for_direct_number = bool(DIRECT_NUMERIC_REQUEST_RE.search(question_text))

    context_has_papers = '## Relevant Published Papers' in context_text or 'Source: full-text PDF' in context_text
    context_has_metadata = '## Relevant Variables' in context_text or '## Relevant Tables' in context_text
    asks_paper_style = bool(PUBLISHED_PAPER_QUESTION_RE.search(question_text)) and not bool(VARIABLE_METADATA_QUESTION_RE.search(question_text))

    source_priority_rule = ''
    if context_has_papers and context_has_metadata and asks_paper_style:
        source_priority_rule = (
            'For published-paper questions, answer from the Relevant Published Papers/full-text PDF text. '
            'Ignore Relevant Variables and Relevant Tables metadata unless the question explicitly asks for variables, tables, columns, labels, coding, or non-missing records. '
        )

    presence_rule = (
        'Positive evidence has priority over abstention. '
        'If any sentence, table row, header-expanded row, or labelled excerpt contains the requested value, relationship, or finding, answer it directly; do not abstain. '
        'Before abstaining, check repeated headers, row labels, timepoints, outcomes, comparison groups, paper excerpts, and adjacent context for an exact match. '
        'Do not answer "The provided context does not contain the requested information" when a matching answer is present under the requested label, source, study, or outcome. '
    )

    abstention_rule = (
        'Only if the exact requested value, comparison, timepoint, work package, criterion, source, funding body, or finding is not present after that check, '
        'answer exactly: "The provided context does not contain the requested information." and stop. '
        'Do not add partial related information after abstaining. '
        'Do not infer, calculate, or substitute a nearby value from another outcome, age, time period, work package, exposure, table row, cited study, or metadata section. '
    )

    anchor_match_rule = (
        'Wrong-neighbor rule: identify the required anchors in the question, including outcome, measure, subgroup, cohort, model, timepoint, emotion, frequency, exposure, and comparison groups. '
        'Use only evidence where all requested anchors match. '
        'Reject nearby rows or sentences if they are about a different outcome, measure, group, model, timepoint, emotion, frequency, exposure, or comparison, even when their numbers look plausible. '
        'If a matching row contains multiple requested groups, return all requested values from that row; do not omit one group. '
    )

    if asks_for_relationship and not asks_for_direct_number:
        return (
            'Response rule: ' + source_priority_rule + presence_rule + anchor_match_rule +
            'State the requested relationship or finding in words using only the context. '
            'Match the requested exposure, outcome, population, cited study, and timepoint exactly. '
            'If the exact relationship statement is present, restate it rather than abstaining, even if the context also contains adjacent unrelated findings. '
            'Keep numbers that identify requested groups or timepoints, but do not add model coefficients, '
            'confidence intervals, p-values, table/figure/page references, variable names, or adjacent numeric details unless the question asks for them. '
            + abstention_rule
        )

    is_numeric_like = bool(NUMERIC_RESPONSE_RULE_RE.search(question_text) or re.search(r'\d', question_text))
    if is_numeric_like:
        return (
            'Response rule: ' + source_priority_rule + presence_rule + anchor_match_rule +
            'Extract only the value(s) requested by the question. '
            'Use nearby headers and labels to choose the matching column, outcome, timepoint, comparison group, or row; do not default to the first number in a row. '
            'Prefer the value whose row and column labels both match the question. If that matching value is present, return it rather than abstaining. '
            'Match the requested age, time period, subgroup, exposure, outcome, and direction exactly before copying a number. '
            'Copy requested numbers, ranges, units, comparison groups, and categories exactly from the context. '
            'If the question asks for multiple groups or contrasts two reported values, include each requested group. '
            'Do not include adjacent rows, unrelated categories, p-values, confidence intervals, variable names, '
            'table/figure/page references, or explanations unless the question asks for them. '
            + abstention_rule
        )
    return (
        'Response rule: ' + source_priority_rule + presence_rule + anchor_match_rule +
        'Answer directly and concisely using only the context. '
        'Match the requested paper, study aim, source, criterion, work package, population, exposure, and outcome exactly. '
        'Do not add table, figure, page, row, variable, or section references unless the question asks for them. '
        + abstention_rule
    )

def coerce_text_value(value) -> str:
    if value is None:
        return ''
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, (int, float, bool)):
        return str(value).strip()
    if isinstance(value, list):
        parts = []
        for item in value:
            text = coerce_text_value(item)
            if text:
                parts.append(text)
        return '\n'.join(parts).strip()
    if isinstance(value, dict):
        for key in ('content', 'text', 'answer', 'question', 'context', 'source_chunk', 'evidence'):
            if key in value:
                text = coerce_text_value(value.get(key))
                if text:
                    return text
        return json.dumps(value, ensure_ascii=False)
    return str(value).strip()


def first_nonempty_field(row: Dict, field_names: List[str]) -> str:
    for field_name in field_names:
        if field_name in row:
            text = coerce_text_value(row.get(field_name))
            if text:
                return text
    return ''


def row_answer_text(row: Dict, source_messages: Optional[List[Dict[str, str]]] = None) -> str:
    answer = first_nonempty_field(
        row,
        ['answer', 'target', 'gold_answer', 'reference_answer', 'completion', 'response', 'output', 'label'],
    )
    if answer:
        return answer
    source_messages = source_messages if source_messages is not None else row.get('messages') or []
    if isinstance(source_messages, list) and source_messages:
        for message in reversed(source_messages):
            if isinstance(message, dict) and message.get('role') == 'assistant':
                answer = coerce_text_value(message.get('content'))
                if answer:
                    return answer
    return ''


def row_question_text(row: Dict, source_messages: Optional[List[Dict[str, str]]] = None) -> str:
    question = first_nonempty_field(
        row,
        ['question', 'query', 'researcher_question', 'prompt_question', 'instruction_question'],
    )
    if question:
        return question
    source_messages = source_messages if source_messages is not None else row.get('messages') or []
    question = extract_question_from_messages(source_messages) if isinstance(source_messages, list) else ''
    if question:
        return question
    prompt_text = first_nonempty_field(row, ['prompt', 'input', 'text'])
    if 'Question:' in prompt_text:
        return prompt_text.rsplit('Question:', 1)[1].split('\n', 1)[0].strip()
    if 'Researcher question:' in prompt_text:
        return prompt_text.rsplit('Researcher question:', 1)[1].split('\n', 1)[0].strip()
    return ''


def row_context_text(row: Dict, source_messages: Optional[List[Dict[str, str]]] = None) -> str:
    context = first_nonempty_field(
        row,
        [
            'source_chunk', 'context', 'retrieved_context', 'retrieved_knowledge', 'source_text',
            'chunk', 'chunk_text', 'passage', 'document', 'documents', 'evidence_context',
        ],
    )
    if context:
        return context
    source_messages = source_messages if source_messages is not None else row.get('messages') or []
    context = extract_context_from_messages(source_messages) if isinstance(source_messages, list) else ''
    if context:
        return context
    # Last-resort fallback: for extraction SFT, evidence is still better context than dropping the row.
    return first_nonempty_field(row, ['evidence', 'supporting_evidence', 'citation', 'quote'])

def build_production_messages(row: Dict) -> Optional[List[Dict[str, str]]]:
    source_messages = row.get('messages') or []
    if not isinstance(source_messages, list):
        source_messages = []

    answer = row_answer_text(row, source_messages)
    question = row_question_text(row, source_messages)
    context = row_context_text(row, source_messages)
    system_prompt = first_nonempty_field(row, ['system', 'system_prompt']) or cfg.production_system_prompt

    if not answer or not question or not context:
        return None

    response_rule = coerce_text_value(row.get('response_rule')) or build_response_rule(question, context)
    user_content = (
        f"Instruction: {str(system_prompt).strip()}\n\n"
        f"Context:\n{str(context).strip()}\n\n"
        f"Question: {str(question).strip()}\n\n"
        f"{response_rule}"
    )
    return [
        {'role': 'user', 'content': user_content},
        {'role': 'assistant', 'content': str(answer).strip()},
    ]

def normalized_answer(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def token_f1(prediction: str, target: str) -> float:
    pred_tokens = normalized_answer(prediction).split()
    target_tokens = normalized_answer(target).split()
    if not pred_tokens and not target_tokens:
        return 1.0
    if not pred_tokens or not target_tokens:
        return 0.0
    pred_counter = Counter(pred_tokens)
    target_counter = Counter(target_tokens)
    overlap = sum((pred_counter & target_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(target_tokens)
    return 2 * precision * recall / (precision + recall)

NUMERIC_PATTERN = re.compile(
    r'(?<![A-Za-z0-9])(?:p\s*[<=>]\s*0?[\.·]\d+|[<>≤≥~]?\s*-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:[\.·]\d+)?(?:\s*[–-]\s*-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:[\.·]\d+)?)?\s*%?)(?![A-Za-z0-9])',
    flags=re.IGNORECASE,
)

UNSUPPORTED_REFERENCE_PATTERN = re.compile(
    r'\b(?:table|row|page|fig(?:ure)?s?|section)\s*[0-9ivxlcdm]+\b',
    flags=re.IGNORECASE,
)

NUMBER_WORD_UNITS = {
    'zero': 0,
    'one': 1,
    'two': 2,
    'three': 3,
    'four': 4,
    'five': 5,
    'six': 6,
    'seven': 7,
    'eight': 8,
    'nine': 9,
    'ten': 10,
    'eleven': 11,
    'twelve': 12,
    'thirteen': 13,
    'fourteen': 14,
    'fifteen': 15,
    'sixteen': 16,
    'seventeen': 17,
    'eighteen': 18,
    'nineteen': 19,
}
NUMBER_WORD_TENS = {
    'twenty': 20,
    'thirty': 30,
    'forty': 40,
    'fifty': 50,
    'sixty': 60,
    'seventy': 70,
    'eighty': 80,
    'ninety': 90,
}
NUMBER_WORD_PATTERN = re.compile(
    r'\b((?:zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand)(?:[\s-]+(?:zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand))*)(?:\s+point\s+((?:zero|one|two|three|four|five|six|seven|eight|nine)(?:[\s-]+(?:zero|one|two|three|four|five|six|seven|eight|nine))*))?(\s*(?:percent|per cent))?\b',
    flags=re.IGNORECASE,
)


def normalize_numeric_token(token: str) -> str:
    token = token.lower().strip()
    token = token.replace('−', '-').replace('–', '-').replace('—', '-')
    token = token.replace('≤', '<=').replace('≥', '>=')
    token = token.replace('·', '.').replace('∙', '.')
    token = token.replace(',', '')
    token = re.sub(r'\s+', '', token)
    return token


def number_words_to_int(words: str) -> Optional[int]:
    total = 0
    current = 0
    seen_number = False
    for word in re.split(r'[\s-]+', (words or '').lower().strip()):
        if not word:
            continue
        if word in NUMBER_WORD_UNITS:
            current += NUMBER_WORD_UNITS[word]
            seen_number = True
        elif word in NUMBER_WORD_TENS:
            current += NUMBER_WORD_TENS[word]
            seen_number = True
        elif word == 'hundred':
            current = max(1, current) * 100
            seen_number = True
        elif word == 'thousand':
            total += max(1, current) * 1000
            current = 0
            seen_number = True
        else:
            return None
    if not seen_number:
        return None
    return total + current


def extract_word_numeric_tokens(text: str) -> List[str]:
    tokens = []
    for match in NUMBER_WORD_PATTERN.finditer(text or ''):
        integer_words = match.group(1)
        decimal_words = match.group(2)
        percent_words = match.group(3)
        # Only convert written numbers when they carry numeric intent. This avoids turning
        # ordinary prose such as "one exception" into a numeric answer requirement.
        if not decimal_words and not percent_words:
            continue
        integer_value = number_words_to_int(integer_words)
        if integer_value is None:
            continue
        if decimal_words:
            decimal_digits = []
            for word in re.split(r'[\s-]+', decimal_words.lower().strip()):
                if word not in NUMBER_WORD_UNITS or NUMBER_WORD_UNITS[word] > 9:
                    decimal_digits = []
                    break
                decimal_digits.append(str(NUMBER_WORD_UNITS[word]))
            if not decimal_digits:
                continue
            token = f"{integer_value}.{''.join(decimal_digits)}"
        else:
            token = str(integer_value)
        if percent_words:
            token = f'{token}%'
        tokens.append(normalize_numeric_token(token))
    return tokens


def extract_numeric_tokens(text: str) -> List[str]:
    regex_tokens = [normalize_numeric_token(match.group(0)) for match in NUMERIC_PATTERN.finditer(text or '')]
    word_tokens = extract_word_numeric_tokens(text or '')
    return sorted(set(token for token in regex_tokens + word_tokens if token))


def required_numeric_tokens(target: str, question: str = '') -> List[str]:
    target_numbers = set(extract_numeric_tokens(target))
    question_numbers = set(extract_numeric_tokens(question))
    required = target_numbers - question_numbers
    return sorted(required)


def unexpected_numeric_tokens(prediction: str, target: str, question: str = '') -> List[str]:
    prediction_numbers = set(extract_numeric_tokens(prediction))
    allowed_numbers = set(extract_numeric_tokens(target)) | set(extract_numeric_tokens(question))
    return sorted(prediction_numbers - allowed_numbers)


def numeric_preservation_match(prediction: str, target: str, question: str = '') -> Optional[bool]:
    required_numbers = required_numeric_tokens(target, question)
    if not required_numbers:
        return None
    prediction_numbers = set(extract_numeric_tokens(prediction))
    return all(number in prediction_numbers for number in required_numbers)


def numeric_strict_match(prediction: str, target: str, question: str = '') -> Optional[bool]:
    required_numbers = required_numeric_tokens(target, question)
    extra_numbers = unexpected_numeric_tokens(prediction, target, question)
    if not required_numbers and not extra_numbers:
        return None
    preservation = True
    if required_numbers:
        prediction_numbers = set(extract_numeric_tokens(prediction))
        preservation = all(number in prediction_numbers for number in required_numbers)
    return preservation and not extra_numbers


def numeric_context_support_diagnostic(row: Dict) -> Optional[Dict[str, object]]:
    question = str(row.get('question', '') or '')
    answer = str(row.get('answer', '') or '')
    context = str(row.get('context', '') or row.get('source_chunk', '') or row.get('retrieved_context', '') or '')
    required = required_numeric_tokens(answer, question)
    if not required:
        return None
    context_numbers = set(extract_numeric_tokens(context))
    present = sorted(number for number in required if number in context_numbers)
    missing = sorted(number for number in required if number not in context_numbers)
    return {'required': required, 'present': present, 'missing': missing, 'all_present': not missing}


def unsupported_reference_hits(prediction: str, allowed_text: str = '') -> List[str]:
    allowed = (allowed_text or '').lower()
    hits = []
    for match in UNSUPPORTED_REFERENCE_PATTERN.finditer(prediction or ''):
        phrase = match.group(0).lower()
        if phrase not in allowed:
            hits.append(phrase)
    return sorted(set(hits))


def has_unsupported_reference(prediction: str, allowed_text: str = '') -> bool:
    return bool(unsupported_reference_hits(prediction, allowed_text))

def is_abstain_answer(text: str) -> bool:
    lowered = normalized_answer(text)
    abstain_markers = [
        'not present in the context',
        'not in the context',
        'cannot answer',
        'not enough information',
        'insufficient information',
        'not provided in the context',
    ]
    return any(marker in lowered for marker in abstain_markers)


In [ ]:
def sanitize_llama_tokenizer_export_dir(save_dir: str) -> None:
    sanitize_tokenizer_config_dir(save_dir)
    tok_cfg_path = os.path.join(save_dir, 'tokenizer_config.json')
    if not os.path.exists(tok_cfg_path):
        return

    with open(tok_cfg_path, 'r', encoding='utf-8') as file_obj:
        tok_cfg = json.load(file_obj)

    changed = False
    if tok_cfg.get('tokenizer_class') == 'TokenizersBackend':
        tok_cfg.pop('tokenizer_class', None)
        changed = True

    desired_model_max_length = max(cfg.model_context_limit, cfg.max_seq_len)
    if tok_cfg.get('model_max_length') != desired_model_max_length:
        tok_cfg['model_max_length'] = desired_model_max_length
        changed = True

    if tok_cfg.get('pad_token') != tokenizer.pad_token:
        tok_cfg['pad_token'] = tokenizer.pad_token
        changed = True

    if tok_cfg.get('bos_token') != tokenizer.bos_token:
        tok_cfg['bos_token'] = tokenizer.bos_token
        changed = True

    if tok_cfg.get('eos_token') != tokenizer.eos_token:
        tok_cfg['eos_token'] = tokenizer.eos_token
        changed = True

    if changed:
        with open(tok_cfg_path, 'w', encoding='utf-8') as file_obj:
            json.dump(tok_cfg, file_obj, ensure_ascii=False, indent=2)
            file_obj.write('\\n')
        print('[INFO] Wrote safer tokenizer_config.json for endpoint export.')

def build_supervised_example(prompt_text: str, full_text: str) -> Optional[Dict[str, List[int]]]:
    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    input_ids = encoded['input_ids']
    offsets = encoded['offset_mapping']

    prompt_char_len = len(prompt_text)
    answer_start = 0
    for _, end_char in offsets:
        if end_char <= prompt_char_len:
            answer_start += 1
        else:
            break

    prompt_ids = input_ids[:answer_start]
    answer_ids = input_ids[answer_start:]

    if not answer_ids:
        return None
    if len(answer_ids) >= cfg.max_seq_len:
        return None

    max_prompt_tokens = max(1, cfg.max_seq_len - len(answer_ids))
    truncated_prompt_ids = prompt_ids[-max_prompt_tokens:]
    combined_ids = truncated_prompt_ids + answer_ids

    attention_mask = [1] * len(combined_ids)
    labels = ([-100] * len(truncated_prompt_ids)) + answer_ids

    pad_len = cfg.max_seq_len - len(combined_ids)
    if pad_len > 0:
        combined_ids = combined_ids + ([tokenizer.pad_token_id] * pad_len)
        attention_mask = attention_mask + ([0] * pad_len)
        labels = labels + ([-100] * pad_len)

    return {
        'input_ids': combined_ids,
        'attention_mask': attention_mask,
        'labels': labels,
        'prompt_tokens_kept': len(truncated_prompt_ids),
        'prompt_tokens_dropped': max(0, len(prompt_ids) - len(truncated_prompt_ids)),
        'answer_tokens': len(answer_ids),
    }

def write_safe_llama_config(save_dir: str) -> None:
    config_path = os.path.join(save_dir, 'config.json')
    if not os.path.exists(config_path):
        return

    with open(config_path, 'r', encoding='utf-8') as file_obj:
        model_config = json.load(file_obj)

    changed = False
    rope_config = model_config.get('rope_scaling') or model_config.get('rope_parameters') or {}
    original_context = int(rope_config.get('original_max_position_embeddings') or 0)
    existing_context = int(model_config.get('max_position_embeddings') or 0)
    desired_context_limit = max(cfg.model_context_limit, cfg.max_seq_len, existing_context)
    if original_context:
        desired_context_limit = max(desired_context_limit, original_context + 1)

    if model_config.get('pad_token_id') != tokenizer.pad_token_id:
        model_config['pad_token_id'] = tokenizer.pad_token_id
        changed = True

    if model_config.get('bos_token_id') != tokenizer.bos_token_id:
        model_config['bos_token_id'] = tokenizer.bos_token_id
        changed = True

    eos_token_id = model_config.get('eos_token_id')
    if eos_token_id is None:
        model_config['eos_token_id'] = tokenizer.eos_token_id
        changed = True

    if model_config.get('max_position_embeddings', desired_context_limit) != desired_context_limit:
        model_config['max_position_embeddings'] = desired_context_limit
        changed = True

    if model_config.get('use_cache') is not True:
        model_config['use_cache'] = True
        changed = True

    if '_name_or_path' in model_config:
        model_config.pop('_name_or_path', None)
        changed = True

    if changed:
        with open(config_path, 'w', encoding='utf-8') as file_obj:
            json.dump(model_config, file_obj, ensure_ascii=False, indent=2)
            file_obj.write('\\n')
        print('[INFO] Wrote safer config.json for endpoint export.')

SAMPLING_ONLY_GENERATION_FIELDS = [
    'temperature',
    'top_p',
    'top_k',
    'typical_p',
    'epsilon_cutoff',
    'eta_cutoff',
]


def deterministic_generation_config_dict() -> Dict:
    return {
        'bos_token_id': tokenizer.bos_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'do_sample': False,
        'repetition_penalty': cfg.generation_repetition_penalty,
        'no_repeat_ngram_size': cfg.generation_no_repeat_ngram_size,
        'renormalize_logits': True,
        'max_new_tokens': cfg.generation_max_new_tokens,
    }


def clear_sampling_generation_fields(generation_config) -> None:
    for field_name in SAMPLING_ONLY_GENERATION_FIELDS:
        if hasattr(generation_config, field_name):
            try:
                setattr(generation_config, field_name, None)
            except Exception:
                pass


def write_grounded_generation_config(save_dir: str) -> None:
    generation_config = deterministic_generation_config_dict()
    with open(os.path.join(save_dir, 'generation_config.json'), 'w', encoding='utf-8') as file_obj:
        json.dump(generation_config, file_obj, ensure_ascii=False, indent=2)
        file_obj.write('\\n')


def apply_grounded_generation_defaults(model) -> None:
    generation_config = getattr(model, 'generation_config', None)
    if generation_config is None:
        return
    generation_config.bos_token_id = tokenizer.bos_token_id
    generation_config.eos_token_id = tokenizer.eos_token_id
    generation_config.pad_token_id = tokenizer.pad_token_id
    generation_config.do_sample = False
    clear_sampling_generation_fields(generation_config)
    generation_config.repetition_penalty = cfg.generation_repetition_penalty
    generation_config.no_repeat_ngram_size = cfg.generation_no_repeat_ngram_size
    generation_config.renormalize_logits = True
    generation_config.max_new_tokens = cfg.generation_max_new_tokens

def build_endpoint_handler_source() -> str:
    return '''import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import AutoPeftModelForCausalLM

DEFAULT_INSTRUCTION = (
    "You are an expert research assistant for the Born in Bradford study. "
    "Use only the provided context. "
    "If the context contains the requested value, relationship, or finding, answer it directly; do not say it is absent. "
    "Only answer exactly: The provided context does not contain the requested information after checking paper excerpts, labels, headings, rows, and adjacent context for an exact match. "
    "For table-like or multi-finding context, choose the answer whose row label, column label, outcome, subgroup, timepoint, model, and measurement all match the question; ignore nearby rows where any anchor differs. "
    "Follow the response rule after the question, because it identifies the exact extraction style needed. "
    "Preserve all numbers, ranges, units, p-values, confidence intervals, dates, "
    "distances, comparison groups, and categories exactly as written in the context. "
    "Do not add page numbers, figure numbers, table positions, p-values, methods, "
    "ranges, or qualifiers unless they appear in the provided context. "
    "Answer only the requested value, relationship, or finding; do not include "
    "adjacent table values, extra categories, variable metadata, or explanatory detail unless asked. "
    "Do not add partial related information after abstaining."
)

class EndpointHandler:
    def __init__(self, path: str = ""):
        model_dir = path or "/repository"
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            trust_remote_code=True,
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.truncation_side = "left"

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        adapter_config_path = os.path.join(model_dir, "adapter_config.json")
        if os.path.exists(adapter_config_path):
            self.model = AutoPeftModelForCausalLM.from_pretrained(
                model_dir,
                trust_remote_code=True,
                torch_dtype=dtype,
                low_cpu_mem_usage=True,
                device_map="auto" if torch.cuda.is_available() else None,
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_dir,
                trust_remote_code=True,
                torch_dtype=dtype,
                low_cpu_mem_usage=True,
                device_map="auto" if torch.cuda.is_available() else None,
            )

        self.model.eval()

    def _coerce_messages(self, raw_messages):
        messages = []
        for message in raw_messages:
            if not isinstance(message, dict):
                continue
            role = str(message.get("role", "")).strip()
            content = message.get("content", "")
            if role not in {"system", "user", "assistant"}:
                continue
            if content is None:
                content = ""
            messages.append({"role": role, "content": str(content)})
        return messages

    def _merge_system_into_first_user(self, messages):
        system_text = "\\n\\n".join(
            message["content"] for message in messages if message["role"] == "system"
        ).strip()
        non_system_messages = [message for message in messages if message["role"] != "system"]

        if not system_text:
            return non_system_messages
        if not non_system_messages:
            return [{"role": "user", "content": f"Instruction: {system_text}"}]

        for index, message in enumerate(non_system_messages):
            if message["role"] == "user":
                non_system_messages[index] = {
                    "role": "user",
                    "content": f"Instruction: {system_text}\\n\\n{message['content']}",
                }
                return non_system_messages

        return [{"role": "user", "content": f"Instruction: {system_text}"}] + non_system_messages

    def _extract_question_from_content(self, content):
        text = str(content or "")
        marker = None
        for candidate in ("Researcher question:", "Question:"):
            if candidate in text:
                marker = candidate
        if marker is None:
            return ""
        question = text.rsplit(marker, 1)[1]
        question = question.split("\\nAnswer:", 1)[0]
        question = question.split("\\n\\nResponse rule:", 1)[0]
        return question.strip()

    def _response_rule(self, question, context=""):
        question_text = str(question or "")
        context_text = str(context or "")
        lowered = question_text.lower()
        relationship_terms = (
            "relationship", "association", "associated", "correlation",
            "linked", "impact", "effect", "finding", "according to the study",
        )
        direct_numeric_terms = (
            "odds ratio", " percentage", "percent", "proportion", "count", "number",
            "value", "values", "cutoff", "cut-off", "coefficient", "confidence interval",
            "ci", "p-value", "sample size", "mean", "median", "coded", "code",
            "classified", "unit",
        )
        paper_terms = ("paper", "study", "article", "published", "according to the study", "according to the paper", "systematic review", "meta-analysis")
        metadata_terms = ("variable", "table", "dataset", "data table", "field", "column", "label", "coded", "code", "non-missing", "records", "entities", "rows")
        context_has_papers = "## Relevant Published Papers" in context_text or "Source: full-text PDF" in context_text
        context_has_metadata = "## Relevant Variables" in context_text or "## Relevant Tables" in context_text
        asks_paper_style = any(term in lowered for term in paper_terms) and not any(term in lowered for term in metadata_terms)
        source_priority_rule = ""
        if context_has_papers and context_has_metadata and asks_paper_style:
            source_priority_rule = (
                "For published-paper questions, answer from the Relevant Published Papers/full-text PDF text. "
                "Ignore Relevant Variables and Relevant Tables metadata unless the question explicitly asks for variables, tables, columns, labels, coding, or non-missing records. "
            )
        presence_rule = (
            "Positive evidence has priority over abstention. "
            "If any sentence, table row, header-expanded row, or labelled excerpt contains the requested value, relationship, or finding, answer it directly; do not abstain. "
            "Before abstaining, check repeated headers, row labels, timepoints, outcomes, comparison groups, paper excerpts, and adjacent context for an exact match. "
            "Do not use the abstention answer when a matching answer is present under the requested label, source, study, or outcome. "
        )
        abstention_rule = (
            "Only if the exact requested value, comparison, timepoint, work package, criterion, source, funding body, or finding is not present after that check, "
            "answer exactly: \\\"The provided context does not contain the requested information.\\\" and stop. "
            "Do not add partial related information after abstaining. "
            "Do not infer, calculate, or substitute a nearby value from another outcome, age, time period, work package, exposure, table row, cited study, or metadata section. "
        )
        anchor_match_rule = (
            "Wrong-neighbor rule: identify the required anchors in the question, including outcome, measure, subgroup, cohort, model, timepoint, emotion, frequency, exposure, and comparison groups. "
            "Use only evidence where all requested anchors match. "
            "Reject nearby rows or sentences if they are about a different outcome, measure, group, model, timepoint, emotion, frequency, exposure, or comparison, even when their numbers look plausible. "
            "If a matching row contains multiple requested groups, return all requested values from that row; do not omit one group. "
        )
        asks_for_relationship = any(term in lowered for term in relationship_terms)
        asks_for_direct_number = any(term in lowered for term in direct_numeric_terms) or " or " in f" {lowered} "
        if asks_for_relationship and not asks_for_direct_number:
            return (
                "Response rule: " + source_priority_rule + presence_rule + anchor_match_rule +
                "State the requested relationship or finding in words using only the context. "
                "Match the requested exposure, outcome, population, cited study, and timepoint exactly. "
                "If the exact relationship statement is present, restate it rather than abstaining, even if the context also contains adjacent unrelated findings. "
                "Keep numbers that identify requested groups or timepoints, but do not add model coefficients, "
                "confidence intervals, p-values, table/figure/page references, variable names, or adjacent numeric details unless the question asks for them. "
                + abstention_rule
            )
        numeric_terms = (
            "odds ratio", "percentage", "percent", "proportion", "count", "number",
            "value", "values", "cutoff", "cut-off", "coefficient", "confidence interval",
            "ci", "p-value", "sample size", "mean", "median", "bmi", "z-score",
            "quintile", "buffer", "year", "month", "date", "coded", "code",
            "classified", "unit",
        )
        is_numeric_like = any(char.isdigit() for char in question_text) or any(
            term in lowered for term in numeric_terms
        )
        if is_numeric_like:
            return (
                "Response rule: " + source_priority_rule + presence_rule + anchor_match_rule +
                "Extract only the value(s) requested by the question. "
                "Use nearby headers and labels to choose the matching column, outcome, timepoint, comparison group, or row; do not default to the first number in a row. "
                "Prefer the value whose row and column labels both match the question. If that matching value is present, return it rather than abstaining. "
                "Match the requested age, time period, subgroup, exposure, outcome, and direction exactly before copying a number. "
                "Copy requested numbers, ranges, units, comparison groups, and categories exactly from the context. "
                "If the question asks for multiple groups or contrasts two reported values, include each requested group. "
                "Do not include adjacent rows, unrelated categories, p-values, confidence intervals, variable names, "
                "table/figure/page references, or explanations unless the question asks for them. "
                + abstention_rule
            )
        return (
            "Response rule: " + source_priority_rule +
            "Answer directly and concisely using only the context. "
            "Match the requested paper, study aim, source, criterion, work package, population, exposure, and outcome exactly. "
            "Do not add table, figure, page, row, variable, or section references unless the question asks for them. "
            + abstention_rule
        )

    def _append_response_rule_to_last_user(self, messages):
        updated = list(messages)
        for index in range(len(updated) - 1, -1, -1):
            message = updated[index]
            if message.get("role") != "user":
                continue
            content = message.get("content", "")
            if "Response rule:" in content:
                return updated
            question = self._extract_question_from_content(content)
            if question:
                updated[index] = {
                    "role": "user",
                    "content": f"{content}\\n\\n{self._response_rule(question, content)}",
                }
            return updated
        return updated

    def _build_messages(self, inputs, params):
        use_system_role = bool(params.get("use_system_role", False))

        if isinstance(inputs, list):
            messages = self._coerce_messages(inputs)
            if not messages:
                return [{"role": "user", "content": ""}]
            if use_system_role:
                return self._append_response_rule_to_last_user(messages)
            return self._append_response_rule_to_last_user(self._merge_system_into_first_user(messages))

        if isinstance(inputs, dict) and "messages" in inputs:
            return self._build_messages(inputs["messages"], params)

        instruction = DEFAULT_INSTRUCTION
        if isinstance(inputs, dict) and inputs.get("system"):
            instruction = str(inputs["system"])

        if isinstance(inputs, dict) and "context" in inputs and "question" in inputs:
            question = str(inputs["question"])
            context = str(inputs["context"])
            response_rule = self._response_rule(question, context)
            if use_system_role:
                return [
                    {"role": "system", "content": instruction},
                    {"role": "user", "content": f"Context:\\n{context}\\n\\nQuestion: {question}\\n\\n{response_rule}"},
                ]
            return [
                {
                    "role": "user",
                    "content": f"Instruction: {instruction}\\n\\nContext:\\n{context}\\n\\nQuestion: {question}\\n\\n{response_rule}",
                }
            ]

        prompt_text = str(inputs.get("prompt", "")) if isinstance(inputs, dict) else str(inputs)
        if use_system_role:
            return self._append_response_rule_to_last_user([
                {"role": "system", "content": instruction},
                {"role": "user", "content": prompt_text},
            ])
        return self._append_response_rule_to_last_user([
            {"role": "user", "content": f"Instruction: {instruction}\\n\\n{prompt_text}"},
        ])

    def __call__(self, data):
        inputs = data.get("inputs", "")
        params = data.get("parameters", {}) or {}

        max_new_tokens = min(int(params.get("max_new_tokens", 64)), 512)
        max_input_tokens = int(params.get("max_input_tokens", 4096))
        max_input_tokens = max(512, min(max_input_tokens, 8192))
        do_sample = bool(params.get("do_sample", False))
        temperature = float(params.get("temperature", 0.7 if do_sample else 0.0))
        top_p = float(params.get("top_p", 1.0))
        repetition_penalty = float(params.get("repetition_penalty", 1.0))
        no_repeat_ngram_size = int(params.get("no_repeat_ngram_size", 0))
        debug = bool(params.get("debug", False))

        messages = self._build_messages(inputs, params)
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        enc = self.tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=max_input_tokens,
        )
        truncated = enc["input_ids"].shape[-1] >= max_input_tokens

        if torch.cuda.is_available():
            enc = {key: value.to(self.model.device) for key, value in enc.items()}

        eos_token_id = getattr(self.model.config, "eos_token_id", None)
        if eos_token_id is None:
            eos_token_id = self.tokenizer.eos_token_id

        generate_kwargs = dict(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            repetition_penalty=repetition_penalty,
            renormalize_logits=True,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=eos_token_id,
        )
        if do_sample:
            generate_kwargs["temperature"] = max(temperature, 1e-5)
            generate_kwargs["top_p"] = top_p
        if no_repeat_ngram_size > 0:
            generate_kwargs["no_repeat_ngram_size"] = no_repeat_ngram_size

        with torch.no_grad():
            out = self.model.generate(**generate_kwargs)

        generated_ids = out[0][enc["input_ids"].shape[-1]:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        response = {"generated_text": text}
        if debug:
            response["prompt"] = prompt
            response["messages"] = messages
            response["input_tokens"] = int(enc["input_ids"].shape[-1])
            response["truncated"] = bool(truncated)
        return response
'''

def write_endpoint_handler_artifacts(save_dir: str) -> None:
    handler_path = os.path.join(save_dir, 'handler.py')
    requirements_path = os.path.join(save_dir, 'requirements.txt')

    with open(handler_path, 'w', encoding='utf-8') as file_obj:
        file_obj.write(build_endpoint_handler_source())
        if not build_endpoint_handler_source().endswith('\n'):
            file_obj.write('\\n')

    endpoint_requirements = [
        'transformers==4.51.3',
        'tokenizers==0.21.1',
        'huggingface_hub==0.30.2',
        'accelerate==1.6.0',
        'peft==0.15.2',
        'safetensors>=0.4.3',
    ]
    with open(requirements_path, 'w', encoding='utf-8') as file_obj:
        file_obj.write('\n'.join(endpoint_requirements) + '\n')

def prepare_endpoint_export_dir(save_dir: str) -> None:
    sanitize_llama_tokenizer_export_dir(save_dir)
    write_safe_llama_config(save_dir)
    write_grounded_generation_config(save_dir)
    write_endpoint_handler_artifacts(save_dir)


In [ ]:
pdf_files = sorted(glob.glob(cfg.pdf_glob))
print(f'Found {len(pdf_files)} PDFs')
for path in pdf_files[:10]:
    print(' -', path)

if not os.path.exists(cfg.sft_train_path):
    raise ValueError(f'Missing SFT train file: {cfg.sft_train_path}')
if not os.path.exists(cfg.sft_dev_path):
    raise ValueError(f'Missing SFT dev file: {cfg.sft_dev_path}')

sft_train_raw = load_jsonl(cfg.sft_train_path)
sft_dev_raw = load_jsonl(cfg.sft_dev_path)
print(f'SFT train examples: {len(sft_train_raw)}')
print(f'SFT dev examples: {len(sft_dev_raw)}')

if sft_train_raw:
    print('Sample SFT train keys:', sorted(sft_train_raw[0].keys()))

In [ ]:
stage1_train_ds = None
stage1_eval_ds = None
raw_docs = []

if cfg.run_stage1 and pdf_files:
    for path in pdf_files:
        text = extract_pdf_text(path)[: cfg.max_chars_per_doc]
        if text:
            raw_docs.append({'source': os.path.basename(path), 'text': text})

    print(f'Parsed docs for Stage 1: {len(raw_docs)}')

    unique_sources = [doc['source'] for doc in raw_docs]
    random.Random(SEED).shuffle(unique_sources)
    cut = max(1, min(len(unique_sources) - 1, int(len(unique_sources) * (1.0 - cfg.stage1_test_size))))
    train_sources = set(unique_sources[:cut])
    eval_sources = set(unique_sources[cut:])

    stage1_domain_train = []
    stage1_domain_eval = []

    for doc in raw_docs:
        chunks = paragraph_pack_chunks(
            doc['text'],
            chunk_target_tokens=cfg.stage1_chunk_target_tokens,
            overlap_tokens=cfg.stage1_chunk_overlap_tokens,
            min_chunk_tokens=cfg.stage1_min_chunk_tokens,
        )
        for chunk in chunks:
            record = {'text': chunk, 'source': doc['source'], 'kind': 'domain'}
            if doc['source'] in train_sources:
                stage1_domain_train.append(record)
            else:
                stage1_domain_eval.append(record)

    retention_train_texts = [render_chat_transcript(messages) for example in sft_train_raw if (messages := build_production_messages(example))]
    retention_eval_texts = [render_chat_transcript(messages) for example in sft_dev_raw if (messages := build_production_messages(example))]

    random.Random(SEED).shuffle(retention_train_texts)
    max_retention_train = int(len(stage1_domain_train) * cfg.stage1_retention_fraction / max(1e-6, 1.0 - cfg.stage1_retention_fraction))
    max_retention_eval = max(1, min(len(retention_eval_texts), max(8, int(len(stage1_domain_eval) * cfg.stage1_retention_fraction))))

    stage1_retention_train = [
        {'text': text, 'source': 'sft_train_retention', 'kind': 'retention'}
        for text in retention_train_texts[:max_retention_train]
    ]
    stage1_retention_eval = [
        {'text': text, 'source': 'sft_dev_retention', 'kind': 'retention'}
        for text in retention_eval_texts[:max_retention_eval]
    ]

    stage1_train_records = stage1_domain_train + stage1_retention_train
    stage1_eval_records = stage1_domain_eval + stage1_retention_eval

    random.Random(SEED).shuffle(stage1_train_records)
    random.Random(SEED).shuffle(stage1_eval_records)

    stage1_train_ds = Dataset.from_list(stage1_train_records)
    stage1_eval_ds = Dataset.from_list(stage1_eval_records)

    print('Stage 1 domain train chunks:', len(stage1_domain_train))
    print('Stage 1 retention train examples:', len(stage1_retention_train))
    print('Stage 1 train total:', len(stage1_train_ds))
    print('Stage 1 eval total:', len(stage1_eval_ds))
else:
    print('Stage 1 will be skipped because run_stage1=False or no PDFs were found.')

In [ ]:
def tokenize_lm_batch(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=cfg.max_seq_len,
    )

    labels = []
    for input_ids, attention_mask in zip(out['input_ids'], out['attention_mask']):
        row_labels = input_ids.copy()
        for idx, mask_value in enumerate(attention_mask):
            if mask_value == 0:
                row_labels[idx] = -100
        labels.append(row_labels)

    out['labels'] = labels
    return out

if stage1_train_ds is not None and stage1_eval_ds is not None:
    stage1_train_tok = stage1_train_ds.map(tokenize_lm_batch, batched=True, remove_columns=stage1_train_ds.column_names)
    stage1_eval_tok = stage1_eval_ds.map(tokenize_lm_batch, batched=True, remove_columns=stage1_eval_ds.column_names)
    stage1_train_tok.set_format(type='torch')
    stage1_eval_tok.set_format(type='torch')
    print(stage1_train_tok)
    print(stage1_eval_tok)

In [ ]:
def validate_bitsandbytes_cuda() -> None:
    if not cfg.use_4bit:
        return
    try:
        import bitsandbytes as bnb
        print('bitsandbytes import OK:', getattr(bnb, '__version__', 'unknown'))
        if torch.cuda.is_available():
            probe = torch.randn(64, device='cuda', dtype=torch.float16)
            bnb.functional.quantize_4bit(probe, quant_type='nf4')
            torch.cuda.synchronize()
            print('bitsandbytes CUDA NF4 probe OK.')
    except Exception as exc:
        if cfg.allow_4bit_fallback:
            print('[WARN] bitsandbytes 4-bit CUDA probe failed; falling back to non-4-bit LoRA loading.')
            print('[WARN] Original bitsandbytes error:', repr(exc))
            cfg.use_4bit = False
            return
        raise RuntimeError(
            'bitsandbytes 4-bit CUDA setup failed. Rerun the dependency setup cell so it installs '
            'the CUDA-13-capable bitsandbytes stack, restarts the runtime, then rerun from the imports cell. '
            'Alternatively set cfg.use_4bit=False before model loading.'
        ) from exc


def checkpointing_kwargs() -> Dict:
    return {'use_reentrant': cfg.gradient_checkpointing_use_reentrant}


def enable_input_grads_for_checkpointing(model) -> None:
    if not cfg.gradient_checkpointing:
        return
    if getattr(model, '_bib_input_grads_enabled', False):
        return
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
        setattr(model, '_bib_input_grads_enabled', True)
        print('Enabled input require grads for gradient checkpointing.')
        return

    input_embeddings = model.get_input_embeddings() if hasattr(model, 'get_input_embeddings') else None
    if input_embeddings is None:
        print('[WARN] Could not find input embeddings to enable input grads for checkpointing.')
        return

    def make_inputs_require_grad(module, inputs, output):
        output.requires_grad_(True)

    input_embeddings.register_forward_hook(make_inputs_require_grad)
    setattr(model, '_bib_input_grads_enabled', True)
    print('Registered input embedding grad hook for gradient checkpointing.')


def assert_trainable_parameters(model, stage_name: str) -> None:
    trainable_params = [(name, param) for name, param in model.named_parameters() if param.requires_grad]
    trainable_count = sum(param.numel() for _, param in trainable_params)
    total_count = sum(param.numel() for param in model.parameters())
    if trainable_count == 0:
        raise RuntimeError(
            f'{stage_name}: no trainable parameters found. The PEFT adapter is frozen, so training would fail '
            'with tensors that do not require grad.'
        )
    print(f'{stage_name}: trainable parameters = {trainable_count:,} / {total_count:,} ({100 * trainable_count / max(1, total_count):.4f}%)')
    print(f'{stage_name}: sample trainable parameter names:', [name for name, _ in trainable_params[:8]])


def prepare_peft_model_for_training(model, stage_name: str):
    model.config.use_cache = False
    if cfg.gradient_checkpointing:
        enable_input_grads_for_checkpointing(model)
    assert_trainable_parameters(model, stage_name)
    model.train()
    return model


def build_peft_model():
    validate_bitsandbytes_cuda()
    if cfg.use_4bit and not torch.cuda.is_available():
        raise RuntimeError('use_4bit=True requires CUDA.')

    model_kwargs = {'trust_remote_code': True}
    if cfg.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        )
        model_kwargs['quantization_config'] = bnb_config
        model_kwargs['device_map'] = 'auto'
    else:
        model_kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    model = AutoModelForCausalLM.from_pretrained(cfg.base_model, **model_kwargs)
    model.config.use_cache = False

    if cfg.use_4bit:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=cfg.gradient_checkpointing,
            gradient_checkpointing_kwargs=checkpointing_kwargs(),
        )
    elif cfg.gradient_checkpointing and hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=checkpointing_kwargs())

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return prepare_peft_model_for_training(model, 'initial PEFT model')


def training_flags():
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16 = torch.cuda.is_available() and not bf16
    return bf16, fp16


In [ ]:
if cfg.use_wandb:
    os.environ.setdefault('WANDB_PROJECT', cfg.wandb_project)

bf16, fp16 = training_flags()
stage1_model = build_peft_model()

if cfg.run_stage1 and stage1_train_ds is not None and len(stage1_train_ds) > 0:
    stage1_args = TrainingArguments(
        output_dir=cfg.stage1_output_dir,
        num_train_epochs=cfg.stage1_epochs,
        learning_rate=cfg.stage1_learning_rate,
        per_device_train_batch_size=cfg.stage1_train_batch_size,
        per_device_eval_batch_size=cfg.stage1_eval_batch_size,
        gradient_accumulation_steps=cfg.stage1_grad_accum_steps,
        warmup_ratio=cfg.warmup_ratio,
        weight_decay=cfg.weight_decay,
        logging_steps=10,
        save_strategy='epoch',
        eval_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        lr_scheduler_type='cosine',
        bf16=bf16,
        fp16=fp16,
        gradient_checkpointing=cfg.gradient_checkpointing,
        gradient_checkpointing_kwargs=checkpointing_kwargs(),
        report_to=['wandb'] if cfg.use_wandb else ['none'],
        run_name=f'{cfg.wandb_run_name}-stage1' if cfg.use_wandb else None,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
        label_names=['labels'],
    )

    stage1_callbacks = []
    if cfg.early_stopping_patience > 0:
        stage1_callbacks.append(EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience))

    stage1_trainer = Trainer(
        model=stage1_model,
        args=stage1_args,
        train_dataset=stage1_train_tok,
        eval_dataset=stage1_eval_tok,
        data_collator=default_data_collator,
        callbacks=stage1_callbacks,
    )

    stage1_trainer.train()
    stage1_trainer.save_model(cfg.stage1_output_dir)
    tokenizer.save_pretrained(cfg.stage1_output_dir)
    stage2_model = stage1_trainer.model
else:
    print('Skipping Stage 1 training.')
    stage2_model = stage1_model

stage2_model = prepare_peft_model_for_training(stage2_model, 'stage2 model before SFT')


In [ ]:
TARGETED_NUMERIC_RESPONSE_RULE = (
    'Response rule: Extract only the value(s) requested by the question. '
    'Read the Question anchors, Matching evidence row, and Distractor rows blocks before answering. '
    'Use the repeated table header or label text to choose the correct column/group. '
    'First identify the requested anchors: outcome, measure, subgroup, cohort, model, timepoint, emotion, frequency, exposure, and comparison groups. '
    'Reject adjacent rows or sentences when any requested anchor differs, even if the nearby number looks plausible. '
    'If the requested value is the second or later value in a row, return that requested value, not the first value. '
    'If the exact requested value is present under matching row and column labels, answer it; do not abstain. '
    'Treat a labelled sentence or header-expanded row as sufficient evidence when it directly states the requested value or finding. '
    'Match the requested outcome, age, timepoint, subgroup, exposure, comparison, and direction exactly before copying a number. '
    'If the question asks for multiple groups, include every requested group. '
    'Do not include adjacent rows, unrelated categories, p-values, confidence intervals, table/figure/page references, variable names, or explanations unless the question asks for them. '
    'If the exact requested value is absent, answer exactly: "The provided context does not contain the requested information." and stop.'
)

TABLE_HEADER_SIGNAL_RE = re.compile(
    r'\b(?:or|odds ratio|95%|ci|p-value|p value|mean|median|sd|n\s*=|percent|percentage|quintile|group|category|code|coded|overweight|obese|normal|healthy|buffer|coefficient|outcome|value|row|column)\b',
    flags=re.IGNORECASE,
)


def compact_spaces(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def get_row_question(row: Dict) -> str:
    return str(row.get('question') or extract_question_from_messages(row.get('messages') or []) or '').strip()


def get_row_answer(row: Dict) -> str:
    answer = row.get('answer')
    messages = row.get('messages') or []
    if answer is None and isinstance(messages, list) and messages and messages[-1].get('role') == 'assistant':
        answer = messages[-1].get('content')
    return str(answer or '').strip()


def get_row_context(row: Dict) -> str:
    return str(row.get('source_chunk') or row.get('context') or extract_context_from_messages(row.get('messages') or []) or '').strip()


def get_row_evidence(row: Dict) -> str:
    return str(row.get('evidence') or '').strip()


def has_table_header_signal(line: str) -> bool:
    line = compact_spaces(line)
    if not line:
        return False
    if TABLE_HEADER_SIGNAL_RE.search(line):
        return True
    numeric_count = len(extract_numeric_tokens(line))
    alpha_count = sum(char.isalpha() for char in line)
    return numeric_count >= 2 and alpha_count >= 8


def infer_table_header_hint(context: str, evidence: str, max_header_lines: int = 3, search_window: int = 12) -> str:
    context = str(context or '')
    evidence = compact_spaces(evidence)
    if not context or not evidence:
        return ''

    lines = [compact_spaces(line) for line in context.splitlines() if compact_spaces(line)]
    if not lines:
        lines = [compact_spaces(sentence) for sentence in re.split(r'(?<=[.;])\s+', context) if compact_spaces(sentence)]

    evidence_norm = compact_spaces(evidence).lower()
    evidence_anchor = ' '.join(evidence_norm.split()[:8])
    row_index = None
    for idx, line in enumerate(lines):
        line_norm = compact_spaces(line).lower()
        if evidence_norm and evidence_norm in line_norm:
            row_index = idx
            break
        if evidence_anchor and len(evidence_anchor) > 20 and evidence_anchor in line_norm:
            row_index = idx
            break

    if row_index is None:
        target_numbers = extract_numeric_tokens(evidence)
        target_words = set(re.findall(r'[A-Za-z]{4,}', evidence.lower()))
        best_score = 0
        for idx, line in enumerate(lines):
            line_numbers = extract_numeric_tokens(line)
            line_words = set(re.findall(r'[A-Za-z]{4,}', line.lower()))
            score = len(set(target_numbers) & set(line_numbers)) + len(target_words & line_words)
            if score > best_score:
                best_score = score
                row_index = idx
        if best_score < 2:
            row_index = None

    if row_index is not None:
        preceding = lines[max(0, row_index - search_window):row_index]
        candidates = [line for line in preceding if has_table_header_signal(line)]
        if candidates:
            return '\n'.join(candidates[-max_header_lines:])

    global_candidates = [line for line in lines if has_table_header_signal(line) and compact_spaces(line).lower() != evidence_norm]
    return '\n'.join(global_candidates[:max_header_lines])


def compact_context_around_evidence(context: str, evidence: str, max_chars: Optional[int] = None) -> str:
    context = str(context or '').strip()
    if not context:
        return ''
    max_chars = max_chars or cfg.targeted_stage2_augmentation_max_context_chars
    if len(context) <= max_chars:
        return context

    evidence_text = compact_spaces(evidence)
    evidence_lower = evidence_text.lower()
    context_lower = context.lower()
    anchor = evidence_lower[:80]
    anchor_index = context_lower.find(anchor) if anchor else -1
    if anchor_index < 0:
        evidence_words = ' '.join(evidence_lower.split()[:6])
        anchor_index = context_lower.find(evidence_words) if evidence_words else -1

    if anchor_index < 0:
        return context[:max_chars].strip()

    half_window = max_chars // 2
    start = max(0, anchor_index - half_window)
    end = min(len(context), anchor_index + half_window)
    return context[start:end].strip()


QUESTION_ANCHOR_STOPWORDS = {
    'what', 'which', 'where', 'when', 'were', 'was', 'are', 'is', 'did', 'does', 'do',
    'the', 'this', 'that', 'these', 'those', 'and', 'or', 'of', 'for', 'to', 'in', 'on',
    'by', 'with', 'from', 'between', 'among', 'according', 'paper', 'study', 'reported',
    'value', 'values', 'relationship', 'association', 'effect', 'difference', 'compared',
    'comparison', 'born', 'bradford', 'bib', 'cohort', 'children', 'women', 'mothers',
    'participants', 'young', 'people', 'question', 'asked', 'complete', 'used', 'using',
}


def extract_question_anchor_terms(question: str, max_terms: int = 18) -> List[str]:
    question = str(question or '')
    quoted_phrases = [compact_spaces(match) for match in re.findall(r'["\']([^"\']{3,80})["\']', question)]
    tokens = []
    for token in re.findall(r'[A-Za-z][A-Za-z0-9+/-]*|\d+(?:\.\d+)?%?|\d+-\d+', question):
        normalized = token.strip('.,;:()[]{}').lower()
        if not normalized:
            continue
        if normalized in QUESTION_ANCHOR_STOPWORDS:
            continue
        if len(normalized) < 3 and not any(char.isdigit() for char in normalized):
            continue
        tokens.append(normalized)

    terms = []
    seen = set()
    for phrase in quoted_phrases + tokens:
        normalized = compact_spaces(phrase).lower()
        if normalized and normalized not in seen:
            terms.append(normalized)
            seen.add(normalized)
        if len(terms) >= max_terms:
            break
    return terms


def split_context_lines_for_anchor_format(context: str) -> List[str]:
    lines = [compact_spaces(line) for line in str(context or '').splitlines() if compact_spaces(line)]
    if len(lines) <= 1:
        lines = [compact_spaces(part) for part in re.split(r'(?<=[.;])\s+', str(context or '')) if compact_spaces(part)]
    return lines


def line_anchor_overlap(line: str, anchors: List[str]) -> int:
    lowered = compact_spaces(line).lower()
    return sum(1 for anchor in anchors if anchor and anchor in lowered)


def clean_header_hint_for_anchor_format(header_hint: str) -> str:
    cleaned_lines = []
    for line in str(header_hint or '').splitlines():
        clean = compact_spaces(line)
        if not clean:
            continue
        numbers = extract_numeric_tokens(clean)
        has_pipe = '|' in clean
        # Keep pipe-delimited header lines, but avoid treating previous numeric data rows as headers.
        if has_pipe and len(numbers) <= 1:
            cleaned_lines.append(clean)
    return '\n'.join(cleaned_lines)


def infer_distractor_rows_to_ignore(
    context: str,
    evidence: str,
    question: str,
    answer: str,
    header_hint: str = '',
    max_rows: int = 4,
) -> List[str]:
    context_excerpt = compact_context_around_evidence(context, evidence, max_chars=2200)
    lines = split_context_lines_for_anchor_format(context_excerpt or context)
    if not lines:
        return []

    evidence_norm = compact_spaces(evidence).lower()
    answer_norm = compact_spaces(answer).lower()
    cleaned_header_hint = clean_header_hint_for_anchor_format(header_hint)
    header_lines = {compact_spaces(line).lower() for line in cleaned_header_hint.splitlines() if compact_spaces(line)}
    anchors = extract_question_anchor_terms(question)
    required_numbers = set(required_numeric_tokens(answer, question))
    evidence_numbers = set(extract_numeric_tokens(evidence))

    scored = []
    for line in lines:
        clean = compact_spaces(line)
        lowered = clean.lower()
        if not clean or lowered in header_lines:
            continue
        if lowered.startswith(('hard ', 'targeted ', 'positive answer calibration')):
            continue
        if evidence_norm and (evidence_norm in lowered or lowered in evidence_norm):
            continue
        if answer_norm and len(answer_norm) > 8 and answer_norm in lowered:
            continue

        numbers = set(extract_numeric_tokens(clean))
        overlap = line_anchor_overlap(clean, anchors)
        table_like = has_table_header_signal(clean) or '|' in clean or len(numbers) >= 2
        numeric_distractor = bool(numbers and (numbers - required_numbers) and (numbers - evidence_numbers))
        anchor_distractor = overlap >= 1 and (table_like or numbers)
        if not (numeric_distractor or anchor_distractor or table_like):
            continue

        score = (3 * overlap) + (2 if table_like else 0) + len(numbers) + (1 if numeric_distractor else 0)
        scored.append((score, clean[:360]))

    distractors = []
    seen = set()
    for _, clean in sorted(scored, key=lambda item: item[0], reverse=True):
        key = clean.lower()
        if key in seen:
            continue
        seen.add(key)
        distractors.append(clean)
        if len(distractors) >= max_rows:
            break
    return distractors


def build_anchor_structured_context(
    question: str,
    answer: str,
    context: str,
    evidence: str,
    header_hint: str = '',
) -> str:
    context_excerpt = compact_context_around_evidence(context, evidence)
    anchors = extract_question_anchor_terms(question)
    header_block = clean_header_hint_for_anchor_format(header_hint)
    evidence_row = compact_spaces(evidence)
    distractors = infer_distractor_rows_to_ignore(context, evidence, question, answer, header_hint)
    answer_is_abstain = is_abstain_answer(answer)

    parts = []
    if anchors:
        parts.append('Question anchors to match exactly:\n' + ', '.join(anchors))

    if evidence_row:
        evidence_label = 'Evidence showing the requested answer is absent' if answer_is_abstain else 'Matching evidence row or sentence'
        if header_block:
            parts.append(evidence_label + ':\n' + header_block + '\n' + evidence_row)
        else:
            parts.append(evidence_label + ':\n' + evidence_row)

    if distractors:
        parts.append(
            'Distractor rows or sentences to ignore because one or more anchors differ:\n'
            + '\n'.join('- ' + row for row in distractors)
        )

    if context_excerpt:
        parts.append('Original retrieved context excerpt:\n' + context_excerpt)
    return '\n\n'.join(parts).strip()


def build_header_expanded_context(context: str, evidence: str, header_hint: str, question: str = '', answer: str = '') -> str:
    return build_anchor_structured_context(
        question=question,
        answer=answer,
        context=context,
        evidence=evidence,
        header_hint=header_hint,
    )


def targeted_augmentation_types(row: Dict) -> List[str]:
    question = get_row_question(row)
    answer = get_row_answer(row)
    evidence = get_row_evidence(row)
    if not question or not answer or not evidence:
        return []

    evidence_numbers = extract_numeric_tokens(evidence)
    required_numbers = required_numeric_tokens(answer, question)
    question_numbers = set(extract_numeric_tokens(question))
    answer_numbers = extract_numeric_tokens(answer)
    types = []

    required_positions = [evidence_numbers.index(number) for number in required_numbers if number in evidence_numbers]
    if required_positions and min(required_positions) > 0:
        types.append('second_or_later_value')

    adjacent_numbers = [number for number in evidence_numbers if number not in required_numbers and number not in question_numbers]
    if required_numbers and adjacent_numbers:
        types.append('adjacent_value_suppression')

    multi_group_question = bool(re.search(r'\b(?:both|respectively| and | compared with | for .* and .*)\b', question, flags=re.IGNORECASE))
    if len(required_numbers) >= 2 or (multi_group_question and len(answer_numbers) >= 2):
        types.append('multi_value_target')

    if types:
        types.append('header_expanded_row')
    return sorted(set(types))


def make_targeted_augmented_row(row: Dict, augmentation_types: List[str], index: int) -> Dict:
    question = get_row_question(row)
    answer = get_row_answer(row)
    context = get_row_context(row)
    evidence = get_row_evidence(row)
    header_hint = infer_table_header_hint(context, evidence)
    source_id = row.get('source_chunk_id') or (row.get('source_metadata') or {}).get('source') or f'train_row_{index}'

    return {
        'question': question,
        'answer': answer,
        'source_chunk': build_header_expanded_context(context, evidence, header_hint, question=question, answer=answer),
        'evidence': evidence,
        'source_chunk_id': f'{source_id}#targeted_augmentation_{index}',
        'source_metadata': {
            'source': f'{source_id}#targeted_augmentation',
            'augmentation_types': augmentation_types,
        },
        'response_rule': TARGETED_NUMERIC_RESPONSE_RULE,
        'system': cfg.production_system_prompt,
    }


def build_auto_targeted_augmentation_rows(raw_rows: List[Dict], max_examples: Optional[int] = None) -> List[Dict]:
    max_examples = cfg.targeted_stage2_augmentation_max_auto_examples if max_examples is None else max_examples
    if max_examples <= 0:
        return []

    augmented_rows = []
    seen_keys = set()
    type_counts = Counter()

    for idx, row in enumerate(raw_rows):
        augmentation_types = targeted_augmentation_types(row)
        if not augmentation_types:
            continue

        question = get_row_question(row)
        answer = get_row_answer(row)
        evidence = get_row_evidence(row)
        key = (question, answer, evidence, tuple(augmentation_types))
        if key in seen_keys:
            continue
        seen_keys.add(key)

        augmented_rows.append(make_targeted_augmented_row(row, augmentation_types, idx))
        type_counts.update(augmentation_types)
        if len(augmented_rows) >= max_examples:
            break

    print(f'Auto targeted Stage 2 augmentation rows: {len(augmented_rows)}')
    if augmented_rows:
        print('Auto targeted augmentation type counts:', dict(type_counts))
    return augmented_rows


def build_targeted_extraction_drill_rows() -> List[Dict]:
    drill_rows = [
        {
            'source_chunk_id': 'targeted_extraction_drill_second_value_or',
            'source_chunk': (
                'Targeted extraction drill: table headers are repeated beside each row.\n'
                'Group | Outcome A odds ratio | Outcome A 95% CI | Outcome A p-value | Outcome B odds ratio | Outcome B 95% CI | Outcome B p-value\n'
                'Higher education | 0.71 | 0.55-0.92 | 0.010 | 1.24 | 1.01-1.52 | 0.040'
            ),
            'evidence': 'Higher education | 0.71 | 0.55-0.92 | 0.010 | 1.24 | 1.01-1.52 | 0.040',
            'question': 'What is the odds ratio for Outcome B in the higher education group?',
            'answer': '1.24',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_third_value_mean',
            'source_chunk': (
                'Targeted extraction drill: table headers are repeated beside each row.\n'
                'Group | Low score mean | Middle score mean | High score mean\n'
                'Group C | 3.1 | 4.7 | 6.2'
            ),
            'evidence': 'Group C | 3.1 | 4.7 | 6.2',
            'question': 'What is the high score mean for Group C?',
            'answer': '6.2',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_code_only',
            'source_chunk': (
                'Targeted extraction drill: answer only the requested category code.\n'
                'Ethgrp2 coding: 1 = White British/Irish mothers; 2 = South Asian mothers; 3 = Other ethnic group.'
            ),
            'evidence': 'Ethgrp2 coding: 1 = White British/Irish mothers; 2 = South Asian mothers; 3 = Other ethnic group.',
            'question': 'How is Ethgrp2 coded for White British/Irish mothers?',
            'answer': '1',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_multi_percentage',
            'source_chunk': (
                'Targeted extraction drill: include all requested comparison groups and no adjacent groups.\n'
                'Buffer | Greenest quintile less likely | Quintiles 3 and 4 less likely | Least green quintile less likely\n'
                '100 m | 18% | 23% | 4%'
            ),
            'evidence': '100 m | 18% | 23% | 4%',
            'question': 'Within a 100 m buffer, what percentage reduction is reported for the greenest quintile and for quintiles 3 and 4?',
            'answer': 'Greenest quintile: 18%; quintiles 3 and 4: 23%.',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_cutoffs_only',
            'source_chunk': (
                'Targeted extraction drill: answer only the requested overweight cutoffs.\n'
                'Ethnic group | Underweight | Healthy weight | Overweight | Obese\n'
                'WE | <=18.5 | 18.6-24.9 | 25-29.9 | >=30\n'
                'SA | <=18.5 | 18.6-22.9 | 23-27.4 | >=27.5'
            ),
            'evidence': 'WE overweight 25-29.9; SA overweight 23-27.4',
            'question': 'What are the overweight BMI cutoffs for WE and SA women?',
            'answer': 'WE: 25-29.9; SA: 23-27.4.',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_percentage_and_count',
            'source_chunk': (
                'Targeted extraction drill: include both requested percentage and count.\n'
                'Category | Percentage | Count\n'
                'Employed, no access to money | 16% | n = 1722\n'
                'Unemployed, no access to money | 9% | n = 814'
            ),
            'evidence': 'Employed, no access to money | 16% | n = 1722',
            'question': 'What percentage and count are classified as Employed, no access to money?',
            'answer': '16% (n = 1722)',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_means_no_pvalue',
            'source_chunk': (
                'Targeted extraction drill: return the requested means, not the adjacent p-value.\n'
                'Measure | Group A mean | Group B mean | p-value\n'
                'Maternal BMI | 25.67 | 27.67 | <0.0001'
            ),
            'evidence': 'Maternal BMI | 25.67 | 27.67 | <0.0001',
            'question': 'What are the mean maternal BMI values for Group A and Group B?',
            'answer': 'Group A: 25.67; Group B: 27.67.',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_more_than_rwg_second_or',
            'source_chunk': (
                'Targeted extraction drill: table headers are repeated beside each row. The requested column is More than RWG, not Less than RWG.\n'
                'Education | Less than RWG crude OR | Less than RWG 95% CI | Less than RWG p-value | More than RWG crude OR | More than RWG 95% CI | More than RWG p-value\n'
                'Higher certificate | 0.58 | 0.42-0.79 | 0.001 | 1.12 | 0.93-1.34 | 0.210'
            ),
            'evidence': 'Higher certificate | 0.58 | 0.42-0.79 | 0.001 | 1.12 | 0.93-1.34 | 0.210',
            'question': "What is the crude odds ratio for 'more than the recommended weight gain' in women with higher certificate education compared to those with less than 5 GCSEs?",
            'answer': '1.12',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_more_than_rwg_not_first_or',
            'source_chunk': (
                'Targeted extraction drill: the same row reports two outcomes, each with OR, CI, and p-value.\n'
                'Characteristic | Less than recommended weight gain OR | Less than recommended weight gain CI | Less than recommended weight gain p-value | More than recommended weight gain OR | More than recommended weight gain CI | More than recommended weight gain p-value\n'
                'University degree | 0.74 | 0.60-0.91 | 0.004 | 0.97 | 0.82-1.15 | 0.730'
            ),
            'evidence': 'University degree | 0.74 | 0.60-0.91 | 0.004 | 0.97 | 0.82-1.15 | 0.730',
            'question': "What is the OR for more than recommended weight gain for the University degree group?",
            'answer': '0.97',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_compared_to_two_percentages',
            'source_chunk': (
                'Targeted extraction drill: include both reported comparison percentages when the question asks about both groups.\n'
                'For example, within a 250 m buffer zone, those in the greenest quintile were 12% less likely to report symptoms, compared with those within quintiles 2 and 3 who were 21% less likely to report symptoms.'
            ),
            'evidence': 'greenest quintile were 12% less likely; quintiles 2 and 3 were 21% less likely',
            'question': 'What percentage reduction was reported for the greenest quintile compared to those in quintiles 2 and 3 within a 250 m buffer zone?',
            'answer': 'Greenest quintile: 12%; quintiles 2 and 3: 21%.',
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_relationship_no_coefficients',
            'source_chunk': (
                'Targeted extraction drill: for relationship questions, state the direction and do not include coefficients unless asked.\n'
                'Higher consumption of fruit at 12 and 24 months was associated with a lower health score at 24 months (model coefficient (95% CI): -0.18 (-0.30, -0.06) and -0.14 (-0.26, -0.02), respectively).'
            ),
            'evidence': 'Higher consumption of fruit at 12 and 24 months was associated with a lower health score at 24 months.',
            'question': 'What was the relationship between fruit consumption at 12 and 24 months and health scores at 24 months?',
            'answer': 'Higher consumption of fruit at 12 and 24 months was associated with a lower health score at 24 months.',
            'response_rule': (
                'Response rule: State the requested relationship or finding in words using only the context. '
                'Keep numbers that identify requested groups or timepoints, but do not add model coefficients, confidence intervals, p-values, or adjacent numeric details unless the question asks for them.'
            ),
        },
        {
            'source_chunk_id': 'targeted_extraction_drill_binary_code_ignore_source_codes',
            'source_chunk': (
                'Targeted extraction drill: answer the binary variable code, not the source-category codes listed after it.\n'
                'The Ethflag variable is binary, coded 1 "Study group A" if the respondent/cohort ethnic group variables were coded as "Category B", "Category C", "Category D" or "Category E"; otherwise coded 0.'
            ),
            'evidence': 'The Ethflag variable is binary, coded 1 "Study group A" if the respondent/cohort ethnic group variables were coded as "Category B", "Category C", "Category D" or "Category E"; otherwise coded 0.',
            'question': 'How is the Ethflag variable coded for Study group A?',
            'answer': '1',
        },

        {
            'source_chunk_id': 'targeted_drill_absent_adjusted_rr_ignore_seroprevalence',
            'source_chunk': (
                'Targeted abstention drill: the requested adjusted risk ratio is absent even though related percentages are present.\n'
                'VZV seroprevalence was significantly lower among South Asian women. The excerpt reports 98% seroprevalence among White British women and 95% among UK-born South Asian women. No adjusted risk ratio for South Asian women born in South Asia compared with White British women is reported.'
            ),
            'evidence': 'No adjusted risk ratio for South Asian women born in South Asia compared with White British women is reported.',
            'question': 'What was the adjusted risk ratio for VZV seroprevalence among South Asian women born in South Asia compared to White British women?',
            'answer': 'The provided context does not contain the requested information.',
            'response_rule': 'Response rule: If the exact adjusted risk ratio is absent, answer exactly: "The provided context does not contain the requested information." and stop. Do not provide related seroprevalence percentages.',
        },
        {
            'source_chunk_id': 'targeted_drill_age_6_7_not_age_4_5_or',
            'source_chunk': (
                'Targeted extraction drill: choose the requested age band, not the neighboring age band.\n'
                'Comparison | Age 4-5 below expected standards OR | Age 4-5 95% CI | Age 6-7 below expected standards OR | Age 6-7 95% CI | SEN support percentage\n'
                'Congenital heart disease vs healthy peers | 2.36 | 1.80-3.08 | 1.98 | 1.42-2.77 | 33%'
            ),
            'evidence': 'Age 6-7 below expected standards OR: 1.98',
            'question': 'What is the odds ratio for children with congenital heart disease being below the expected educational standards at age 6-7 years compared to healthy children?',
            'answer': '1.98',
        },
        {
            'source_chunk_id': 'targeted_drill_sen_support_not_attainment_percentages',
            'source_chunk': (
                'Targeted extraction drill: answer the requested support percentage, not adjacent attainment percentages.\n'
                'Group | Received some form of SEN support | Age 4-5 achieved expected attainment | Comparator achieved expected attainment\n'
                'Children with congenital heart disease | 33% | 52% | 17%'
            ),
            'evidence': 'Children with congenital heart disease | Received some form of SEN support: 33%',
            'question': 'What percentage of children with congenital heart disease received some form of special educational needs support?',
            'answer': '33%',
        },
        {
            'source_chunk_id': 'targeted_drill_first_year_rr_not_age_five',
            'source_chunk': (
                'Targeted extraction drill: choose the requested follow-up period exactly.\n'
                'Exposure | First year of life relative risk | First year 95% CI | By age five years relative risk | By age five years 95% CI\n'
                'Maternal obesity grade 2-3 | 1.12 | 1.05 to 1.20 | 1.18 | 1.11 to 1.25'
            ),
            'evidence': 'Maternal obesity grade 2-3 | first year of life relative risk 1.12 (95% CI 1.05 to 1.20)',
            'question': 'What was the relative risk of infection in children born to mothers with obesity grade 2-3 during the first year of life compared to children born to mothers of healthy weight?',
            'answer': '1.12 (95% CI 1.05 to 1.20)',
        },
        {
            'source_chunk_id': 'targeted_drill_hdp_all_pakistani_not_subtypes',
            'source_chunk': (
                'Targeted extraction drill: answer the all-births row, not adjacent disorder subtypes.\n'
                'Outcome | All Pakistani births unadjusted OR | 95% CI | Gestational hypertension unadjusted OR | Pre-eclampsia unadjusted OR\n'
                'Hypertensive disorders of pregnancy compared with White British births | 0.74 | 0.61 to 0.90 | 1.51 | 1.35'
            ),
            'evidence': 'All Pakistani births unadjusted OR for hypertensive disorders of pregnancy: 0.74 (0.61 to 0.90)',
            'question': 'What is the unadjusted odds ratio for hypertensive disorders of pregnancy in all Pakistani births compared to White British births?',
            'answer': '0.74 (0.61 to 0.90)',
        },
        {
            'source_chunk_id': 'targeted_drill_cutoff_12th_birthday_not_21_years',
            'source_chunk': (
                'Targeted extraction drill: choose the requested cut-off criterion, not the adjacent maximum follow-up age.\n'
                'Children were followed in linked records up to 21 years where available. For this analysis, the cut-off point used for health records was the child\'s 12th birthday.'
            ),
            'evidence': 'the cut-off point used for health records was the child\'s 12th birthday',
            'question': 'What was the cut-off point used for health records?',
            'answer': "The child's 12th birthday.",
        },
        {
            'source_chunk_id': 'targeted_drill_chips_increased_0_5_to_7_0',
            'source_chunk': (
                'Targeted extraction drill: describe the requested change and ignore IQRs and adjacent food rows.\n'
                'Food item | Median frequency at 12 months | IQR at 12 months | Median frequency at 18 months | IQR at 18 months\n'
                'Chips, roast, and potato shapes | 0.5 portions/week | 0.0-2.5 | 7.0 portions/week | 0.0-14.0\n'
                'Fruit juice | 0.0 portions/week | 0.0-7.0 | 7.0 portions/week | 0.0-7.0'
            ),
            'evidence': 'Chips, roast, and potato shapes increased from 0.5 portions/week at 12 months to 7.0 portions/week at 18 months.',
            'question': 'How did the frequency of intake of chips, roast, and potato shapes change from 12 to 18 months?',
            'answer': 'The median frequency increased from 0.5 portions per week at 12 months to 7.0 portions per week at 18 months.',
            'response_rule': 'Response rule: State the requested change only. Include the two medians and timepoints; do not include IQRs or adjacent food rows.',
        },
        {
            'source_chunk_id': 'targeted_drill_sad_not_happy_pandemic',
            'source_chunk': (
                'Targeted extraction drill: answer the requested emotion, not the adjacent emotion.\n'
                "Emotion | Pre-pandemic all of the time | During pandemic all of the time\n"
                "Happy | 36.2% | 53.2%\n"
                "Sad | 4.1% | 1.4%"
            ),
            'evidence': "Sad all of the time decreased from 4.1% pre-pandemic to 1.4% during the pandemic.",
            'question': "What was the change in the percentage of children who reported feeling sad 'all of the time' during the pandemic compared to pre-pandemic?",
            'answer': "The percentage decreased from 4.1% pre-pandemic to 1.4% during the pandemic.",
        },
        {
            'source_chunk_id': 'targeted_drill_vegetarian_not_adjacent_restriction',
            'source_chunk': (
                'Targeted extraction drill: answer the requested row only, not the adjacent dietary row.\n'
                'Characteristic | START percentage | BiB percentage\n'
                'Vegetarian participants | 36.4% | 1.3%\n'
                'Any dietary restriction | 40.9% | 31.7%'
            ),
            'evidence': 'Vegetarian participants | START 36.4% | BiB 1.3%',
            'question': 'What was the difference in the proportion of vegetarian participants between the START and BiB studies?',
            'answer': 'The proportion of vegetarian participants was higher in START (36.4%) than in BiB (1.3%).',
        },
        {
            'source_chunk_id': 'targeted_drill_paper_not_variable_metabolite_future',
            'source_chunk': (
                '## Relevant Published Papers\n'
                'Future studies could address whether 4-hydroxyglutamate is also associated with the risk of other cardiovascular or renal conditions outside pregnancy.\n\n'
                '## Relevant Variables\n'
                'Variable: metm2r40499 | Label: 4-hydroxyglutamate [raw] | Topic: metabolomics | Non-missing records: 1980\n'
                'Variable: metm2s40499 | Label: 4-hydroxyglutamate [scaled] | Topic: metabolomics | Non-missing records: 2000'
            ),
            'evidence': 'Future studies could address whether 4-hydroxyglutamate is also associated with the risk of other cardiovascular or renal conditions outside pregnancy.',
            'question': 'What is one potential future direction for research regarding 4-hydroxyglutamate?',
            'answer': 'Future studies could address whether 4-hydroxyglutamate is also associated with the risk of other cardiovascular or renal conditions outside pregnancy.',
            'response_rule': 'Response rule: Answer the published-paper question from the Relevant Published Papers text. Ignore variable metadata and do not list variables, tables, labels, or non-missing record counts.',
        },
        {
            'source_chunk_id': 'targeted_drill_thm_inverse_not_lit_review',
            'source_chunk': (
                'Targeted relationship drill: answer the observed study relationship, not a nearby literature-review sentence.\n'
                'In this study, maternal exposure to THMs during pregnancy was inversely associated with term birth weight for infants of Pakistani-origin. A later sentence notes that previous studies reported mixed directions of association.'
            ),
            'evidence': 'Maternal exposure to THMs during pregnancy was inversely associated with term birth weight for infants of Pakistani-origin.',
            'question': 'What was the observed relationship between maternal exposure to THMs and term birth weight for infants of Pakistani-origin?',
            'answer': 'Maternal exposure to THMs during pregnancy was inversely associated with term birth weight for infants of Pakistani-origin.',
            'response_rule': 'Response rule: State the observed relationship for the requested population. Do not substitute background literature-review statements.',
        },
        {
            'source_chunk_id': 'targeted_drill_home_food_availability_both_timepoints',
            'source_chunk': (
                'Targeted relationship drill: include both requested timepoints and do not answer immigrant-generation differences.\n'
                'The variety and quantity of snack foods and SSBs available in the home were positively associated with dietary intake at 18 and 36 months. There were no differences in home food availability by immigrant generation at 36 months.'
            ),
            'evidence': 'The variety and quantity of snack foods and SSBs available in the home were positively associated with dietary intake at 18 and 36 months.',
            'question': 'What is the relationship between home food availability of snack foods and SSBs and dietary intake in toddlers at 18 and 36 months of age?',
            'answer': 'The variety and quantity of snack foods and SSBs available in the home were positively associated with dietary intake at 18 and 36 months.',
            'response_rule': 'Response rule: State the requested relationship across both requested timepoints. Do not add immigrant-generation findings unless asked.',
        },
        {
            'source_chunk_id': 'targeted_drill_age_of_wonder_goal_not_qual_protocol',
            'source_chunk': (
                "Targeted source-selection drill: answer the cohort goal, not a narrower protocol title.\n"
                "The primary goal of the Born in Bradford's Age of Wonder cohort study is to better understand the interplay of factors that influence health and wellbeing during adolescence and transition into adulthood, and to inform and evaluate interventions to improve them and reduce inequalities. A separate paper is titled a protocol for qualitative longitudinal research."
            ),
            'evidence': "The primary goal is to better understand the interplay of factors that influence health and wellbeing during adolescence and transition into adulthood, and to inform and evaluate interventions to improve them and reduce inequalities.",
            'question': "What is the primary goal of the Born in Bradford's Age of Wonder cohort study?",
            'answer': 'To better understand the interplay of factors that influence health and wellbeing during adolescence and transition into adulthood, and to inform/evaluate interventions to improve them and reduce inequalities.',
            'response_rule': 'Response rule: Answer the cohort-level goal. Do not substitute a narrower protocol title or data-collection method.',
        },
        {
            'source_chunk_id': 'targeted_drill_steepest_gradient_subjective_status_not_social_support',
            'source_chunk': (
                'Targeted extraction drill: choose the named socioeconomic indicator with the steepest gradient.\n'
                'The steepest gradient in relation to child development was observed for subjective social status. Social support was also associated with expected development but did not show the steepest socioeconomic gradient.'
            ),
            'evidence': 'The steepest gradient in relation to child development was observed for subjective social status.',
            'question': 'Which socioeconomic indicator showed the steepest gradient in relation to child development?',
            'answer': 'Subjective social status.',
        },

        {
            'source_chunk_id': 'targeted_drill_sen_support_33_not_159_52',
            'source_chunk': (
                'Targeted wrong-neighbor numeric drill: answer the support percentage, not attainment counts or adjacent percentages.\n'
                'Outcome | Group | Result\n'
                'Higher education | Children with SEN support | 159 (52%)\n'
                'Special educational needs support | Children in exposed group | 33%\n'
                'Special educational needs support | Children in comparison group | 21%'
            ),
            'evidence': 'Special educational needs support | Children in exposed group | 33%',
            'question': 'What percentage of children in the exposed group received special educational needs support?',
            'answer': '33%',
        },
        {
            'source_chunk_id': 'targeted_drill_vegetarian_start_bib_hard',
            'source_chunk': (
                'Targeted wrong-neighbor numeric drill: choose vegetarian prevalence, not the adjacent restriction categories.\n'
                'Dietary restriction | START cohort | BiB cohort\n'
                'Any restriction | 40.9% | 31.7%\n'
                'Vegetarian | 36.4% | 1.3%\n'
                'Allergy-related restriction | 2.5% | 1.8%'
            ),
            'evidence': 'Vegetarian | 36.4% | 1.3%',
            'question': 'What proportion of participants reported being vegetarian in START and BiB?',
            'answer': 'START: 36.4%; BiB: 1.3%.',
        },
        {
            'source_chunk_id': 'targeted_drill_sad_decreased_not_happy_rose',
            'source_chunk': (
                'Targeted direction drill: match the requested emotion and direction exactly.\n'
                'Emotion | Before pandemic | During pandemic | Direction\n'
                'Happy | 42.6% | 54.6% | increased\n'
                'Sad | 23.6% | 17.5% | decreased\n'
                'Scared | 9.2% | 13.8% | increased'
            ),
            'evidence': 'Sad | 23.6% | 17.5% | decreased',
            'question': 'How did the percentage of children feeling sad change during the pandemic?',
            'answer': 'It decreased from 23.6% before the pandemic to 17.5% during the pandemic.',
        },
        {
            'source_chunk_id': 'targeted_drill_gdm_parental_t2d_not_prs',
            'source_chunk': (
                'Targeted genetics extraction drill: choose the requested risk factor, not the adjacent PRS row.\n'
                'Risk factor | GDM odds ratio | 95% CI\n'
                'Parental history of type 2 diabetes | 1.57 | 1.29-1.92\n'
                'Type 2 diabetes PRS per 1 SD | 1.45 | 1.32-1.60\n'
                'Fasting glucose PRS per 1 SD | 1.21 | 1.08-1.35'
            ),
            'evidence': 'Parental history of type 2 diabetes | 1.57 | 1.29-1.92',
            'question': 'What is the odds ratio for gestational diabetes mellitus associated with a parental history of type 2 diabetes?',
            'answer': '1.57 [1.29-1.92].',
            'response_rule': 'Response rule: Match the requested predictor and outcome exactly. Return the parental-history odds ratio and confidence interval only; do not substitute adjacent PRS values.',
        },
        {
            'source_chunk_id': 'targeted_drill_cord_leptin_fully_adjusted_all_birthweights',
            'source_chunk': (
                'Targeted model-column drill: select the fully adjusted association for all birthweights, not interaction rows or unadjusted models.\n'
                'Analysis | Model | Result\n'
                'Cord leptin and childhood adiposity, all birthweights | Minimally adjusted | 1.28 (1.19, 1.38)\n'
                'Cord leptin and childhood adiposity, all birthweights | Fully adjusted | 1.36 (1.26, 1.46)\n'
                'Cord leptin by birthweight interaction | Fully adjusted | 1.01 (0.98, 1.04)'
            ),
            'evidence': 'Cord leptin and childhood adiposity, all birthweights | Fully adjusted | 1.36 (1.26, 1.46)',
            'question': 'What was the fully adjusted association between cord leptin and childhood adiposity for all birthweights?',
            'answer': '1.36 (1.26, 1.46).',
            'response_rule': 'Response rule: Match both the requested model and population. Return only the fully adjusted all-birthweights result, not interaction or adjacent model values.',
        },
        {
            'source_chunk_id': 'targeted_drill_auc_glucose_prs_not_t2d_prs',
            'source_chunk': (
                'Targeted genetics extraction drill: choose the AUC-glucose PRS effect, not type 2 diabetes PRS or BMI category rows.\n'
                'Predictor | Outcome | Effect\n'
                'AUC glucose PRS | Glucose AUC | 0.165 increase per SD\n'
                'Type 2 diabetes PRS | Gestational diabetes | OR 1.45\n'
                'Lower BMI category | Glucose AUC | -0.060 difference'
            ),
            'evidence': 'AUC glucose PRS | Glucose AUC | 0.165 increase per SD',
            'question': 'What was the effect of the AUC glucose PRS on glucose AUC?',
            'answer': 'A 0.165 increase per SD.',
            'response_rule': 'Response rule: Match the requested PRS and outcome exactly. Return only the AUC glucose PRS effect; do not substitute type 2 diabetes PRS or BMI-category values.',
        },
        {
            'source_chunk_id': 'targeted_drill_mdnlr_direction_europeans_higher',
            'source_chunk': (
                'Targeted direction drill: answer the direction of the ethnic comparison exactly.\n'
                'The study reported that pregnant Europeans had an elevated methylation-derived neutrophil-to-lymphocyte ratio (mdNLR) compared with South Asians.\n'
                'A separate sensitivity analysis did not reverse this direction.'
            ),
            'evidence': 'pregnant Europeans had an elevated methylation-derived neutrophil-to-lymphocyte ratio (mdNLR) compared with South Asians',
            'question': 'How did mdNLR compare between pregnant Europeans and South Asians?',
            'answer': 'Pregnant Europeans had elevated mdNLR compared with South Asians.',
        },
        {
            'source_chunk_id': 'targeted_drill_lasso_o_tapes_not_measurement_protocol',
            'source_chunk': (
                'Targeted source drill: answer the specific tape type, not general measurement guidance.\n'
                'The child waist circumference measurements used Lasso-o tapes specially manufactured by Harlow Printing Ltd.\n'
                'The protocol also advised keeping the tape horizontal and snug but not tight.'
            ),
            'evidence': 'used Lasso-o tapes specially manufactured by Harlow Printing Ltd.',
            'question': 'What type of tape was used for the child waist circumference measurements?',
            'answer': 'Lasso-o tapes specially manufactured by Harlow Printing Ltd.',
        },
        {
            'source_chunk_id': 'targeted_drill_calcium_reduces_three_outcomes',
            'source_chunk': (
                'Targeted published-paper finding drill: answer the calcium supplementation finding, not adjacent vitamin D text.\n'
                'Calcium supplementation during pregnancy was associated with reduced risk of high blood pressure, pre-eclampsia, and preterm birth.\n'
                'The next paragraph discussed vitamin D evidence and did not change the calcium finding.'
            ),
            'evidence': 'Calcium supplementation during pregnancy was associated with reduced risk of high blood pressure, pre-eclampsia, and preterm birth.',
            'question': 'What outcomes were reduced by calcium supplementation during pregnancy?',
            'answer': 'High blood pressure, pre-eclampsia, and preterm birth.',
        },

        {
            'source_chunk_id': 'positive_calibration_sen_support_present_answer_33',
            'source_chunk': (
                'Positive answer calibration drill: the requested value is present, so answer instead of abstaining.\n'
                'Children with congenital heart disease received some form of special educational needs support in 33% of cases.\n'
                'Other outcomes in the same paper included being below expected educational standards at age 4-5 years and age 6-7 years.'
            ),
            'evidence': 'received some form of special educational needs support in 33% of cases',
            'question': 'What percentage of children with congenital heart disease received some form of special educational needs support?',
            'answer': '33%',
            'response_rule': 'Response rule: The requested percentage is explicitly present in the context. Answer 33%; do not abstain or substitute educational-standard odds ratios.',
        },
        {
            'source_chunk_id': 'positive_calibration_lasso_tapes_present',
            'source_chunk': (
                'Positive answer calibration drill: the equipment answer is present.\n'
                'Routine clinical measurements of neonatal circumferences in Bradford used Lasso-o tapes specially manufactured by Harlow Printing Ltd.\n'
                'The paragraph also discusses measurement reliability, but the tape type is directly stated.'
            ),
            'evidence': 'used Lasso-o tapes specially manufactured by Harlow Printing Ltd.',
            'question': 'What type of tape measures are used for routine clinical measurements of neonatal circumferences in Bradford?',
            'answer': 'Lasso-o tapes specially manufactured by Harlow Printing Ltd.',
            'response_rule': 'Response rule: The requested tape type is explicitly present. Answer it directly; do not abstain.',
        },
        {
            'source_chunk_id': 'positive_calibration_postload_glucose_model3_present',
            'source_chunk': (
                'Positive answer calibration drill: select the requested model and outcome.\n'
                'Outcome | Model 1 | Model 2 | Model 3\n'
                'Fasting glucose, Pakistani minus White women | 0.12 (0.05 to 0.19) | 0.10 (0.03 to 0.17) | 0.08 (0.01 to 0.15)\n'
                'Postload glucose, Pakistani minus White women | 0.50 (0.41 to 0.59) | 0.43 (0.34 to 0.52) | 0.37 (0.28 to 0.45)'
            ),
            'evidence': 'Postload glucose, Pakistani minus White women | Model 3 | 0.37 (0.28 to 0.45)',
            'question': 'What is the adjusted mean difference in postload glucose levels between Pakistani and White women in model 3?',
            'answer': '0.37 (0.28 to 0.45).',
            'response_rule': 'Response rule: Match postload glucose and model 3 exactly. The value is present; answer it directly and do not abstain.',
        },
        {
            'source_chunk_id': 'positive_calibration_higher_than_a_level_present_21_6',
            'source_chunk': (
                'Positive answer calibration drill: the requested percentage is present in the descriptive table.\n'
                'Maternal education | Percentage\n'
                'Less than 5 GCSEs | 18.4%\n'
                'A-level | 22.7%\n'
                'Higher than A-level | 21.6%'
            ),
            'evidence': 'Higher than A-level | 21.6%',
            'question': 'What percentage of mothers in the study had a higher level of education than A-level?',
            'answer': '21.6%',
            'response_rule': 'Response rule: The requested education category and percentage are present. Answer 21.6%; do not abstain or return adjacent education categories.',
        },
        {
            'source_chunk_id': 'positive_calibration_calcium_finding_present',
            'source_chunk': (
                'Positive answer calibration drill: the relationship is present, so answer it.\n'
                'Calcium supplementation during pregnancy seems to reduce the risk of high blood pressure, pre-eclampsia, and preterm birth.\n'
                'A separate paragraph discusses other supplements but does not alter this calcium finding.'
            ),
            'evidence': 'Calcium supplementation during pregnancy seems to reduce the risk of high blood pressure, pre-eclampsia, and preterm birth.',
            'question': 'What is the relationship between calcium supplementation during pregnancy and the risk of high blood pressure, pre-eclampsia, and preterm birth?',
            'answer': 'Calcium supplementation during pregnancy seems to reduce the risk of high blood pressure, pre-eclampsia, and preterm birth.',
            'response_rule': 'Response rule: The exact relationship is present. Restate it directly; do not abstain.',
        },
        {
            'source_chunk_id': 'positive_calibration_home_food_availability_present',
            'source_chunk': (
                'Positive answer calibration drill: answer the reported relationship, not an absence statement.\n'
                'The variety and quantity of snack foods and sugar-sweetened beverages available in the home were positively associated with toddlers\' dietary intake at 18 and 36 months.\n'
                'The same section also reports no differences in home food availability by immigrant generation at 36 months.'
            ),
            'evidence': 'The variety and quantity of snack foods and sugar-sweetened beverages available in the home were positively associated with toddlers dietary intake at 18 and 36 months.',
            'question': 'What was the relationship between home food availability of snack foods and SSBs and dietary intake in toddlers at 18 and 36 months?',
            'answer': 'The variety and quantity of snack foods and SSBs available in the home were positively associated with dietary intake in toddlers at 18 and 36 months.',
            'response_rule': 'Response rule: The relationship is explicitly present. Answer it directly and do not abstain or switch to immigrant-generation findings.',
        },
        {
            'source_chunk_id': 'positive_calibration_subjective_status_gradient_present',
            'source_chunk': (
                'Positive answer calibration drill: answer the named socioeconomic indicator that is present.\n'
                'Among socioeconomic indicators, subjective social status showed the steepest gradient in relation to child development.\n'
                'Neighbourhood deprivation and social support were discussed as separate indicators.'
            ),
            'evidence': 'subjective social status showed the steepest gradient in relation to child development',
            'question': 'Which socioeconomic indicator showed the steepest gradient in relation to child development?',
            'answer': 'Subjective social status.',
            'response_rule': 'Response rule: The requested indicator is present. Answer subjective social status; do not abstain or substitute social support.',
        },
        {
            'source_chunk_id': 'positive_calibration_thm_pakistani_inverse_present',
            'source_chunk': (
                'Positive answer calibration drill: answer the relationship stated in the paper excerpt.\n'
                'Maternal exposure to trihalomethanes (THMs) during pregnancy was inversely associated with term birth weight for infants of Pakistani origin.\n'
                'No similar inverse association was observed for every subgroup in the adjacent text.'
            ),
            'evidence': 'THMs during pregnancy was inversely associated with term birth weight for infants of Pakistani origin',
            'question': 'What was the observed relationship between maternal exposure to THMs and term birth weight for infants of Pakistani-origin?',
            'answer': 'Maternal exposure to THMs during pregnancy was inversely associated with term birth weight for infants of Pakistani-origin.',
            'response_rule': 'Response rule: The exact relationship is present. Answer it directly; do not abstain.',
        },
        {
            'source_chunk_id': 'positive_calibration_time_weighted_thm_haa_present',
            'source_chunk': (
                'Positive answer calibration drill: select the time-weighting method, not activity-exposure calculations.\n'
                'Time-weighted average concentrations were calculated based on the proportion of the whole pregnancy or trimester falling into each month for THMs or quarter for HAAs.\n'
                'Another paragraph describes showering and bathing exposure routes but not this time-weighting calculation.'
            ),
            'evidence': 'based on the proportion of the whole pregnancy or trimester falling into each month for THMs or quarter for HAAs',
            'question': 'How were time-weighted average concentrations of THMs and HAAs calculated for each pregnancy or trimester?',
            'answer': 'They were calculated based on the proportion of the whole pregnancy or trimester falling into each month for THMs or quarter for HAAs.',
            'response_rule': 'Response rule: The requested calculation method is present. Answer it directly; do not switch to showering, bathing, or swimming exposure calculations.',
        },
        {
            'source_chunk_id': 'positive_calibration_age_of_wonder_goal_present',
            'source_chunk': (
                'Positive answer calibration drill: answer the cohort goal that is explicitly stated.\n'
                "The primary goal of the Born in Bradford's Age of Wonder cohort study is to better understand the interplay of factors that influence health and wellbeing during adolescence and the transition into adulthood, and to inform and evaluate interventions to improve them and reduce inequalities.\n"
                'A recruitment paragraph also mentions including young people who were not in the original BiB cohort.'
            ),
            'evidence': "to better understand the interplay of factors that influence health and wellbeing during adolescence and the transition into adulthood, and to inform and evaluate interventions to improve them and reduce inequalities",
            'question': "What is the primary goal of the Born in Bradford's Age of Wonder cohort study?",
            'answer': 'To better understand the interplay of factors that influence health and wellbeing during adolescence and the transition into adulthood, and to inform/evaluate interventions to improve them and reduce inequalities.',
            'response_rule': 'Response rule: The primary goal is present. Answer the goal directly; do not substitute recruitment/generalizability details.',
        },
        {
            'source_chunk_id': 'positive_calibration_work_package_5_present',
            'source_chunk': (
                'Positive answer calibration drill: answer the requested work package when it is present.\n'
                'Work Package 1 focuses on co-production and determinants of health.\n'
                'Work Package 5 aims to maximise engagement with and ownership of the project with young people aged 16+.\n'
                'Work Package 6 concerns data linkage and dissemination.'
            ),
            'evidence': 'Work Package 5 aims to maximise engagement with and ownership of the project with young people aged 16+.',
            'question': 'What is the main goal of Work Package 5 in the Age of Wonder programme?',
            'answer': 'To maximise engagement with and ownership of the project with young people aged 16+.',
            'response_rule': 'Response rule: Work Package 5 is explicitly present. Answer its goal directly; do not abstain or give Work Package 1 objectives.',
        },
        {
            'source_chunk_id': 'positive_calibration_auc_glucose_prs_gdm_present',
            'source_chunk': (
                'Positive answer calibration drill: select the AUC glucose PRS effect on gestational diabetes risk, not the type 2 diabetes PRS row.\n'
                'Predictor | Outcome | Effect\n'
                'AUC glucose PRS, per 1 SD increase | Risk of gestational diabetes in South Asian women | 0.165 increase\n'
                'Type 2 diabetes PRS, per 1 SD increase | Risk of gestational diabetes in lower BMI groups | stronger association\n'
                'Fasting glucose PRS, per 1 SD increase | Fasting glucose | 0.080 increase'
            ),
            'evidence': 'AUC glucose PRS, per 1 SD increase | Risk of gestational diabetes in South Asian women | 0.165 increase',
            'question': 'What is the effect of a 1 SD increase in the AUC glucose PRS on the risk of gestational diabetes in South Asian women?',
            'answer': 'A 1 SD increase in the AUC glucose PRS was associated with a 0.165 increase in the risk of gestational diabetes.',
            'response_rule': 'Response rule: The requested AUC glucose PRS effect is present. Answer 0.165 increase; do not substitute type 2 diabetes PRS or BMI-category findings.',
        },
        {
            'source_chunk_id': 'positive_calibration_sad_all_time_4_1_to_1_4_present',
            'source_chunk': (
                'Positive answer calibration drill: match the emotion and frequency exactly.\n'
                'Feeling happy all of the time increased from 36.2% pre-pandemic to 53.2% during the pandemic.\n'
                "Feeling sad all of the time decreased from 4.1% pre-pandemic to 1.4% during the pandemic.\n"
                'Feeling worried some of the time changed in a different direction.'
            ),
            'evidence': 'Feeling sad all of the time decreased from 4.1% pre-pandemic to 1.4% during the pandemic.',
            'question': "What was the change in the percentage of children who reported feeling sad 'all of the time' during the pandemic compared to pre-pandemic?",
            'answer': "It decreased from 4.1% pre-pandemic to 1.4% during the pandemic.",
            'response_rule': 'Response rule: Match sad and all of the time exactly. Answer 4.1% to 1.4%; do not use the happy percentages.',
        },
        {
            'source_chunk_id': 'positive_calibration_vegetarian_start_bib_exact_present',
            'source_chunk': (
                'Positive answer calibration drill: match vegetarian, not any restriction.\n'
                'Restriction type | START | BiB\n'
                'Any dietary restriction | 40.9% | 31.7%\n'
                'Vegetarian | 36.4% | 1.3%\n'
                'Religious restriction | 12.0% | 18.4%'
            ),
            'evidence': 'Vegetarian | 36.4% | 1.3%',
            'question': 'What was the difference in the proportion of vegetarian participants between the START and BiB studies?',
            'answer': 'The proportion of vegetarian participants was higher in START (36.4%) than in BiB (1.3%).',
            'response_rule': 'Response rule: The vegetarian row is present. Answer 36.4% for START and 1.3% for BiB; do not use any-dietary-restriction percentages.',
        },

        {
            'source_chunk_id': 'wrong_neighbor_gdm_higher_not_macrosomia_lower',
            'source_chunk': (
                'Hard wrong-neighbor drill: match the requested outcome, not the adjacent outcome.\n'
                'Gestational diabetes prevalence was higher in women of Pakistani origin.\n'
                'Macrosomia was lower in Pakistani compared to White British women at all BMI levels.\n'
                'Both findings appeared in the same paragraph.'
            ),
            'evidence': 'Gestational diabetes prevalence was higher in women of Pakistani origin.',
            'question': 'What is the difference in the prevalence of gestational diabetes between Pakistani and White British women?',
            'answer': 'Gestational diabetes prevalence was higher in women of Pakistani origin.',
            'response_rule': 'Response rule: Match the requested outcome gestational diabetes exactly. Do not answer with the adjacent macrosomia finding.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_age_questionnaire_12_15_not_cohort_11_16',
            'source_chunk': (
                'Hard wrong-neighbor drill: match the action being asked about, not the broader cohort age.\n'
                'The Age of Wonder cohort includes young people aged 11-16 years.\n'
                'Young people aged 12-15 years are being asked to complete questionnaires.\n'
                'Young people aged 16+ are invited to other engagement activities.'
            ),
            'evidence': 'Young people aged 12-15 years are being asked to complete questionnaires.',
            'question': 'What is the age range of the young people being asked to complete questionnaires in the Born in Bradford Age of Wonder study?',
            'answer': '12-15 years old.',
            'response_rule': 'Response rule: Match being asked to complete questionnaires. Answer 12-15 years, not the broader cohort age range.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_bullying_more_than_half_not_variable_metadata',
            'source_chunk': (
                'Hard source-priority drill: answer the published-paper finding, not variable metadata.\n'
                'Relevant Published Papers: More than half of BiB children reported being bullied during primary school.\n'
                'Relevant Variables: awb8_2_bullied has categories 1 to 6 and 1701 non-missing records.\n'
                'Relevant Variables: psychdFbully has 15368 non-missing records.'
            ),
            'evidence': 'More than half of BiB children reported being bullied during primary school.',
            'question': 'What percentage of BiB children reported being bullied during primary school?',
            'answer': 'More than half of BiB children reported being bullied during primary school.',
            'response_rule': 'Response rule: This is a published-paper finding question. Ignore variable categories and non-missing records; answer the paper finding.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_core_funding_wellcome_and_nihr',
            'source_chunk': (
                'Hard multi-value drill: return both requested funders from the same funding statement.\n'
                'The Born in Bradford study receives core infrastructure funding from the Wellcome Trust and the National Institute for Health Research (NIHR).\n'
                'A separate project grant was provided by the Medical Research Council.'
            ),
            'evidence': 'core infrastructure funding from the Wellcome Trust and the National Institute for Health Research (NIHR)',
            'question': 'Who provided the core infrastructure funding for the Born in Bradford (BiB) study?',
            'answer': 'The Wellcome Trust and the National Institute for Health Research (NIHR).',
            'response_rule': 'Response rule: Return all funders in the core infrastructure funding statement. Do not omit NIHR and do not substitute project grants.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_rsv_any_infection_86_not_sibling_8',
            'source_chunk': (
                'Hard wrong-neighbor numeric drill: match the requested infection outcome, not an antibody covariate.\n'
                'By age 2 years, 86% of sampled children had serological evidence of any RSV infection.\n'
                'Children with 3 siblings had 8% higher maternally-derived antibody concentrations.\n'
                'The sibling antibody result is not the infection percentage.'
            ),
            'evidence': '86% of sampled children had serological evidence of any RSV infection',
            'question': 'What percentage of sampled children had serological evidence of any RSV infection by age 2 years?',
            'answer': '86%.',
            'response_rule': 'Response rule: Match any RSV infection by age 2 years. Answer 86%; do not use the sibling antibody percentage.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_maternal_bmi_offspring_adiposity_not_baseline_ethnicity',
            'source_chunk': (
                'Hard relationship drill: answer the exposure-outcome relationship, not baseline ethnic differences.\n'
                'Children of mothers with higher BMI were larger and more adipose, and these positive associations were stronger in Pakistani children than in white British children.\n'
                'Elsewhere the paper notes that Pakistani mothers had lower BMI and higher fasting glucose than white British mothers.'
            ),
            'evidence': 'Children of mothers with higher BMI were larger and more adipose, and these positive associations were stronger in Pakistani children than in white British children.',
            'question': 'What was the relationship between maternal BMI and offspring adiposity in Pakistani and white British children?',
            'answer': 'Children of mothers with higher BMI were larger and more adipose, with stronger positive associations in Pakistani children than in white British children.',
            'response_rule': 'Response rule: Match maternal BMI and offspring adiposity. Do not answer with baseline ethnic differences in maternal BMI or glucose.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_midpregnancy_pa_triglycerides_not_glucose',
            'source_chunk': (
                'Hard outcome drill: match triglycerides, not glucose.\n'
                'In white British women, more mid-pregnancy physical activity was associated with lower triglyceride levels.\n'
                'In a nearby sentence, physical activity was also discussed in relation to fasting and postload glucose.'
            ),
            'evidence': 'more mid-pregnancy physical activity was associated with lower triglyceride levels',
            'question': 'What was the association between mid-pregnancy physical activity and triglyceride levels in white British women?',
            'answer': 'More mid-pregnancy physical activity was associated with lower triglyceride levels in white British women.',
            'response_rule': 'Response rule: Match triglyceride levels exactly. Do not answer with fasting or postload glucose findings.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_subjective_status_not_parental_education',
            'source_chunk': (
                'Hard socioeconomic-indicator drill: choose the named steepest gradient indicator.\n'
                'Subjective social status showed the steepest gradient in relation to child development.\n'
                'Higher parental education was also associated with child development, but it was not the steepest gradient.'
            ),
            'evidence': 'Subjective social status showed the steepest gradient in relation to child development.',
            'question': 'Which socioeconomic indicator showed the steepest gradient in relation to child development?',
            'answer': 'Subjective social status.',
            'response_rule': 'Response rule: Match the indicator described as the steepest gradient. Answer subjective social status, not parental education.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_cord_leptin_all_birthweights_fully_adjusted_not_interaction',
            'source_chunk': (
                'Hard model/population drill: match all birthweights and fully adjusted model.\n'
                'Analysis | Population | Model | Ratio of geometric means\n'
                'Pakistani vs White British cord leptin | All birthweights | Fully adjusted | 1.36 (1.26, 1.46)\n'
                'Pakistani vs White British cord leptin | Birthweight interaction | Fully adjusted | 1.01 (0.93, 1.10)\n'
                'Pakistani vs White British cord leptin | All birthweights | Unadjusted | 1.28 (1.19, 1.38)'
            ),
            'evidence': 'All birthweights | Fully adjusted | 1.36 (1.26, 1.46)',
            'question': 'What is the ratio of geometric means comparing Pakistani with White British infants for cord leptin levels in the fully adjusted model for all birthweights?',
            'answer': '1.36 (1.26, 1.46).',
            'response_rule': 'Response rule: Match all birthweights and fully adjusted model. Do not use birthweight interaction or unadjusted values.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_auc_glucose_prs_gdm_risk_not_t2d_prs',
            'source_chunk': (
                'Hard PRS drill: match the requested PRS and outcome together.\n'
                'Predictor | Outcome | Effect\n'
                'AUC glucose PRS, per 1 SD increase | Risk of gestational diabetes in South Asian women | 0.165 increase\n'
                'Type 2 diabetes PRS, per 1 SD increase | Gestational diabetes mellitus | higher risk in lower BMI categories\n'
                'AUC glucose PRS, per 1 SD increase | Glucose AUC | 0.165 increase'
            ),
            'evidence': 'AUC glucose PRS, per 1 SD increase | Risk of gestational diabetes in South Asian women | 0.165 increase',
            'question': 'What is the effect of a 1 SD increase in the AUC glucose PRS on the risk of gestational diabetes in South Asian women?',
            'answer': 'A 1 SD increase in the AUC glucose PRS is associated with a 0.165 increase in the risk of gestational diabetes.',
            'response_rule': 'Response rule: Match AUC glucose PRS and risk of gestational diabetes. Do not substitute type 2 diabetes PRS findings.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_sad_all_time_not_happy_all_time',
            'source_chunk': (
                'Hard emotion/frequency drill: match emotion and frequency exactly.\n'
                'Feeling happy all of the time rose from 36.2% pre-pandemic to 53.2% during the pandemic.\n'
                'Feeling sad all of the time decreased from 4.1% pre-pandemic to 1.4% during the pandemic.\n'
                'Feeling sad some of the time followed a different pattern.'
            ),
            'evidence': 'Feeling sad all of the time decreased from 4.1% pre-pandemic to 1.4% during the pandemic.',
            'question': "What was the change in the percentage of children who reported feeling sad 'all of the time' during the pandemic compared to pre-pandemic?",
            'answer': 'It decreased from 4.1% pre-pandemic to 1.4% during the pandemic.',
            'response_rule': 'Response rule: Match sad and all of the time. Do not use happy all of the time percentages.',
        },
        {
            'source_chunk_id': 'wrong_neighbor_vegetarian_start_bib_not_primiparous',
            'source_chunk': (
                'Hard row-label drill: match vegetarian, not primiparous.\n'
                'Characteristic | START | BiB\n'
                'Primiparous women | 40.9% | 31.7%\n'
                'Vegetarian participants | 36.4% | 1.3%\n'
                'Participants with any dietary restriction | 45.0% | 32.0%'
            ),
            'evidence': 'Vegetarian participants | 36.4% | 1.3%',
            'question': 'What was the difference in the proportion of vegetarian participants between the START and BiB studies?',
            'answer': 'The proportion of vegetarian participants was higher in START (36.4%) than in BiB (1.3%).',
            'response_rule': 'Response rule: Match the row label vegetarian participants. Do not use primiparous or any-dietary-restriction rows.',
        },
        {
            'source_chunk_id': 'targeted_drill_abstain_no_age_range_aow',
            'source_chunk': (
                'Targeted abstention drill: the context does not provide the requested age range.\n'
                'The study will collect repeated data from existing BiB participants and their peers, with approximately 30,000 adolescents. This excerpt does not state the age range of young people being asked to complete questionnaires.'
            ),
            'evidence': 'This excerpt does not state the age range of young people being asked to complete questionnaires.',
            'question': 'What is the age range of the young people being asked to complete questionnaires in the Born in Bradford Age of Wonder study?',
            'answer': 'The provided context does not contain the requested information.',
            'response_rule': 'Response rule: If the exact age range is absent, answer exactly: "The provided context does not contain the requested information." and stop. Do not infer from the word adolescents or from sample size.',
        },
        {
            'source_chunk_id': 'targeted_drill_abstain_core_funding_absent',
            'source_chunk': (
                'Targeted abstention drill: the context includes acknowledgements but not the requested core infrastructure funders.\n'
                'The excerpt thanks schools, families, and study staff. It does not state who provided the core infrastructure funding for the Born in Bradford study.'
            ),
            'evidence': 'It does not state who provided the core infrastructure funding for the Born in Bradford study.',
            'question': 'Who provided the core infrastructure funding for the Born in Bradford study?',
            'answer': 'The provided context does not contain the requested information.',
            'response_rule': 'Response rule: If the requested funding source is absent, answer exactly: "The provided context does not contain the requested information." and stop. Do not name funders from memory.',
        },
    ]

    for row in drill_rows:
        row.setdefault('response_rule', TARGETED_NUMERIC_RESPONSE_RULE)
        row['source_chunk'] = build_anchor_structured_context(
            question=row.get('question', ''),
            answer=row.get('answer', ''),
            context=row.get('source_chunk', ''),
            evidence=row.get('evidence', ''),
            header_hint=infer_table_header_hint(row.get('source_chunk', ''), row.get('evidence', '')),
        )
        row['system'] = cfg.production_system_prompt
        row['source_metadata'] = {'source': 'targeted_extraction_drills'}

    print(f'Synthetic targeted extraction drill rows: {len(drill_rows)}')
    return drill_rows


def build_targeted_stage2_augmentation_rows(raw_rows: List[Dict]) -> List[Dict]:
    if not cfg.targeted_stage2_augmentation:
        print('Targeted Stage 2 augmentation disabled.')
        return []

    augmented_rows = build_auto_targeted_augmentation_rows(raw_rows)
    if cfg.targeted_stage2_augmentation_include_drills:
        augmented_rows.extend(build_targeted_extraction_drill_rows())

    print(f'Total targeted Stage 2 augmentation rows: {len(augmented_rows)}')
    if augmented_rows:
        print('Sample targeted augmentation question:', augmented_rows[0]['question'])
        print('Sample targeted augmentation answer:', augmented_rows[0]['answer'])
    return augmented_rows


In [ ]:
NUMERIC_SENSITIVE_QUESTION_RE = re.compile(
    r'\b(?:odds ratio|or|percentage|percent|proportion|count|number|value|values|cutoff|cut-off|coefficient|confidence interval|ci|p-value|sample size|mean|median|bmi|z-score|quintile|buffer|years?|months?|dates?|coded|code|classified|units?)\b',
    flags=re.IGNORECASE,
)


def is_numeric_sensitive_record(record: Dict) -> bool:
    return bool(
        extract_numeric_tokens(record.get('answer', ''))
        or NUMERIC_SENSITIVE_QUESTION_RE.search(record.get('question', '') or '')
    )


def build_stage2_records(raw_rows: List[Dict], oversample_numeric: bool = False) -> List[Dict]:
    records = []
    skipped_missing_fields = 0
    skipped_render_errors = 0
    skip_examples = []

    for row_index, row in enumerate(raw_rows):
        source_messages = row.get('messages') if isinstance(row.get('messages'), list) else []
        answer_text = row_answer_text(row, source_messages)
        question_text = row_question_text(row, source_messages)
        context_text = row_context_text(row, source_messages)
        messages = build_production_messages(row)
        if not messages:
            skipped_missing_fields += 1
            if len(skip_examples) < 3:
                skip_examples.append({
                    'row_index': row_index,
                    'keys': sorted(row.keys()),
                    'has_answer': bool(answer_text),
                    'has_question': bool(question_text),
                    'has_context': bool(context_text),
                    'answer_preview': answer_text[:120],
                    'question_preview': question_text[:120],
                    'context_preview': context_text[:120],
                })
            continue

        try:
            prompt_text, full_text = render_prompt_and_answer(messages)
        except Exception:
            skipped_render_errors += 1
            continue

        source_metadata = row.get('source_metadata') or {}
        if not isinstance(source_metadata, dict):
            source_metadata = {}
        records.append({
            'messages': messages,
            'prompt_text': prompt_text,
            'full_text': full_text,
            'answer': messages[-1]['content'],
            'evidence': first_nonempty_field(row, ['evidence', 'supporting_evidence', 'citation', 'quote']),
            'question': question_text,
            'context': context_text,
            'source': source_metadata.get('pdf_file') or source_metadata.get('source') or row.get('source_chunk_id', 'unknown'),
        })

    if skipped_missing_fields:
        print(f'Skipped Stage 2 rows missing answer/question/context: {skipped_missing_fields}')
        print('Skipped Stage 2 row examples:', skip_examples)
    if skipped_render_errors:
        print(f'Skipped Stage 2 rows due to prompt rendering errors: {skipped_render_errors}')

    if oversample_numeric and cfg.numeric_sensitive_oversample_copies > 1:
        numeric_records = [record for record in records if is_numeric_sensitive_record(record)]
        extra_records = []
        for copy_index in range(cfg.numeric_sensitive_oversample_copies - 1):
            for record in numeric_records:
                duplicated = dict(record)
                duplicated['source'] = f"{record.get('source', 'unknown')}#numeric_copy_{copy_index + 1}"
                extra_records.append(duplicated)
        records = records + extra_records
        print(
            f'Numeric-sensitive Stage 2 oversampling: base={len(records) - len(extra_records)}, '
            f'numeric={len(numeric_records)}, added={len(extra_records)}, total={len(records)}'
        )

    return records

stage2_targeted_aug_raw = build_targeted_stage2_augmentation_rows(sft_train_raw)
stage2_train_input_raw = sft_train_raw + stage2_targeted_aug_raw

stage2_train_records = build_stage2_records(stage2_train_input_raw, oversample_numeric=True)
stage2_dev_records = build_stage2_records(sft_dev_raw, oversample_numeric=False)

if not stage2_train_records or not stage2_dev_records:
    raise ValueError('No Stage 2 records were built. Check that SFT rows include answer plus question/context or messages.')

print('Stage 2 base train rows:', len(sft_train_raw))
print('Stage 2 targeted augmentation rows:', len(stage2_targeted_aug_raw))
print('Stage 2 train records after augmentation/oversampling:', len(stage2_train_records))
print('Stage 2 dev records:', len(stage2_dev_records))
print('Stage 2 numeric-sensitive train records:', sum(is_numeric_sensitive_record(record) for record in stage2_train_records))
print('Stage 2 numeric-sensitive dev records:', sum(is_numeric_sensitive_record(record) for record in stage2_dev_records))
print('Sample question:', stage2_train_records[0]['question'])
print('Sample evidence:', stage2_train_records[0]['evidence'])
print('Sample prompt tokens:', len(tokenizer(stage2_train_records[0]['prompt_text'], add_special_tokens=False)['input_ids']))
print('Sample prompt head:', stage2_train_records[0]['prompt_text'][:700])
if stage2_targeted_aug_raw:
    print('Sample targeted augmented prompt head:', build_production_messages(stage2_targeted_aug_raw[0])[0]['content'][:900])

stage2_train_raw_ds = Dataset.from_list(stage2_train_records)
stage2_dev_raw_ds = Dataset.from_list(stage2_dev_records)


In [ ]:
# Fixed sanity set captured from the faithfulness evaluation on 2026-05-22.
# These rows are for sense checks only; they are not appended to Stage 2 training data.
EVALUATION_SANITY_SOURCE = 'faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json'
EVALUATION_SANITY_RAW = [
  {
    "query_id": "q_1",
    "question": "In the paper \"Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in Bradford dataset\" (2024), what is the relationship between prenatal alcohol exposure and early education outcomes?",
    "reference_answer": "The study shows a significant association between prenatal alcohol exposure and adverse outcomes in a child’s early education.",
    "retrieved_context": "## Exact Registry Matches\n\n[PAPER TITLE NOT IN REGISTRY: 'Associations between prenatal alcohol exposure and early education outcomes a ma' not found in BiB paper index]\n\n## Relevant Published Papers\n\n**Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B** (2024) — \nTitle: Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B\nYear: 2024\nSource: full-text PDF\n\nic Health, University of Salford, Salford, UK; 2Psychology\nDepartment Bradford University, Bradford, UK and 3Centre for Applied Education Research, Bradford, UK\nAbstract\nPrenatal alcohol exposure (PAE) is associated with cognitive, behavi\n\n**Associations between prenatal alcohol exposure and early education outcomes: a matched controls study using the born in ** (2024) — McCarthy Robyn, Cook Penny A, Pink Joshua, Eddy Lucy H\nAbstract: Prenatal alcohol exposure (PAE) is associated with cognitive, behavioural, and developmental impairments throughout the lifespan of affected individuals, but there is limited evidence on how early this impact can be identified through routinely collected childhood data. This paper explores the relationship between PAE and the Early Years Foundation Stage Profile (EYFSP), a statutory teacher-based summative assessment of early development in relation to learning goals. This analysis uses the Born in Bradford dataset, a UK based cohort (n = 13,959; full dataset), which collected self-r\n\n**Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B** (2024) — \nTitle: Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B\nYear: 2024\nSource: full-text PDF\n\nlation in Bradford, with a higher proportion\nof younger people.\nReported South Asian ethnicity is associated in the UK with\nabstinence from alcohol.24,40 In our dataset, for each drinking\npattern, more than 80% of mothers who report that \n\n**Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B** (2024) — \nTitle: Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B\nYear: 2024\nSource: full-text PDF\n\nPublic Health.\n2016; 16(1), 683.\n21. Paul SE, Hatoum AS, Fine JD, et al. Associations between prenatal cannabis\nexposure and childhood outcomes: results from the ABCD study. JAMA\nPsychiat. 2021; 78(1), 64–76.\n22. Long X, Lebel C. Evaluati\n\n**Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B** (2024) — \nTitle: Associations between prenatal alcohol exposure and early education outcomes a matched controls study using the born in B\nYear: 2024\nSource: full-text PDF\n\ne EYFSP has also been used to study the relationship between early\ndevelopment and long-term educational attainment,12 as well as to explore the influence of\nfactors such as socio-economic status, parental involvement, and the quality of \n\n\n## Relevant Variables\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: alc0drpreg\nLabel: Mother drank alcohol during pregnancy or 3 months before\nTopic: alcohol_consumption\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't remember]\nNon-missing records: 11369\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: ght_2alcinpreg\nLabel: Do you drink any alcohol during this pregnancy?\nTopic: alcohol_consumption\nType: categorical\nCategories/Values: [1 No] [2 Yes] [3 Do not wish to answer] [4 NT]\nNon-missing records: 2553\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: ght_2alcbeforepreg\nLabel: Did you drink any alcohol during the three months before this pregnancy?\nTopic: alcohol_consumption\nType: categorical\nCategories/Values: [1 No] [2 Yes] [3 Do not wish to answer] [4 Don't know] [5 NT]\nNon-missing records: 2548\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: alc0drfr3m\nLabel: Mother drank alcohol in the first 3 months of pregnancy\nTopic: alcohol_consumption\nType: categorical\nCategories/Values: [1 Yes, once a week] [2 Yes, occasionally] [3 Yes, not specified] [4 No] [5 Don't remember]\nNon-missing records: 3480\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: alc0oamx3b\nLabel: Maximum units of other alcohol at one time in 3 months before pregnancy\nTopic: alcohol_consumption\nType: integer\nNon-missing records: 608\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: alc0oamxf3\nLabel: Maximum units of other alcohol at one time in first 3 months of pregnancy\nTopic: alcohol_consumption\nType: integer\nNon-missing records: 185\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: alc0dr3mb4\nLabel: Mother drank alcohol 3 months before pregnancy\nTopic: alcohol_consumption\nType: categorical\nCategories/Values: [1 Yes, once a week] [2 Yes, occasionally] [3 Yes, not specified] [4 No] [5 Don't remember]\nNon-missing records: 3472\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: alc0oaav3b\nLabel: Average units of other alcohol per week in 3 months before pregnancy\nTopic: alcohol_consumption\nType: integer\nNon-missing records: 613\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: alc0oaavf3\nLabel: Average units of other alcohol per week in first 3 months of pregnancy\nTopic: alcohol_consumption\nType: integer\nNon-missing records: 186\n```\n\n```\nTable: BiB_AgeOfWonder.survey_mod03_dr23\nVariable: awb5_2y_alcohol_lctn___1\nLabel: I got alcohol from parents to drink with them in past week\nTopic: alcohol_consumption\nType: categorical\nCategories/Values: [0 Unchecked] [1 Checked: I got my alcohol from my parents to drink with them]\nNon-missing records: 2026\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_CohortInfo.preg_info\nDisplay name: Pregnancy information\nProject: BiB_CohortInfo\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 10 | Rows: 13776 | Entities: 12453\nLast updated: 2024-05-14 09:12:26\n```\n\n```\nTable: BiB_Pregnancy.eclipse_preg\nDisplay name: Eclipse pregnancy record\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 35 | Rows: 13361 | Entities: 12075\nLast updated: 2024-05-14 09:13:56\n```\n\n```\nTable: BiB_Pregnancy.eclipse_baby\nDisplay name: Eclipse neonatal record\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 29 | Rows: 13525 | Entities: 13525\nLast updated: 2024-05-14 09:13:55\n```\n\n```\nTable: BiB_Pregnancy.matrecs_prebib_preg\nDisplay name: Maternity records pre-BiB pregnancy\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 13 | Rows: 10642 | Entities: 5611\nLast updated: 2024-05-14 09:14:05\n```\n\n```\nTable: BiBBS_CohortInfo.pregnancy\nDisplay name: Recruited pregnancy information\nProject: BiBBS_CohortInfo\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 15 | Rows: 2626 | Entities: 2425\nLast updated: 2024-05-14 09:15:46\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_associations_between_prenatal_alcohol_exposure_and_early_edu_chunk_36",
    "source_metadata": {
      "source": "evaluation:q_1",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_associations_between_prenatal_alcohol_exposure_and_early_edu_chunk_36"
    }
  },
  {
    "query_id": "q_2",
    "question": "What was the adjusted risk ratio for VZV seroprevalence among South Asian women born in South Asia compared to White British women?",
    "reference_answer": "The adjusted risk ratio for VZV seroprevalence among South Asian women born in South Asia, compared with White British women, was 0.93 (95% CI 0.89 to 0.97, p=0.001).",
    "retrieved_context": "## Relevant Published Papers\n\n**Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor** (2013) — \nTitle: Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor\nYear: 2013\nSource: full-text PDF\n\n. 95% or 98%\nin the other groups, p=0.01).\nEthnic group/country of birth category and CMV\nThe association between ethnic group/country of birth\ncategory and CMV seropositivity remained after adjustment for\nparity and household size (Table\n\n**Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor** (2013) — \nTitle: Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor\nYear: 2013\nSource: full-text PDF\n\n(7%) in India, 6 (2%) in Bangladesh, 9 (3%) elsewhere (Kenya\n7, Uganda 1, Burma 1) and only 10 (3%) in England or Wales.\nEthnic group/country of birth category and VZV\nVZV seroprevalence was significantly lower among the\nSouth Asian women\n\n**Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor** (2013) — \nTitle: Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor\nYear: 2013\nSource: full-text PDF\n\nCommittee (ref: 08/H1302/21) and\nthe London School of Hygiene and Tropical Medicine Ethics\nCommittee (ref: 2008/5320). Women gave written informed\nconsent before taking part.\nStatistical methods\nData were analysed using Stata version 12.1\n\n**Seroprevalence of cytomegalovirus, Epstein Barr virus and varicella zoster virus among pregnant women in Bradford: a coh** (2013) — Pembrey Lucy, Raynor Pauline, Griffiths Paul, Chaytor Shelley, Wright John et al\nAbstract: To estimate the seroprevalence of cytomegalovirus (CMV), Epstein Barr virus (EBV) and varicella zoster virus (VZV) among pregnant women in Bradford by ethnic group and country of birth. A stratified random sample of 949 pregnant women enrolled in the Born in Bradford birth cohort was selected to ensure sufficient numbers of White UK born women, Asian UK born women and Asian women born in Asia. Serum samples taken at 24-28 weeks' gestation were tested for CMV IgG, EBV IgG and VZV IgG. Each woman completed a questionnaire which included socio-demographic information. CMV seroprevalence\n\n**Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor** (2013) — \nTitle: Seroprevalence of cytomegalovirus Epstein Barr virus and varicella zoster virus among pregnant women in Bradford a cohor\nYear: 2013\nSource: full-text PDF\n\nuth Asian UK born women\nand 98% among South Asian women born in South Asia. These differences remained after adjusting for socio-\ndemographic factors. In contrast, VZV seroprevalence was 95% among women born in the UK but significantly lo\n\n\n## Relevant Variables\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: eth_ethnic\nLabel: Mother ethnicity all categories\nTopic: ethnic_group\nType: categorical\nCategories/Values: [1 White British] [2 White Irish] [3 Pakistani] [4 Indian] [5 Bangladeshi] [6 White Polish] [7 White Slovakian] [8 White Romanian] [9 White Czech] [10 Other White] [11 White Gypsy/Roma/Irish traveller] [12 Chinese] [13 African] [14 Caribbean] [15 Mixed White/ Black Caribbean] [16 Mixed White/ Black African] [17 Mixed White/ Asian] [18 Do not wish to answer] [19 Other]\nNon-missing records: 2554\n```\n\n```\nTable: COVID19_Survey.adult_phase1\nVariable: ethnicity\nLabel: Ethnic group combined from different cohort data sources\nTopic: demographics_cv19\nType: categorical\nCategories/Values: [1 White British] [2 Pakistani] [3 Other]\nNon-missing records: 1973\n```\n\n```\nTable: COVID19_Survey.adult_bibplus_phase3\nVariable: ethnicity\nLabel: Ethnicity - 16 categories\nTopic: demographics_cv19\nType: categorical\nCategories/Values: [1 English, Welsh, Scottish, Northern Irish or British] [2 Irish] [3 Any other White background] [4 White and Black Caribbean] [5 White and Black African] [6 White and Asian] [7 Any other Mixed background] [8 Indian] [9 Pakistani] [10 Bangladeshi] [11 Chinese] [12 Any other Asian background] [13 African] [14 Caribbean] [15 Other Black, African or Caribbean background] [16 Any other ethnic group]\nNon-missing records: 1104\n```\n\n```\nTable: COVID19_Survey.child_phase2_covid\nVariable: edcont_ethnic_origin\nLabel: Child Ethnic Origin Code\nTopic: demographics_cv19\nType: categorical\nCategories/Values: [1 Bangladeshi] [2 Indian] [3 Pakistani] [4 Chinese/ Other Asian] [5 African] [6 Black Carribbean/ Black Other] [7 Mixed] [8 White British] [9 White Other] [10 Other Ethnic Groups]\nNon-missing records: 600\n```\n\n```\nTable: COVID19_Survey.child_phase1_covid\nVariable: edcont_ethnic_origin\nLabel: Child Ethnic Origin Code - From Child Education Record\nTopic: demographics_cv19\nType: categorical\nCategories/Values: [1 Bangladeshi] [2 Indian] [3 Pakistani] [4 Chinese/ Other Asian] [5 African] [6 Black Carribbean/ Black Other] [7 Mixed] [8 White British] [9 White Other] [10 Other Ethnic Groups]\nNon-missing records: 910\n```\n\n```\nTable: COVID19_Survey.adult_bib_phase2\nVariable: ethnicity_father\nLabel: Ethnic group (father)\nTopic: demographics_cv19\nType: categorical\nCategories/Values: [1 White] [2 Mixed] [3 Black/Black British] [4 Asian/Asian British] [5 Chinese] [6 Other]\nNon-missing records: 34\n```\n\n```\nTable: COVID19_Survey.child_phase1_covid\nVariable: ethnicity_mother\nLabel: Ethnic group (mother)\nTopic: demographics_cv19\nType: categorical\nCategories/Values: [1 White British] [2 Pakistani] [3 Other]\nNon-missing records: 951\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: eth0ethall\nLabel: Mother's ethnic group - all categories\nTopic: ethnic_group\nType: categorical\nCategories/Values: [1 White - British] [2 White - Irish] [3 White - Other] [4 White & Black Caribbean] [5 White & Black African] [6 White & Indian] [7 White & Pakistani] [8 White & Bangladeshi] [9 White & Indian Caribbean] [10 White & African-Indian] [11 Mixed - Other] [12 Black - Caribbean] [13 Black - African] [14 Black - Other] [15 Asian - Indian] [16 Asian - Pakistani] [17 Asian - Bangladeshi] [18 Asian - Indian\nNon-missing records: 11371\n```\n\n```\nTable: COVID19_Survey.child_phase2_covid\nVariable: ethnicity_mother\nLabel: Ethnic group (mother)\nTopic: demographics_cv19\nType: categorical\nCategories/Values: [1 White British] [2 Pakistani] [3 Other]\nNon-missing records: 618\n```\n\n```\nTable: COVID19_Survey.adult_bib_phase2\nVariable: c19a2_fh_ch_brsh_oft\nLabel: 37) Compared to before the pandemic, is this\nTopic: health_behaviour_cv19\nType: categorical\nCategories/Values: [1 Less] [2 About the same] [3 More]\nNon-missing records: 623\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_SDQs.COVID_phase1_sdq\nDisplay name: Strength and Difficulties questionnaire from the COVID phase 1 survey\nProject: BiB_SDQs\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 71 | Rows: 913 | Entities: 913\nLast updated: 2024-05-14 09:15:36\n```\n\n```\nTable: BiB_SDQs.COVID_phase2_sdq\nDisplay name: Strength and Difficulties questionnaire from the COVID phase 2 survey\nProject: BiB_SDQs\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 71 | Rows: 631 | Entities: 631\nLast updated: 2024-05-14 09:15:36\n```\n\n```\nTable: BiB_SDQs.COVID_phase3_sdq\nDisplay name: Strength and Difficulties questionnaire from the COVID phase 3 survey\nProject: BiB_SDQs\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 69 | Rows: 582 | Entities: 582\nLast updated: 2024-05-14 09:15:36\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_pregbp\nDisplay name: Maternity records pregnancy blood pressure\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 8 | Rows: 117578 | Entities: 10005\nLast updated: 2024-05-14 09:14:04\n```\n\n```\nTable: BiB_Biosamples.pregnancy_gtt\nDisplay name: Maternal baseline GTT\nProject: BiB_Biosamples\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 8 | Rows: 12331 | Entities: 11231\nLast updated: 2024-05-14 09:12:09\n```\n",
    "evidence": "VZV seroprevalence was significantly lower among the South Asian women | The adjusted risk ratio for VZV seroprevalence among South Asian women born in South Asia compared to White British women was not reported in the provided context. | 98% among South Asian women born in South Asia. These differences remained after adjusting for socio-demographic factors.",
    "source_chunk_id": "pdf_seroprevalence_of_cytomegalovirus_epstein_barr_virus_and_var_chunk_16",
    "source_metadata": {
      "source": "evaluation:q_2",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_seroprevalence_of_cytomegalovirus_epstein_barr_virus_and_var_chunk_16"
    }
  },
  {
    "query_id": "q_3",
    "question": "What is the odds ratio for caesarean section per 5 kg/m² increase in BMI among Pakistani women?",
    "reference_answer": "The odds ratio for caesarean section per 5 kg/m² increase in BMI among Pakistani women is 1.36 (1.27, 1.45).",
    "retrieved_context": "## Relevant Published Papers\n\n**A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and** (2014) — Bryant M, Santorelli G, Lawlor D A, Farrar D, Tuffnell D et al.\nTitle: A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\nYear: 2014\nSource: full-text PDF\n\n/m2 in Pakistani women or at\n30.0kg/m2 in either ethnic group.\nTable 2 further illustrates the generally linear association of BMI with outcomes in both\nethnic groups, by highlighting the association of BMI as a linear exposure (per 5 kg/m\n\n**A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and** (2014) — Bryant M, Santorelli G, Lawlor D A, Farrar D, Tuffnell D et al.\nTitle: A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\nYear: 2014\nSource: full-text PDF\n\ntani women (N = 4547)\nWhite British women (N = 3931)\np-interactiona\nOdds ratio\n(95%CI) per\n5kg/m2\np-linearb\np-deviation from\nlinearc\nOdds ratio\n(95%CI) per\n5kg/m2\np-linearb\np-deviation from\nlinearc\nCaesarean section\n1.36 (1.27, 1.45)\n<0.00\n\n**Association between mode of delivery and body mass index at 4-5 years in White British and Pakistani children: the Born ** (2021) — Ralphs Eleanor, Pembrey Lucy, West Jane, Santorelli Gillian\nAbstract: Globally, it is becoming more common for pregnant women to deliver by caesarean section (CS). In 2020, 31% of births in England were CS, surpassing the recommended prevalence of CS. Concerns have been raised regarding potential unknown consequences of this mode of delivery. Childhood adiposity is also an increasing concern. Previous research provides inconsistent conclusions on the association between CS and childhood adiposity. More studies are needed to investigate the consequences of CS in different populations and ethnicities. Therefore, this study investigates the association be\n\n**Risk factor screening to identify women requiring oral glucose tolerance testing to diagnose gestational diabetes A syst** (2017) — Farrar Diane, Simmonds Mark, Bryant Maria, Lawlor Debbie A, Dunne Fidelma et al.\nTitle: Risk factor screening to identify women requiring oral glucose tolerance testing to diagnose gestational diabetes A syst\nYear: 2017\nSource: full-text PDF\n\n94.1\n22.7\n78.7\nAge\u001530, BMI\u001530, non-white ethnicity, prior GDM\n94.1\n22.7\n78.7\nAge\u001525, BMI\u001525\n95.9\n16.5\n84.5\nAge\u001525, BMI\u001525, prior GDM\n95.9\n16.5\n84.5\nNICE guideline recommended risk factors[12]\n78.2\n31.7\n67.2\nBMI = body mass index (kg/m2)\nF\n\n**Antenatal depression and anxiety and early pregnancy BMI among White British and South Asian women retrospective analysi** (2020) — Insan Nafisa, Slack Emma, Heslehurst Nicola, Rankin Judith\nTitle: Antenatal depression and anxiety and early pregnancy BMI among White British and South Asian women retrospective analysi\nYear: 2020\nSource: full-text PDF\n\n8)\n1.10 (0.75–1.62)\n–\nOR odds ratio\nAOR adjusted odds ratio\nCI Confidence Interval\nWB White British\n+White British BMI categories: Underweight, < 18.5 kg/m2; recommended weight, 18.5–24.9 kg/m2; overweight, 25–29.9 kg/m2; obese, ≥30 kg/m2\n\n\n## Relevant Variables\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_bmi\nLabel: BMI calculated\nTopic: anthropometry\nType: float\nNon-missing records: 977\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_bmicat5\nLabel: BMI 5 categories\nTopic: anthropometry\nType: categorical\nCategories/Values: [1 Underweight] [2 Normal Weight] [3 Overweight] [4 Obese] [5 Severe Obese]\nNon-missing records: 977\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_bmicalc\nLabel: Whether BMI could be calculated\nTopic: administration\nType: categorical\nCategories/Values: [0 No] [1 Yes]\nNon-missing records: 1452\n```\n\n```\nTable: BiB_AgeOfWonder.heightweight_dr23\nVariable: bmi\nLabel: BMI (kg/m2)\nTopic: anthropometry\nType: float\nNon-missing records: 2003\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_qweight\nLabel: Weight (kg)\nTopic: anthropometry\nType: float\nNon-missing records: 1180\n```\n\n```\nTable: BiB_Maternal_Measurements.maternal_measurements_research_data\nVariable: bmi\nLabel: BMI\nType: float\nNon-missing records: 39822\n```\n\n```\nTable: BiB_Pregnancy.pregnancy_ultrasound\nVariable: mussstomach\nLabel: Anatomy and position of stomach\nTopic: pregnancy\nType: categorical\nCategories/Values: [1 Abnormal] [2 Absent] [3 Abutting OS] [4 Anterior] [5 Breech] [6 Breech legs extended] [7 Breech maternal left] [8 Breech maternal right] [9 Bulky] [10 Cephalic] [11 Cephalic maternal left] [12 Cephalic maternal right] [13 Cervical] [14 Clear of OS] [15 Covering OS] [16 Ectopic] [17 Footling breech] [18 Fundal] [19 Head to mat. Left] [20 Head to mat. Right] [21 Irregular] [22 Left lateral] [23 L\nNon-missing records: 27867\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: mms0mbkbmi\nLabel: Mother's booking BMI (derived)\nTopic: anthropometry\nType: float\nNon-missing records: 10495\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: mms0weight\nLabel: Mother's weight (at questionnaire) (kg)\nTopic: anthropometry\nType: float\nNon-missing records: 10980\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: vit0pregca\nLabel: Pregnacare multivitamins during pregnancy\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Daily] [2 5-6 per week] [3 2-4 per week] [4 Once a week] [5 Less often]\nNon-missing records: 1671\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_Pregnancy.matrecs_bib_pregbp\nDisplay name: Maternity records pregnancy blood pressure\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 8 | Rows: 117578 | Entities: 10005\nLast updated: 2024-05-14 09:14:04\n```\n\n```\nTable: BiBBS_CohortInfo.pregnancy\nDisplay name: Recruited pregnancy information\nProject: BiBBS_CohortInfo\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 15 | Rows: 2626 | Entities: 2425\nLast updated: 2024-05-14 09:15:46\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_postbp\nDisplay name: Maternity records post-partum blood pressure\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 7 | Rows: 36379 | Entities: 9292\nLast updated: 2024-05-14 09:13:59\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_labbp\nDisplay name: Maternity records labour blood pressure\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 8 | Rows: 8643 | Entities: 7995\nLast updated: 2024-05-14 09:13:58\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nDisplay name: Mother baseline pregnancy survey\nProject: BiBBS_Baseline\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 328 | Rows: 2564 | Entities: 2378\nLast updated: 2024-05-14 09:15:45\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_a_comparison_of_south_asian_specific_and_established_bmi_thr_chunk_29",
    "source_metadata": {
      "source": "evaluation:q_3",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_a_comparison_of_south_asian_specific_and_established_bmi_thr_chunk_29"
    }
  },
  {
    "query_id": "q_4",
    "question": "In the paper \"Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\" (2020), what factors were not included in the study due to lack of information?",
    "reference_answer": "Parental substance misuse, parental physical ill health, and out-of-home placements.",
    "retrieved_context": "## Exact Registry Matches\n\n[PAPER TITLE FOUND: 'Child mental health and resilience in the context of socioeconomic disadvantage ']\nYear: 2020\nDOI: \nHas full-text PDF chunks in index: yes\n\n\n## Relevant Published Papers\n\n**Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort** (2020) — \nTitle: Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\nYear: 2020\nSource: full-text PDF\n\nical ill health and \nout-of-home placements. We also did not have information \nregarding mental health diagnoses or treatments, instead \nrelying on screening measures.\nMaternal warmth was self-assessed by mothers which \nhave created bias \n\n**Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort** (2020) — \nTitle: Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\nYear: 2020\nSource: full-text PDF\n\nugging, kissing and holding \nthis child?’ rated on a scale from 1 (‘Never/Almost never’) \nto 5 (‘Always/Almost always’). Questions were completed by \nmothers when children were 24 months old. Higher scores \nindicated increasing efficacy a\n\n**Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort** (2020) — \nTitle: Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\nYear: 2020\nSource: full-text PDF\n\ny was identified (r > 0.8), individual highly correlated \nvariables were removed from the analysis.\nExploratory analyses using univariate linear regression \nwere used to identify variables demonstrating significant \nrelationships with Tot\n\n**Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort** (2020) — \nTitle: Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\nYear: 2020\nSource: full-text PDF\n\n− 0.807 (0.374)*\nRisk factors\n Behind with bills\n− 0.636 (0.686)\n0.403 (0.800)\n0.226 (0.689)\nResource factors\n Warmth\n− .0.458 (0.443)\n− .0.886 (0.414)\n− 0 .887 (0.413)\n Literacy\n− 2.820 (0.446)***\n− 2.765 (0.462)***\n− 2.753 \n(0.449)***\n \n\n**Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort** (2020) — \nTitle: Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\nYear: 2020\nSource: full-text PDF\n\noth parents were not employed \nat baseline. 30% of mothers considered themselves to be \nsingle parents. 32.5% of the sample described some sub-\njective financial worries and 12.6% reported being behind \nwith their household bills. The mea\n\n\n## Relevant Variables\n\n```\nTable: BiB_AgeOfWonder.survey_mod02_dr23\nVariable: brs_total\nLabel: Brief Resilience Scale (higher = more resilient)\nTopic: mental_disorders\nType: integer\nNon-missing records: 3932\n```\n\n```\nTable: BiB_AgeOfWonder.survey_mod02_dr23\nVariable: brs_missing\nLabel: Brief Resilience Scale missing\nTopic: mental_disorders\nType: integer\nNon-missing records: 3932\n```\n\n```\nTable: BiB4All_Geographic.bib4all_geog_lsoa\nVariable: IDACI_2019_decile\nLabel: Income Deprivation Affecting Children Index (IDACI) 2019 decile\nTopic: social_classification\nType: integer\nNon-missing records: 978\n```\n\n```\nTable: BiB_AgeOfWonder.survey_mod02_dr23\nVariable: rcad_total\nLabel: Revised childrens anxiety and depression scale. Total\nTopic: mental_disorders\nType: float\nNon-missing records: 3845\n```\n\n```\nTable: BiB_AgeOfWonder.survey_mod02_dr23\nVariable: brs_cat\nLabel: Brief Resilience Scale category\nTopic: mental_disorders\nType: text\nNon-missing records: 3932\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtChildWhatChron_4\nLabel: Does your child have a chronic mental/behavioural condition\nTopic: mental_disorders\nType: categorical\nCategories/Values: [1 Yes]\nNon-missing records: 194\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtChildWhatChron_other\nLabel: Other chronic health condition your child has\nTopic: general_health\nType: text\nNon-missing records: 5304\n```\n\n```\nTable: BiBBS_Geographic.bibbs_geog_lsoa\nVariable: IDACI_2019_decile\nLabel: Income Deprivation Affecting Children Index (IDACI) 2019 decile\nTopic: social_classification\nType: integer\nNon-missing records: 716\n```\n\n```\nTable: BiB_PrimarySchoolYears.sdq_data_bib\nVariable: psysdqBDfficultDistr\nLabel: Do the difficulties upset or distress the child?\nTopic: personality_temperament\nType: categorical\nCategories/Values: [4 A great deal] [1 Not at all] [2 Only a little] [3 Quite a lot]\nNon-missing records: 335\n```\n\n```\nTable: BiB_PrimarySchoolYears.child_quiz\nVariable: psychdEillorunwell\nLabel: How often are you ill or unwell?\nTopic: wellbeing\nType: categorical\nCategories/Values: [1 Never] [2 Some of the time] [3 All of the time]\nNon-missing records: 15429\n```\n\n\n## Relevant Tables\n\n```\nTable: BiBBS_CohortInfo.child\nDisplay name: Child information\nProject: BiBBS_CohortInfo\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 13 | Rows: 2510 | Entities: 2510\nLast updated: 2024-05-14 09:15:46\n```\n\n```\nTable: BiB_PrimarySchoolYears.child_info\nDisplay name: Primary School Years child information\nProject: BiB_PrimarySchoolYears\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 23 | Rows: 18149 | Entities: 18149\nLast updated: 2024-05-14 09:14:09\n```\n\n```\nTable: BiB_SDQs.PrimarySchoolYears_sdq\nDisplay name: Strength and Difficulties questionnaire from the Primary School Years surve\nProject: BiB_SDQs\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 58 | Rows: 3502 | Entities: 3234\nLast updated: 2024-05-14 09:15:37\n```\n\n```\nTable: BiBBS_CohortInfo.mother\nDisplay name: Recruited mother information\nProject: BiBBS_CohortInfo\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 10 | Rows: 2425 | Entities: 2425\nLast updated: 2024-05-14 09:15:46\n```\n\n```\nTable: BiB_1000.bib1000_18m_b18qc4\nDisplay name: BiB 1000 18m Parent-reported child diagnoses\nProject: BiB_1000\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 3 | Rows: 1307 | Entities: 1293\nLast updated: 2024-05-14 09:11:47\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_child_mental_health_and_resilience_in_the_context_of_socioec_chunk_27",
    "source_metadata": {
      "source": "evaluation:q_4",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_child_mental_health_and_resilience_in_the_context_of_socioec_chunk_27"
    }
  },
  {
    "query_id": "q_5",
    "question": "What is the recommended mean sleep duration for infants and toddlers according to the systematic review by Galland et al.?",
    "reference_answer": "The parent-reported mean sleep duration for infants (0-23 months) was 12.8 hours and 11.9 hours for toddlers and preschoolers (2-5 years old).",
    "retrieved_context": "## Relevant Published Papers\n\n**Sociodemographic temporal and bedtime routine correlates of sleep timing and duration in South Asian and white children** (2023) — \nTitle: Sociodemographic temporal and bedtime routine correlates of sleep timing and duration in South Asian and white children\nYear: 2023\nSource: full-text PDF\n\no maintain both physical and\npsychological health. Inadequate sleep in children is associated\nwith poorer cognitive performance, health markers [1], psychoso-\ncial well-being and family relationships [2,3]. The amount of sleep\nrecommended \n\n**Investigating causal relations between sleep duration and risks of adverse pregnancy and perinatal outcomes linear and n** (2022) — Yang Qian, Magnus Maria C, Kilpi Fanny, Santorelli Gillian, Soares Ana Gonçalves\nTitle: Investigating causal relations between sleep duration and risks of adverse pregnancy and perinatal outcomes linear and n\nYear: 2022\nSource: full-text PDF\n\nA. Lawlor, Kate Tilling and Maria Carolina Borges \nare senior authors who equally supervised this work.\n*Correspondence:  qian.yang@bristol.ac.uk\n2 Population Health Sciences, Bristol Medical School, University of Bristol, \nBristol, UK\nFu\n\n**Investigating causal relations between sleep duration and risks of adverse pregnancy and perinatal outcomes linear and n** (2022) — Yang Qian, Magnus Maria C, Kilpi Fanny, Santorelli Gillian, Soares Ana Gonçalves\nTitle: Investigating causal relations between sleep duration and risks of adverse pregnancy and perinatal outcomes linear and n\nYear: 2022\nSource: full-text PDF\n\ngroups in UK Biobank women. We present MR \nestimates of a linear effect of increasing duration on the outcome across the length of residual duration (h/day) covered in each group. Further \ndetails about identifying the pattern of nonlinea\n\n**Sociodemographic temporal and bedtime routine correlates of sleep timing and duration in South Asian and white children** (2023) — \nTitle: Sociodemographic temporal and bedtime routine correlates of sleep timing and duration in South Asian and white children\nYear: 2023\nSource: full-text PDF\n\n274e80. https://doi.org/\n10.1016/j.sleep.2009.04.012.\n[10] Smith JP, Hardy ST, Hale LE, Gazmararian JA. Racial disparities and sleep\namong preschool aged children: a systematic review. Sleep Health 2019;5(1):\n49e57. https://doi.org/10.1016\n\n**Sleep Duration and Adiposity in Early Childhood Evidence for Bidirectional Associations from the Born in Bradford Study** (2017) — \nTitle: Sleep Duration and Adiposity in Early Childhood Evidence for Bidirectional Associations from the Born in Bradford Study\nYear: 2017\nSource: full-text PDF\n\nunity. Diabetologia. 2006; \n49(10): 2234–2246.\n29.\t Fernando E, Razak F, Lear SA, Anand SS. Cardiovascular Disease in \nSouth Asian Migrants. Can J Cardiol. 2015; 31(9): 1139–1150.\n30.\t Bryant M, Santorelli G, Fairley L, et al. Design and c\n\n\n## Relevant Variables\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtSleepDisturb\nLabel: In past 12 mths, how often has your child's sleep been disturbed due to wheeze\nTopic: respiratory_system\nType: categorical\nCategories/Values: [1 Never woken with wheezing] [2 Less than one night per week] [3 One or more nights per week]\nNon-missing records: 269\n```\n\n```\nTable: BiB_GrowingUp.adult_survey_MeDALL_sample\nVariable: guadtSleepDisturb\nLabel: In past 12 mths, how often has your child's sleep been disturbed due to wheeze\nTopic: respiratory_system\nType: categorical\nCategories/Values: [1 Never woken with wheezing] [2 Less than one night per week] [3 One or more nights per week]\nNon-missing records: 242\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtWeekdaySleep\nLabel: What is your childâ€™s usual amount of sleep each night on week days?\nTopic: sleep\nType: categorical\nCategories/Values: [1 Less than five hours] [2 Five hours] [3 Six hours] [4 Seven hours] [5 Eight hours] [6 Nine hours] [7 Ten hours] [8 Eleven hours] [9 Twelve hours] [10 Thirteen hours] [11 More than thirteen hours]\nNon-missing records: 5133\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtWeekendSleep\nLabel: What is your childâ€™s usual amount of sleep each night at weekends?\nTopic: sleep\nType: categorical\nCategories/Values: [1 Less than five hours] [2 Five hours] [3 Six hours] [4 Seven hours] [5 Eight hours] [6 Nine hours] [7 Ten hours] [8 Eleven hours] [9 Twelve hours] [10 Thirteen hours] [11 More than thirteen hours]\nNon-missing records: 5197\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtWheezingSpeech\nLabel: In past 12 mths, has wheezing limited your child's sleep\nTopic: respiratory_system\nType: categorical\nCategories/Values: [1 No] [2 Yes]\nNon-missing records: 269\n```\n\n```\nTable: BiB_MeDALL.medall_quest\nVariable: meda13\nLabel: How often has childs sleep been disturbed by wheezing in past 12 months\nTopic: respiratory_system\nType: categorical\nCategories/Values: [1 Never] [2 Less than 1 night per week] [3 One or more nights per week]\nNon-missing records: 549\n```\n\n```\nTable: BiB_GrowingUp.adult_survey_MeDALL_sample\nVariable: guadtWheezingSpeech\nLabel: In past 12 mths, has wheezing limited your child's sleep\nTopic: respiratory_system\nType: categorical\nCategories/Values: [1 No] [2 Yes]\nNon-missing records: 242\n```\n\n```\nTable: BiB_1000.bib1000_18m_main\nVariable: bib18d01a\nLabel: How many hours does child sleep during the day\nTopic: sleep\nType: float\nNon-missing records: 1283\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtRashKeptAwake\nLabel: In past 12 mths how often has child been kept awake by rash\nTopic: skin_diseases_dermatology\nType: categorical\nCategories/Values: [1 Never in the past 12 months] [2 Less than one night per week] [3 One or more nights per week]\nNon-missing records: 244\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtBedWetting\nLabel: Which of these best applies to your child:\nTopic: sleep\nType: categorical\nCategories/Values: [1 Never wets the bed at night] [2 Occasionally wets the bed at night] [3 Wets the bed at night once or twice a week] [4 Wets the bed at night three or more times a week] [5 Wears nappies or pull-ups at night] [6 Do not wish to answer]\nNon-missing records: 5106\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_ChildGrowth.anthropometry\nDisplay name: Child anthropometry\nProject: BiB_ChildGrowth\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 12 | Rows: 72386 | Entities: 13597\nLast updated: 2024-05-14 09:12:18\n```\n\n```\nTable: BiBBS_CohortInfo.child\nDisplay name: Child information\nProject: BiBBS_CohortInfo\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 13 | Rows: 2510 | Entities: 2510\nLast updated: 2024-05-14 09:15:46\n```\n\n```\nTable: BiB_GrowingUp.child_survey_child_comp\nDisplay name: BiB Growing Up child survey (child completed)\nProject: BiB_GrowingUp\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 263 | Rows: 4668 | Entities: 4668\nLast updated: 2024-05-14 09:12:47\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nDisplay name: BiB Growing Up child survey (adult completed)\nProject: BiB_GrowingUp\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 443 | Rows: 5304 | Entities: 5304\nLast updated: 2024-05-14 09:12:46\n```\n\n```\nTable: BiB_1000.bib1000_18m_b18qc4\nDisplay name: BiB 1000 18m Parent-reported child diagnoses\nProject: BiB_1000\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 3 | Rows: 1307 | Entities: 1293\nLast updated: 2024-05-14 09:11:47\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_sociodemographic_temporal_and_bedtime_routine_correlates_of__chunk_2",
    "source_metadata": {
      "source": "evaluation:q_5",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_sociodemographic_temporal_and_bedtime_routine_correlates_of__chunk_2"
    }
  },
  {
    "query_id": "q_6",
    "question": "What is the relationship between physical activity and mental health outcomes in children according to a 2011 meta-analysis?",
    "reference_answer": "Increasing physical activity is associated with improved self-esteem and lower rates of depression, anxiety, psychological distress, and emotional difficulties in children aged between 3 and 18 years.",
    "retrieved_context": "## Relevant Published Papers\n\n**Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort** (2020) — \nTitle: Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\nYear: 2020\nSource: full-text PDF\n\nt negatively upon children’s mental health, as \nsuggested by previous research linking literacy difficulties \nwith mental health symptoms in childhood [40]. There is \nless research looking at the effects of physical development \non childr\n\n**Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort** (2020) — \nTitle: Child mental health and resilience in the context of socioeconomic disadvantage results from the Born in Bradford cohort\nYear: 2020\nSource: full-text PDF\n\n(0.635)***\n3.081 \n(0.670)***\nInteractions\n Warmth × behind with bills\n3.361 (1.347)*\n–\n–\n Literacy × behind with bills\n–\n− 0.795 (1.271)\n–\n Physical development × behind \nwith bills\n–\n–\n0.050 (1.362)\n R2\n0.274\n0.265\n0.265\n F change\n6.227*\n\n**The relationship between early life modifiable risk factors for childhood obesity ethnicity and body mass index at age 3** (2015) — Fairley Lesley, Santorelli Gillian, Lawlor Debbie A, Bryant Maria, Bhopal Raj et\nTitle: The relationship between early life modifiable risk factors for childhood obesity ethnicity and body mass index at age 3\nYear: 2015\nSource: full-text PDF\n\nght? Am J Clin Nutr. 2011;94:1772S–5.\n41.\nBrion MJ, Lawlor DA, Matijasevich A, Horta B, Anselmi L, Araujo CL, et al.\nWhat are the causal effects of breastfeeding on IQ, obesity and blood\npressure? Evidence from comparing high-income with \n\n**Association of food security status with overweight and dietary intake exploration of White British and Pakistani-origin** (2018) — \nTitle: Association of food security status with overweight and dietary intake exploration of White British and Pakistani-origin\nYear: 2018\nSource: full-text PDF\n\nsity\nand energy costs. Am J Clin Nutr. 2004;79:6–16.\n9.\nSeligman HK, Schillinger D. Hunger and socioeconomic disparities in\nchronic disease. N Engl J Med. 2010;363:6–9. https://doi.org/10.1056/\nNEJMp1000072.\n10.\nPower M, Uphoff E, Kelly B\n\n**Exploring parental prenatal influences on child health  a multicohort study and data visualisation t** (2025) — \nTitle: Exploring parental prenatal influences on child health  a multicohort study and data visualisation t\nYear: 2025\nSource: full-text PDF\n\nanalysis. Hum Reprod Update. 2018 May 1;24(3):320–89.  \n . \nCC-BY 4.0 International license\nIt is made available under a \n is the author/funder, who has granted medRxiv a license to display the preprint in perpetuity. \n(which was not certified by peer review\n\n\n## Relevant Variables\n\n```\nTable: COVID19_Survey.adult_bib_phase2\nVariable: fl_physact_ch_outdr_c19a2\nLabel: 35) How often do your children do any kind of physical activity outside?\nTopic: physical_health_cv19\nType: categorical\nCategories/Values: [1 Never] [2 1 or 2 days a week] [3 Most days] [4 Every day]\nNon-missing records: 627\n```\n\n```\nTable: COVID19_Survey.adult_bib_phase2\nVariable: fl_physact_ch_c19a2\nLabel: 34) How often do your children do any kind of physical activity?\nTopic: physical_health_cv19\nType: categorical\nCategories/Values: [1 Never] [2 1 or 2 days a week] [3 Most days] [4 Every day]\nNon-missing records: 627\n```\n\n```\nTable: COVID19_Survey.adult_phase1\nVariable: fl_physact_ch_outdr\nLabel: How often do your children do any kind of physical activity outside?\nTopic: health_behaviour_cv19\nType: categorical\nCategories/Values: [1 Never] [2 1 or 2 days a week] [3 Most days] [4 Every day]\nNon-missing records: 2063\n```\n\n```\nTable: COVID19_Survey.adult_phase1\nVariable: fl_physact_ch\nLabel: How often do your children do any kind of physical activity?\nTopic: health_behaviour_cv19\nType: categorical\nCategories/Values: [1 Never] [2 1 or 2 days a week] [3 Most days] [4 Every day]\nNon-missing records: 2084\n```\n\n```\nTable: COVID19_Survey.adult_bibbs_phase2\nVariable: fl_physact_ch_out_c19b2\nLabel: 66) How often does your child do any kind of physical activity outside?\nTopic: health_behaviour_cv19\nType: categorical\nCategories/Values: [1 Never] [2 1 or 2 days a week] [3 Most days] [4 Every day]\nNon-missing records: 92\n```\n\n```\nTable: COVID19_Survey.adult_bibbs_phase2\nVariable: fl_physact_ch_c19b2_rev1\nLabel: 65) How often does your child do any kind of physical activity?\nTopic: health_behaviour_cv19\nType: categorical\nCategories/Values: [1 Never] [2 1 or 2 days a week] [3 Most days] [4 Every day]\nNon-missing records: 92\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtPEincr3Years\nLabel: Has the amount of PE/sports your child does increased since starting school\nTopic: physical_activity\nType: categorical\nCategories/Values: [1 Totally disagree] [2 Disagree] [3 Neither agree nor disagree] [4 Agree] [5 Totally agree] [6 Donâ€™t know]\nNon-missing records: 373\n```\n\n```\nTable: BiB_GrowingUp.adult_survey_BiB1000_sample\nVariable: guadtPEincr3Years\nLabel: Has the amount of PE/sports your child does increased since starting school\nTopic: physical_activity\nType: categorical\nCategories/Values: [1 Totally disagree] [2 Disagree] [3 Neither agree nor disagree] [4 Agree] [5 Totally agree] [6 Donâ€™t know]\nNon-missing records: 365\n```\n\n```\nTable: BiB_GrowingUp.child_survey_adult_comp\nVariable: guadtEncourageSB\nLabel: How much does your behaviour encourage your child to be sedentary?\nTopic: physical_activity\nType: categorical\nCategories/Values: [1 Never] [2 Very rarely] [3 Rarely] [4 Sometimes] [5 Often] [6 Very often]\nNon-missing records: 367\n```\n\n```\nTable: BiB_GrowingUp.child_survey_child_comp\nVariable: guchdDescribeBest\nLabel: Whichoneof the following describes you best for the last 7 days? Listen toall fi\nTopic: physical_activity\nType: categorical\nCategories/Values: [1 All or most of my free time was spent doing things that involve little physical effort] [2 I sometimes (1 - 2 times last week) did physical things in my free time (e.g. played sports, went running, swimming, bike riding, did aerobics)] [3 I often (3 â€” 4 times last week) did physical things in my free time] [4 I quite often (5 - 6 times last week) did physical things in my free time] [5 I very\nNon-missing records: 2042\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_1000.bib1000_36m_b36qi9\nDisplay name: BiB 1000 36m Physical activity clubs (child)\nProject: BiB_1000\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 4 | Rows: 1238 | Entities: 1232\nLast updated: 2024-05-14 09:11:49\n```\n\n```\nTable: BiB_ChildGrowth.anthropometry\nDisplay name: Child anthropometry\nProject: BiB_ChildGrowth\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 12 | Rows: 72386 | Entities: 13597\nLast updated: 2024-05-14 09:12:18\n```\n\n```\nTable: BiB_PrimarySchoolYears.ef_task_inhibition\nDisplay name: Primary School Years EF Inhibition task data\nProject: BiB_PrimarySchoolYears\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 22 | Rows: 14538 | Entities: 14538\nLast updated: 2024-05-14 09:14:26\n```\n\n```\nTable: BiB_ChildGrowth.growth_ResearchData\nDisplay name: Child growth research data\nProject: BiB_ChildGrowth\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 14 | Rows: 42555 | Entities: 13658\nLast updated: 2024-06-04 12:22:55\n```\n\n```\nTable: BiB_ChildGrowth.bioimpedance\nDisplay name: Child bioimpedance\nProject: BiB_ChildGrowth\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 17 | Rows: 6424 | Entities: 5494\nLast updated: 2024-05-14 09:12:18\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_child_mental_health_and_resilience_in_the_context_of_socioec_chunk_23",
    "source_metadata": {
      "source": "evaluation:q_6",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_child_mental_health_and_resilience_in_the_context_of_socioec_chunk_23"
    }
  },
  {
    "query_id": "q_7",
    "question": "What are the four core studies included in the MR-PREG collaboration?",
    "reference_answer": "The four core studies are the Avon Longitudinal Study of Parents and Children (ALSPAC), Born in Bradford (BiB), the Norwegian Mother, Father and Child Cohort Study (MoBa), and UK Biobank (UKB).",
    "retrieved_context": "## Relevant Published Papers\n\n**Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc** (2025) — \nTitle: Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc\nYear: 2025\nSource: full-text PDF\n\n- \n \nGestational age* \n226,810 \n- \n* continuous traits\n . \nCC-BY 4.0 International license\nIt is made available under a \nperpetuity. \n is the author/funder, who has granted medRxiv a license to display the preprint in\n(which was not certified by peer review)\n\n**Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc** (2025) — \nTitle: Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc\nYear: 2025\nSource: full-text PDF\n\nhis\nthis version posted March 23, 2025. \n; \nhttps://doi.org/10.1101/2025.03.22.25324447\ndoi: \nmedRxiv preprint \n\nCohort description  \nAdverse pregnancy and perinatal outcomes (APPOs) \nA key focus of the MR-PREG collaboration is on assessing the causal effect\n\n**Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc** (2025) — \nTitle: Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc\nYear: 2025\nSource: full-text PDF\n\nin each study are presented in supplementary material.  \n \n \n . \nCC-BY 4.0 International license\nIt is made available under a \nperpetuity. \n is the author/funder, who has granted medRxiv a license to display the preprint in\n(which was not certified by peer r\n\n**Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc** (2025) — \nTitle: Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc\nYear: 2025\nSource: full-text PDF\n\nt in\n(which was not certified by peer review)\npreprint \nThe copyright holder for this\nthis version posted March 23, 2025. \n; \nhttps://doi.org/10.1101/2025.03.22.25324447\ndoi: \nmedRxiv preprint \n\n13 \n \nALSPAC: The Avon Longitudinal Study of Parents and Childr\n\n**Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc** (2025) — \nTitle: Cohort Profile  The Mendelian Randomization in Pregnancy  MR-PREG  collaboration - Improving evidenc\nYear: 2025\nSource: full-text PDF\n\nelated key sources of bias. If results are consistent, despite the different sources of bias, \nthis increases the credibility of that being the correct causal effect (whether it suggests a \nprotective, detrimental or no effect), as it would be unlikely for d\n\n\n## Relevant Variables\n\n```\nTable: BiB_Pregnancy.matrecs_prebib_preg\nVariable: prnprebprgperineal\nLabel: Pre BiB preg perineal trauma\nTopic: childbirth\nType: categorical\nCategories/Values: [1 None] [2 1st degree] [3 2nd degree] [4 3rd degree] [5 4th degree] [6 Episiotomy] [7 Laceration] [8 Tear] [9 Graze] [10 Not documented]\nNon-missing records: 10570\n```\n\n```\nTable: BiB_Pregnancy.matrecs_prebib_preg\nVariable: prnprebprgpregid\nLabel: Pre BiB preg ID - links to pre-BiB infant data\nTopic: administration\nType: integer\nNon-missing records: 10642\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_preg\nVariable: prnbpdruglab6\nLabel: Other drugs during labour (6)\nTopic: medications\nType: text\nNon-missing records: 10939\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_preg\nVariable: prnbpdruglab4\nLabel: Other drugs during labour (4)\nTopic: medications\nType: text\nNon-missing records: 10939\n```\n\n```\nTable: BiB_Pregnancy.matrecs_prebib_preg\nVariable: prnprebprggdmt_notdoc\nLabel: Pre BiB preg GDM trt: not documented\nTopic: administration\nType: categorical\nCategories/Values: [1 No] [2 Yes] [3 Not documented]\nNon-missing records: 268\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_preg\nVariable: prnbpdruglab5\nLabel: Other drugs during labour (5)\nTopic: medications\nType: text\nNon-missing records: 10939\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_preg\nVariable: prnbpdruglab8\nLabel: Other drugs during labour (8)\nTopic: medications\nType: text\nNon-missing records: 10939\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_preg\nVariable: prnbpdruglab7\nLabel: Other drugs during labour (7)\nTopic: medications\nType: text\nNon-missing records: 10939\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_preg\nVariable: prnbpdruglab3\nLabel: Other drugs during labour (3)\nTopic: medications\nType: text\nNon-missing records: 10939\n```\n\n```\nTable: BiB_Pregnancy.matrecs_prebib_preg\nVariable: prnprebprggdmt_diet\nLabel: Pre BiB preg GDM trt: Diet\nTopic: endocrine_system\nType: categorical\nCategories/Values: [1 No] [2 Yes] [3 Not documented]\nNon-missing records: 84\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_Metabolomics.metms_pairc_h\nDisplay name: MS metabolomics paired child samples header\nProject: BiB_Metabolomics\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 6 | Rows: 1000 | Entities: 1000\nLast updated: 2024-05-14 09:13:29\n```\n\n```\nTable: BiB_Metabolomics.metms_pairm_h\nDisplay name: MS metabolomics paired mother samples header\nProject: BiB_Metabolomics\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 6 | Rows: 1000 | Entities: 999\nLast updated: 2024-05-14 09:13:37\n```\n\n```\nTable: BiB_Metabolomics.metnmr_gu_father_qc\nDisplay name: NMR metabolomics Growing Up father QC\nProject: BiB_Metabolomics\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 24 | Rows: 200 | Entities: 200\nLast updated: 2024-05-14 09:13:45\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_pregbp\nDisplay name: Maternity records pregnancy blood pressure\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 8 | Rows: 117578 | Entities: 10005\nLast updated: 2024-05-14 09:14:04\n```\n\n```\nTable: BiB_Pregnancy.matrecs_prebib_preg\nDisplay name: Maternity records pre-BiB pregnancy\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 13 | Rows: 10642 | Entities: 5611\nLast updated: 2024-05-14 09:14:05\n```\n",
    "evidence": "ALSPAC: The Avon Longitudinal Study of Parents and Childr | context does not provide evidence for the claim | ALSPAC: The Avon Longitudinal Study of Parents and Children",
    "source_chunk_id": "pdf_cohort_profile__the_mendelian_randomization_in_pregnancy__mr_chunk_16",
    "source_metadata": {
      "source": "evaluation:q_7",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_cohort_profile__the_mendelian_randomization_in_pregnancy__mr_chunk_16"
    }
  },
  {
    "query_id": "q_8",
    "question": "In the paper \"A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\" (2014), what is the difference in the prevalence of gestational diabetes between Pakistani and White British women?",
    "reference_answer": "Gestational diabetes prevalence was higher in women of Pakistani origin.",
    "retrieved_context": "## Exact Registry Matches\n\n[PAPER TITLE FOUND: 'A comparison of South Asian specific and established BMI thresholds for determin']\nYear: 2014\nDOI: 10.1038/ijo.2013.117\nHas full-text PDF chunks in index: yes\n\n\n## Relevant Published Papers\n\n**A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and** (2014) — Bryant M, Santorelli G, Lawlor D A, Farrar D, Tuffnell D et al.\nTitle: A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\nYear: 2014\nSource: full-text PDF\n\ntani women (N = 4547)\nWhite British women (N = 3931)\np-interactiona\nOdds ratio\n(95%CI) per\n5kg/m2\np-linearb\np-deviation from\nlinearc\nOdds ratio\n(95%CI) per\n5kg/m2\np-linearb\np-deviation from\nlinearc\nCaesarean section\n1.36 (1.27, 1.45)\n<0.00\n\n**A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and** (2014) — Bryant M, Santorelli G, Lawlor D A, Farrar D, Tuffnell D et al.\nTitle: A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\nYear: 2014\nSource: full-text PDF\n\ns higher in\nwomen of Pakistani origin, and macrosomia which was lower in Pakistani compared to\nWhite British women at all BMI levels. Again, these findings are consistent with previous\nstudies comparing prevalence rates for of gestational \n\n**A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and** (2014) — Bryant M, Santorelli G, Lawlor D A, Farrar D, Tuffnell D et al.\nTitle: A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\nYear: 2014\nSource: full-text PDF\n\n/m2 in Pakistani women or at\n30.0kg/m2 in either ethnic group.\nTable 2 further illustrates the generally linear association of BMI with outcomes in both\nethnic groups, by highlighting the association of BMI as a linear exposure (per 5 kg/m\n\n**A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and** (2014) — Bryant M, Santorelli G, Lawlor D A, Farrar D, Tuffnell D et al.\nTitle: A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\nYear: 2014\nSource: full-text PDF\n\nnd positive\n(PPV) and negative predictive values (NPV) together with 95% confidence intervals for\neach of these measurements. For all outcomes, except pre-term birth, there was a positive\nmonotonic association between BMI and prevalence of\n\n**A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and** (2014) — Bryant M, Santorelli G, Lawlor D A, Farrar D, Tuffnell D et al.\nTitle: A comparison of South Asian specific and established BMI thresholds for determining obesity prevalence in pregnancy and\nYear: 2014\nSource: full-text PDF\n\nign—Prospective bi-ethnic birth cohort study (The Born in Bradford Cohort).\nSetting—Bradford, a deprived city in the North of the UK.\nParticipants—8,478 South Asian and White British pregnant women participating in the Born in\nBradford coh\n\n\n## Relevant Variables\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_bmicat5\nLabel: BMI 5 categories\nTopic: anthropometry\nType: categorical\nCategories/Values: [1 Underweight] [2 Normal Weight] [3 Overweight] [4 Obese] [5 Severe Obese]\nNon-missing records: 977\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_bmi\nLabel: BMI calculated\nTopic: anthropometry\nType: float\nNon-missing records: 977\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: fat_cbirth\nLabel: Which country was the father of your baby born in\nTopic: place_of_birth\nType: categorical\nCategories/Values: [1 England] [2 Pakistan] [3 Bangladesh] [4 India] [5 Northern Ireland] [6 Scotland] [7 Wales] [8 Republic of Ireland] [9 Poland] [10 Czech Republic] [11 Slovakia] [12 Romania] [13 Hungary] [14 Other] [15 Don't know]\nNon-missing records: 2550\n```\n\n```\nTable: BiB_Maternal_Measurements.maternal_measurements_research_data\nVariable: bmi\nLabel: BMI\nType: float\nNon-missing records: 39822\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_bmicat3\nLabel: BMI 3 categories\nTopic: anthropometry\nType: categorical\nCategories/Values: [1 Underweight/ Normal Weight] [2 Overweight] [3 Obese/ Severe Obese]\nNon-missing records: 977\n```\n\n```\nTable: BiB_Maternal_Measurements.maternal_measurements_research_data\nVariable: weight\nLabel: Weight (kg)\nType: float\nNon-missing records: 39822\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: mes1_qweight\nLabel: Weight (kg)\nTopic: anthropometry\nType: float\nNon-missing records: 1180\n```\n\n```\nTable: BiB_Baseline.base_m_survey\nVariable: mms0weight\nLabel: Mother's weight (at questionnaire) (kg)\nTopic: anthropometry\nType: float\nNon-missing records: 10980\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: hsh_under16\nLabel: Number of children aged under 16 in household\nTopic: household_composition\nType: integer\nNon-missing records: 1960\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nVariable: eth_ethnicoth\nLabel: Mother ethnicity other category details\nTopic: ethnic_group\nType: text\nNon-missing records: 2564\n```\n\n\n## Relevant Tables\n\n```\nTable: BiBBS_CohortInfo.pregnancy\nDisplay name: Recruited pregnancy information\nProject: BiBBS_CohortInfo\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 15 | Rows: 2626 | Entities: 2425\nLast updated: 2024-05-14 09:15:46\n```\n\n```\nTable: BiB_Pregnancy.matrecs_bib_pregbp\nDisplay name: Maternity records pregnancy blood pressure\nProject: BiB_Pregnancy\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 8 | Rows: 117578 | Entities: 10005\nLast updated: 2024-05-14 09:14:04\n```\n\n```\nTable: BiB_Biosamples.pregnancy_gtt\nDisplay name: Maternal baseline GTT\nProject: BiB_Biosamples\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 8 | Rows: 12331 | Entities: 11231\nLast updated: 2024-05-14 09:12:09\n```\n\n```\nTable: BiB_Baseline.base_m1_consumptionoffats\nDisplay name: Mother consumption of fats survey questions phase 1\nProject: BiB_Baseline\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 21 | Rows: 104 | Entities: 104\nLast updated: 2024-05-14 09:12:03\n```\n\n```\nTable: BiBBS_Baseline.pregnancy_survey\nDisplay name: Mother baseline pregnancy survey\nProject: BiBBS_Baseline\nEntity type: participant\nData subjects: \nCohort: bibbs\nVariables: 328 | Rows: 2564 | Entities: 2378\nLast updated: 2024-05-14 09:15:45\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_a_comparison_of_south_asian_specific_and_established_bmi_thr_chunk_18",
    "source_metadata": {
      "source": "evaluation:q_8",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_a_comparison_of_south_asian_specific_and_established_bmi_thr_chunk_18"
    }
  },
  {
    "query_id": "q_9",
    "question": "What is the multi-method approach used in the BiB age 7-11 years follow-up?",
    "reference_answer": "The multi-method approach involves collecting data from children and parents through community and school settings, including face-to-face visits, school nurse teams, and teacher assessments.",
    "retrieved_context": "## Relevant Published Papers\n\n**Growing up in Bradford: protocol for the age 7-11 follow up of the Born in Bradford birth cohort.** (2019) — Bird Philippa K, McEachan Rosemary R C, Mon-Williams Mark, Small Neil, West Jane\nAbstract: Born in Bradford (BiB) is a prospective multi-ethnic pregnancy and birth cohort study that was established to examine determinants of health and development during childhood and, subsequently, adult life in a deprived multi-ethnic population in the north of England. Between 2007 and 2010, the BiB cohort recruited 12,453 women who experienced 13,776 pregnancies and 13,858 births, along with 3353 of their partners. Forty five percent of the cohort are of Pakistani origin. Now that children are at primary school, the first full follow-up of the cohort is taking place. The aims of the fo\n\n**Growing up in Bradford protocol for the age 7-11 follow up of the Born in Bradford birth cohort** (2019) — \nTitle: Growing up in Bradford protocol for the age 7-11 follow up of the Born in Bradford birth cohort\nYear: 2019\nSource: full-text PDF\n\nre employing a multi-method approach across three data collection arms (community-based family\nvisits, school based physical assessment, and whole classroom cognitive, motor function and wellbeing measures)\nto follow-up over 9000 BiB children aged 7–11 years and \n\n**Role of foetal kidney size on kidney function in childhood the born in bradford cohort renal study** (2023) — \nTitle: Role of foetal kidney size on kidney function in childhood the born in bradford cohort renal study\nYear: 2023\nSource: full-text PDF\n\nacks \nthrough into early childhood [18] and lower foetal kidney \nsize was found to be independently associated with both \nreduced eGFR and kidney volume at age 6 years in the \nGeneration R cohort in the Netherlands [19].\nWe aimed to investigate the relationshi\n\n**Prediction of childhood overweight and obesity at age 10-11 findings from the Studying Lifecourse Obesity PrEdictors and** (2023) — \nTitle: Prediction of childhood overweight and obesity at age 10-11 findings from the Studying Lifecourse Obesity PrEdictors and\nYear: 2023\nSource: full-text PDF\n\niB children aged 7–11 years were followed up using\na multi-method approach between 2017 and 2020 (BiB Growing Up Study)\n[27] as part of which anthropometric measurements were collected by\ntrained researchers. Written informed consent was \n\n**Growing up in Bradford protocol for the age 7-11 follow up of the Born in Bradford birth cohort** (2019) — \nTitle: Growing up in Bradford protocol for the age 7-11 follow up of the Born in Bradford birth cohort\nYear: 2019\nSource: full-text PDF\n\nical samples on most participants (Fig. 1),\nprovide a unique opportunity for clinical and public health\ntranslational research on maternal and perinatal health\nand developmental origins of health and wellbeing.\nBiB age 7–11 years follow-up: a resource for future\n\n\n\n## Relevant Variables\n\n```\nTable: BiB_PrimarySchoolYears.ef_task_inhibition_bib\nVariable: AgeYears\nLabel: Child age in years at Inhibition completion\nTopic: age\nType: integer\nNon-missing records: 5741\n```\n\n```\nTable: BiB_PrimarySchoolYears.ef_task_inhibition_bib\nVariable: AgeMonths\nLabel: Child age in months at Inhibition completion\nTopic: age\nType: integer\nNon-missing records: 5741\n```\n\n```\nTable: BiB_GrowingUp.participant_pathway\nVariable: age_recruited\nLabel: Age recruited to Growing Up\nTopic: administration\nType: integer\nNon-missing records: 12213\n```\n\n```\nTable: BiB_1000.bib1000_6m_main\nVariable: agecy_b6mtab\nLabel: Child age (years) BiB1000 6m questionnaire\nTopic: age\nType: integer\nNon-missing records: 1336\n```\n\n```\nTable: BiB_PrimarySchoolYears.ef_task_inhibition\nVariable: AgeYears\nLabel: Child age in years at Inhibition completion\nTopic: age\nType: integer\nNon-missing records: 14538\n```\n\n```\nTable: BiB_StartingSchool.stschool_info\nVariable: agecm_ssclid\nLabel: Age in months at Letter ID\nTopic: age\nType: integer\nNon-missing records: 6582\n```\n\n```\nTable: BiB_PrimarySchoolYears.ef_task_inhibition\nVariable: AgeMonths\nLabel: Child age in months at Inhibition completion\nTopic: age\nType: integer\nNon-missing records: 14538\n```\n\n```\nTable: BiB_GrowingUp.dxa_scan_f\nVariable: tot_bmdage_am_adjust\nLabel: Total age-matched adjusted to normal BMD for age\nTopic: anthropometry\nType: integer\nNon-missing records: 58\n```\n\n```\nTable: BiB_GrowingUp.dxa_scan_m\nVariable: tot_bmdage_am_adjust\nLabel: Total age-matched adjusted to normal BMD for age\nTopic: anthropometry\nType: integer\nNon-missing records: 488\n```\n\n```\nTable: BiB_PrimarySchoolYears.ef_task_bdr_bib\nVariable: AgeYears\nLabel: Child age in years at BDR completion\nTopic: age\nType: integer\nNon-missing records: 5986\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_GrowingUp.dxa_scan_c\nDisplay name: BiB Growing Up DXA scan (child)\nProject: BiB_GrowingUp\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 147 | Rows: 541 | Entities: 541\nLast updated: 2024-05-14 09:12:47\n```\n\n```\nTable: BiB_GrowingUp.dxa_scan_f\nDisplay name: BiB Growing Up DXA scan (father)\nProject: BiB_GrowingUp\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 128 | Rows: 58 | Entities: 58\nLast updated: 2024-05-14 09:12:47\n```\n\n```\nTable: BiB_AgeOfWonder.bioimpedance_dr23\nDisplay name: AoW school visit bioimpedance 2023 release\nProject: BiB_AgeOfWonder\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 28 | Rows: 1583 | Entities: 1583\nLast updated: 2024-05-14 16:40:29.736719\n```\n\n```\nTable: BiB_AgeOfWonder.bioimpedance_dr24\nDisplay name: AoW school visit bioimpedance 2024 release\nProject: BiB_AgeOfWonder\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 27 | Rows: 3723 | Entities: 3721\nLast updated: 2025-01-21 08:42:03.778494\n```\n\n```\nTable: BiB_AgeOfWonder.bloodpressure_dr23\nDisplay name: AoW school visit blood pressure 2023 release\nProject: BiB_AgeOfWonder\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 21 | Rows: 1854 | Entities: 1747\nLast updated: 2024-05-14 16:40:55.067242\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_growing_up_in_bradford_protocol_for_the_age_7_11_follow_up_o_chunk_9",
    "source_metadata": {
      "source": "evaluation:q_9",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_growing_up_in_bradford_protocol_for_the_age_7_11_follow_up_o_chunk_9"
    }
  },
  {
    "query_id": "q_10",
    "question": "In the paper \"Incidence of developmental disorders and special educational needs and disabilities in children in the UK\" (2026), what criteria were used to exclude certain feeding problem codes?",
    "reference_answer": "Codes relating to feeding problems recorded before 6 months of age were excluded because they were unlikely to represent oromotor feeding difficulties as described by NICE.",
    "retrieved_context": "## Exact Registry Matches\n\n[PAPER TITLE FOUND: 'Incidence of developmental disorders and special educational needs and disabilit']\nYear: 2026\nDOI: 10.1111/dmcn.16396\nHas full-text PDF chunks in index: yes\n\n\n## Relevant Published Papers\n\n**Incidence of developmental disorders and special educational needs and disabilities in children in the UK** (2026) — Pettinger Katherine, Blower Sarah, Boyle Elaine, Hewitt Catherine, Fraser Lorna\nTitle: Incidence of developmental disorders and special educational needs and disabilities in children in the UK\nYear: 2026\nSource: full-text PDF\n\nevelopmental disorders were developed by \nthe first author using published lists (see Tables S1 and S2) \nas well as by searching the Clinical Terminology browser \nand NHS Digital's Classifications browser.16 The partici­\npants' medical records were sear\n\n**Incidence of developmental disorders and special educational needs and disabilities in children in the UK** (2026) — Pettinger Katherine, Blower Sarah, Boyle Elaine, Hewitt Catherine, Fraser Lorna\nTitle: Incidence of developmental disorders and special educational needs and disabilities in children in the UK\nYear: 2026\nSource: full-text PDF\n\ne of professionals involved in the care of \nchildren and young people, such as healthcare profession­\nals and those working in education and social care. The \nguideline includes a broad range of conditions, reflecting \nthe reality that developmental dis\n\n**Incidence of developmental disorders and special educational needs and disabilities in children in the UK** (2026) — Pettinger Katherine, Blower Sarah, Boyle Elaine, Hewitt Catherine, Fraser Lorna\nTitle: Incidence of developmental disorders and special educational needs and disabilities in children in the UK\nYear: 2026\nSource: full-text PDF\n\nrs of children who had a code corresponding to \na developmental disorder by the amount of at-­risk person-­\ntime.19 Incidence rate was used as the measure of disease \nfrequency when considering individual disorders, so as not \nto lose information. A chi\n\n**Incidence of developmental disorders and special educational needs and disabilities in children in the UK** (2026) — Pettinger Katherine, Blower Sarah, Boyle Elaine, Hewitt Catherine, Fraser Lorna\nTitle: Incidence of developmental disorders and special educational needs and disabilities in children in the UK\nYear: 2026\nSource: full-text PDF\n\ntainment strategy, the effect sizes were \nonly marginally different (see Table S15).\nThe results of the Kaplan–Meier estimation and Cox re­\ngression are presented in Tables S16 to S21 and Figures S4 to \nS7; the hazard ratios show similar trends to the o\n\n**Incidence of developmental disorders and special educational needs and disabilities in children in the UK** (2026) — Pettinger Katherine, Blower Sarah, Boyle Elaine, Hewitt Catherine, Fraser Lorna\nTitle: Incidence of developmental disorders and special educational needs and disabilities in children in the UK\nYear: 2026\nSource: full-text PDF\n\nlargest FMI, 0.15.\nSEN provision. Pakistani heritage children: imputations, 20; RVI, 0.09; largest FMI, 0.30. See also Figure 4. White British children, imputations, 17, average RVI, 0.05, largest FMI, 0.18.\nAbbreviations: CI, confidence interval; FMI, \n\n\n## Relevant Variables\n\n```\nTable: BiB_1000.bib1000_12m_main\nVariable: bib12d02pcarerfeed\nLabel: Does a local authority creche feed your child\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 28\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24f21anursery\nLabel: Does child eat during evening at nursery(weekends)\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 1228\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24f19anursery\nLabel: Does child eat during afternoon at nursery(weekends)\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 1228\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24e02pcarerfeed\nLabel: Does a local authority creche feed your child\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 10\n```\n\n```\nTable: BiB_1000.bib1000_6m_main\nVariable: bib6g2pcarerfeed\nLabel: Does a local authority creche feed your child\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 5\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24f20anursery\nLabel: Does child eat at tea-time at nursery(weekends)\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 1228\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24f17anursery\nLabel: Does child eat during morning at nursery(weekends)\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 1228\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24f23anursery\nLabel: Does child eat before bed at nursery(weekends)\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 1228\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24f10anursery\nLabel: Does child eat at tea-time at nursery(weekdays)\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 1228\n```\n\n```\nTable: BiB_1000.bib1000_24m_main\nVariable: bib24f21atv\nLabel: Does child eat during evening in front of TV(weekends)\nTopic: diet_and_nutrition\nType: categorical\nCategories/Values: [1 Yes] [2 No] [3 Don't know] [4 Refused to answer]\nNon-missing records: 1228\n```\n\n\n## Relevant Tables\n\n```\nTable: BiB_Baseline.base_m2_foodfrequency\nDisplay name: Mother food frequency and knowledge survey questions phase 2\nProject: BiB_Baseline\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 43 | Rows: 4494 | Entities: 4494\nLast updated: 2024-05-14 09:12:04\n```\n\n```\nTable: BiB_PrimarySchoolYears.ef_task_inhibition\nDisplay name: Primary School Years EF Inhibition task data\nProject: BiB_PrimarySchoolYears\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 22 | Rows: 14538 | Entities: 14538\nLast updated: 2024-05-14 09:14:26\n```\n\n```\nTable: BiB_Baseline.base_m1_foodfrequency\nDisplay name: BiB Baseline mother food frequency phase 1\nProject: BiB_Baseline\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 116 | Rows: 1776 | Entities: 1776\nLast updated: 2024-05-14 09:12:03\n```\n\n```\nTable: BiB_SDQs.PrimarySchoolYears_sdq\nDisplay name: Strength and Difficulties questionnaire from the Primary School Years surve\nProject: BiB_SDQs\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 58 | Rows: 3502 | Entities: 3234\nLast updated: 2024-05-14 09:15:37\n```\n\n```\nTable: BiB_Baseline.base_m1_consumptionoffats\nDisplay name: Mother consumption of fats survey questions phase 1\nProject: BiB_Baseline\nEntity type: participant\nData subjects: \nCohort: bib\nVariables: 21 | Rows: 104 | Entities: 104\nLast updated: 2024-05-14 09:12:03\n```\n",
    "evidence": "",
    "source_chunk_id": "pdf_incidence_of_developmental_disorders_and_special_educational_chunk_7",
    "source_metadata": {
      "source": "evaluation:q_10",
      "evaluation_file": "faithfulness_llama-3-1-8b-bib-grounded-sf-tfx_20260522_133629.json",
      "source_chunk_id": "pdf_incidence_of_developmental_disorders_and_special_educational_chunk_7"
    }
  }
]

def build_evaluation_sanity_records() -> List[Dict]:
    if not getattr(cfg, 'use_evaluation_sanity_records', False):
        return []
    selected_rows = EVALUATION_SANITY_RAW[:max(0, cfg.evaluation_sanity_examples)]
    records = build_stage2_records(selected_rows, oversample_numeric=False)
    print(f'Evaluation sanity records: {len(records)} from {EVALUATION_SANITY_SOURCE}')
    if records:
        print('Evaluation sanity first question:', records[0].get('question', ''))
    return records


evaluation_sanity_records = build_evaluation_sanity_records()


def get_sanity_check_records() -> List[Dict]:
    if getattr(cfg, 'use_evaluation_sanity_records', False) and evaluation_sanity_records:
        return evaluation_sanity_records[:max(0, cfg.evaluation_sanity_examples)]
    return stage2_dev_records[:max(0, cfg.sanity_check_examples)]


def get_sanity_check_source_label() -> str:
    if getattr(cfg, 'use_evaluation_sanity_records', False) and evaluation_sanity_records:
        return 'embedded evaluation rows'
    return 'Stage 2 dev rows'


In [ ]:
hf_api_sanity_records = get_sanity_check_records()
hf_api_sanity_examples = len(hf_api_sanity_records)


def get_hf_token_for_api() -> str:
    token = os.environ.get('HF_TOKEN', '')
    if token:
        return token
    try:
        from google.colab import userdata as colab_userdata
        return colab_userdata.get('HF_TOKEN') or ''
    except Exception:
        return ''


def extract_chat_completion_text(response) -> str:
    try:
        return response.choices[0].message.content.strip()
    except Exception:
        pass
    try:
        message = response['choices'][0]['message']
        return str(message.get('content', '')).strip()
    except Exception:
        return str(response).strip()


def run_hf_api_base_sanity_check() -> None:
    if not cfg.run_hf_api_base_sanity_check:
        print('Skipping hosted Hugging Face base-model sanity check because run_hf_api_base_sanity_check=False.')
        return
    if hf_api_sanity_examples == 0:
        print('Skipping hosted Hugging Face base-model sanity check because there are no sanity examples.')
        return

    token = get_hf_token_for_api()
    if not token:
        print('Skipping hosted Hugging Face base-model sanity check because HF_TOKEN was not found.')
        return

    try:
        from huggingface_hub import InferenceClient
    except Exception as exc:
        print('Skipping hosted Hugging Face base-model sanity check because InferenceClient could not be imported:', repr(exc))
        return

    client = InferenceClient(
    provider="cerebras",   # or "novita"
    token=token)
    
    print(f'Running hosted Hugging Face base-model sanity check on {hf_api_sanity_examples} examples...')
    print('Sanity source:', get_sanity_check_source_label())
    print('Hosted model:', cfg.hf_api_base_model)
    print('This calls the base instruct model through the Hugging Face API, with no local LoRA adapter.')

    numeric_failures = 0
    checked_numeric_rows = 0

    for idx, row in enumerate(hf_api_sanity_records, start=1):
        messages = row.get('messages') or []
        prompt_messages = messages[:-1] if messages and messages[-1].get('role') == 'assistant' else messages
        answer = row['answer']
        question = row.get('question', '').strip()
        evidence = row.get('evidence', '').strip()

        try:
            response = client.chat_completion(
                model=cfg.hf_api_base_model,
                messages=prompt_messages,
                max_tokens=cfg.generation_max_new_tokens,
                temperature=0.2,
            )
            prediction = extract_chat_completion_text(response)
        except Exception as exc:
            prediction = ''
            print(f'--- Hosted HF base sanity example {idx} ---')
            print('Question:', question or '[missing question field]')
            print('API error:', repr(exc))
            if 'hf-inference' in repr(exc) and 'not supported' in repr(exc):
                print("Hint: this runtime routed to provider 'hf-inference', which does not serve this model. Use a provider-enabled InferenceClient or disable this optional hosted check.")
            print()
            continue

        numeric_match = numeric_preservation_match(prediction, answer, question)
        strict_numeric_match = numeric_strict_match(prediction, answer, question)
        unexpected_numbers = unexpected_numeric_tokens(prediction, answer, question)
        if numeric_match is not None:
            checked_numeric_rows += 1
            if not numeric_match:
                numeric_failures += 1

        allowed_text = '\n'.join([str(question), str(answer), str(evidence), str(row.get('context', ''))])
        unsupported_hits = unsupported_reference_hits(prediction, allowed_text)

        print(f'--- Hosted HF base sanity example {idx} ---')
        print('Question:', question or '[missing question field]')
        if evidence:
            print('Evidence:', evidence[:400])
        print('Target:', answer)
        print('Required target numbers:', required_numeric_tokens(answer, question))
        print('Required target numbers in context:', numeric_context_support_diagnostic(row))
        print('Prediction numbers:', extract_numeric_tokens(prediction))
        print('Numeric recall match:', numeric_match)
        print('Strict numeric match:', strict_numeric_match)
        print('Unexpected prediction numbers:', unexpected_numbers)
        print('Unsupported references:', unsupported_hits)
        print('Prediction:', prediction if prediction else '[empty output]')
        print()

    if checked_numeric_rows:
        print(f'Hosted HF base numeric sanity failures: {numeric_failures}/{checked_numeric_rows}')


run_hf_api_base_sanity_check()


In [ ]:
sanity_records = get_sanity_check_records()
sanity_examples = len(sanity_records)

if sanity_examples == 0:
    print('Skipping sanity check because there are no sanity examples.')
else:
    model_device = next(stage2_model.parameters()).device
    was_training = stage2_model.training
    stage2_model.eval()

    print(f'Running pre-training sanity check on {sanity_examples} examples...')
    print('Sanity source:', get_sanity_check_source_label())
    print('This is before Stage 2 SFT, so use it to inspect prompt formatting and decoding, not final model quality.')
    print('Generation defaults:', {
        'max_new_tokens': cfg.generation_max_new_tokens,
        'repetition_penalty': cfg.generation_repetition_penalty,
        'no_repeat_ngram_size': cfg.generation_no_repeat_ngram_size,
    })

    numeric_failures = 0
    checked_numeric_rows = 0

    for idx, row in enumerate(sanity_records, start=1):
        prompt_text = row['prompt_text']
        answer = row['answer']
        question = row.get('question', '').strip()
        evidence = row.get('evidence', '').strip()

        model_inputs = tokenizer(
            prompt_text,
            return_tensors='pt',
            add_special_tokens=False,
        ).to(model_device)

        generation_kwargs = {
            'max_new_tokens': cfg.generation_max_new_tokens,
            'do_sample': False,
            'pad_token_id': tokenizer.pad_token_id,
            'eos_token_id': tokenizer.eos_token_id,
            'renormalize_logits': True,
        }
        if cfg.generation_repetition_penalty and cfg.generation_repetition_penalty != 1.0:
            generation_kwargs['repetition_penalty'] = cfg.generation_repetition_penalty
        if cfg.generation_no_repeat_ngram_size and cfg.generation_no_repeat_ngram_size > 0:
            generation_kwargs['no_repeat_ngram_size'] = cfg.generation_no_repeat_ngram_size

        with torch.no_grad():
            generated = stage2_model.generate(
                **model_inputs,
                **generation_kwargs,
            )

        prompt_len = model_inputs['input_ids'].shape[1]
        prediction = tokenizer.decode(generated[0][prompt_len:], skip_special_tokens=True).strip()
        numeric_match = numeric_preservation_match(prediction, answer, question)
        strict_numeric_match = numeric_strict_match(prediction, answer, question)
        unexpected_numbers = unexpected_numeric_tokens(prediction, answer, question)
        if numeric_match is not None:
            checked_numeric_rows += 1
            if not numeric_match:
                numeric_failures += 1

        allowed_text = '\n'.join([str(question), str(answer), str(evidence), str(row.get('context', ''))])
        unsupported_hits = unsupported_reference_hits(prediction, allowed_text)

        print(f'--- Sanity check example {idx} ---')
        print('Question:', question or '[missing question field]')
        if evidence:
            print('Evidence:', evidence[:400])
        print('Target:', answer)
        print('Required target numbers:', required_numeric_tokens(answer, question))
        print('Required target numbers in context:', numeric_context_support_diagnostic(row))
        print('Prediction numbers:', extract_numeric_tokens(prediction))
        print('Numeric recall match:', numeric_match)
        print('Strict numeric match:', strict_numeric_match)
        print('Unexpected prediction numbers:', unexpected_numbers)
        print('Unsupported references:', unsupported_hits)
        print('Prediction:', prediction if prediction else '[empty output]')
        print()

    if checked_numeric_rows:
        print(f'Pre-training numeric sanity failures: {numeric_failures}/{checked_numeric_rows}')

    if was_training:
        stage2_model.train()


In [ ]:
def tokenize_assistant_only_dataset(raw_dataset: Dataset, split_name: str) -> Dataset:
    tokenized_rows = []
    dropped_no_answer = 0
    dropped_answer_too_long = 0
    truncated_prompt_rows = 0

    for row in raw_dataset:
        encoded = build_supervised_example(row['prompt_text'], row['full_text'])
        if encoded is None:
            full_ids = tokenizer(row['full_text'], add_special_tokens=False)['input_ids']
            prompt_ids = tokenizer(row['prompt_text'], add_special_tokens=False)['input_ids']
            answer_token_estimate = max(0, len(full_ids) - min(len(full_ids), len(prompt_ids)))
            if answer_token_estimate <= 0:
                dropped_no_answer += 1
            else:
                dropped_answer_too_long += 1
            continue

        if encoded['prompt_tokens_dropped'] > 0:
            truncated_prompt_rows += 1

        tokenized_rows.append({
            'input_ids': encoded['input_ids'],
            'attention_mask': encoded['attention_mask'],
            'labels': encoded['labels'],
        })

    if not tokenized_rows:
        raise ValueError(f'No valid Stage 2 examples remained for {split_name}. Increase max_seq_len or inspect the SFT records.')

    print(f'{split_name}: kept {len(tokenized_rows)} examples')
    print(f'{split_name}: dropped with no answer tokens = {dropped_no_answer}')
    print(f'{split_name}: dropped because answer alone exceeded max_seq_len = {dropped_answer_too_long}')
    print(f'{split_name}: truncated prompt/context rows = {truncated_prompt_rows}')

    tokenized_dataset = Dataset.from_list(tokenized_rows)
    tokenized_dataset.set_format(type='torch')
    return tokenized_dataset

stage2_train_tok = tokenize_assistant_only_dataset(stage2_train_raw_ds, 'stage2_train')
stage2_dev_tok = tokenize_assistant_only_dataset(stage2_dev_raw_ds, 'stage2_dev')

print(stage2_train_tok)
print(stage2_dev_tok)

In [ ]:
class GroundedQATrainer(Trainer):
    def __init__(self, *args, generation_eval_dataset=None, generation_max_examples=32, **kwargs):
        super().__init__(*args, **kwargs)
        self.generation_eval_dataset = generation_eval_dataset
        self.generation_max_examples = generation_max_examples

    def _compute_generation_metrics(self, metric_key_prefix='eval'):
        if self.generation_eval_dataset is None or len(self.generation_eval_dataset) == 0:
            return {}

        if self.generation_max_examples is None or self.generation_max_examples <= 0:
            sample = self.generation_eval_dataset
        else:
            sample_size = min(len(self.generation_eval_dataset), self.generation_max_examples)
            sample = self.generation_eval_dataset.select(range(sample_size))

        total_f1 = 0.0
        exact_matches = 0
        evidence_hits = 0
        abstain_matches = 0
        numeric_rows = 0
        numeric_matches = 0
        strict_numeric_rows = 0
        strict_numeric_matches = 0
        unexpected_numeric_rows = 0
        unsupported_reference_rows = 0

        was_training = self.model.training
        model_device = next(self.model.parameters()).device
        self.model.eval()

        for row in sample:
            prompt_text = row['prompt_text']
            answer = row['answer']
            evidence = row['evidence']
            question = row.get('question', '')
            context = row.get('context', '')

            model_inputs = tokenizer(prompt_text, return_tensors='pt', add_special_tokens=False).to(model_device)
            with torch.no_grad():
                generation_kwargs = {
                    'max_new_tokens': cfg.generation_max_new_tokens,
                    'do_sample': False,
                    'renormalize_logits': True,
                    'pad_token_id': tokenizer.pad_token_id,
                    'eos_token_id': tokenizer.eos_token_id,
                }
                if cfg.generation_repetition_penalty and cfg.generation_repetition_penalty != 1.0:
                    generation_kwargs['repetition_penalty'] = cfg.generation_repetition_penalty
                if cfg.generation_no_repeat_ngram_size and cfg.generation_no_repeat_ngram_size > 0:
                    generation_kwargs['no_repeat_ngram_size'] = cfg.generation_no_repeat_ngram_size
                generated = self.model.generate(
                    **model_inputs,
                    **generation_kwargs,
                )

            prompt_len = model_inputs['input_ids'].shape[1]
            prediction = tokenizer.decode(generated[0][prompt_len:], skip_special_tokens=True).strip()

            total_f1 += token_f1(prediction, answer)
            if normalized_answer(prediction) == normalized_answer(answer):
                exact_matches += 1
            if evidence and normalized_answer(evidence) in normalized_answer(prediction):
                evidence_hits += 1
            if is_abstain_answer(prediction) == is_abstain_answer(answer):
                abstain_matches += 1
            numeric_match = numeric_preservation_match(prediction, answer, question)
            strict_numeric_match = numeric_strict_match(prediction, answer, question)
            extra_numbers = unexpected_numeric_tokens(prediction, answer, question)
            if numeric_match is not None:
                numeric_rows += 1
                if numeric_match:
                    numeric_matches += 1
            if strict_numeric_match is not None:
                strict_numeric_rows += 1
                if strict_numeric_match:
                    strict_numeric_matches += 1
            if extra_numbers:
                unexpected_numeric_rows += 1
            allowed_text = "\n".join([str(question), str(answer), str(evidence), str(context)])
            if has_unsupported_reference(prediction, allowed_text):
                unsupported_reference_rows += 1

        if was_training:
            self.model.train()

        sample_size = max(1, len(sample))
        numeric_match_rate = numeric_matches / max(1, numeric_rows)
        strict_numeric_match_rate = strict_numeric_matches / max(1, strict_numeric_rows)
        unexpected_numeric_rate = unexpected_numeric_rows / sample_size
        numeric_precision_rate = 1.0 - unexpected_numeric_rate
        unsupported_reference_rate = unsupported_reference_rows / sample_size
        reference_support_rate = 1.0 - unsupported_reference_rate
        metrics = {
            f'{metric_key_prefix}_answer_f1': total_f1 / sample_size,
            f'{metric_key_prefix}_exact_match': exact_matches / sample_size,
            f'{metric_key_prefix}_evidence_hit_rate': evidence_hits / sample_size,
            f'{metric_key_prefix}_abstain_match_rate': abstain_matches / sample_size,
            f'{metric_key_prefix}_numeric_match_rate': numeric_match_rate,
            f'{metric_key_prefix}_numeric_rows': numeric_rows,
            f'{metric_key_prefix}_strict_numeric_match_rate': strict_numeric_match_rate,
            f'{metric_key_prefix}_strict_numeric_rows': strict_numeric_rows,
            f'{metric_key_prefix}_unexpected_numeric_rate': unexpected_numeric_rate,
            f'{metric_key_prefix}_numeric_precision_rate': numeric_precision_rate,
            f'{metric_key_prefix}_unsupported_reference_rate': unsupported_reference_rate,
            f'{metric_key_prefix}_reference_support_rate': reference_support_rate,
        }
        numeric_component = strict_numeric_match_rate if strict_numeric_rows else metrics[f'{metric_key_prefix}_answer_f1']
        # Do not reward abstention directly here: an over-abstaining model can be faithful but unhelpful.
        # Keep abstain_match_rate as a diagnostic metric, while checkpoint selection favors answer quality,
        # numeric correctness, and supported references.
        metrics[f'{metric_key_prefix}_grounded_score'] = (
            0.45 * metrics[f'{metric_key_prefix}_answer_f1']
            + 0.10 * metrics[f'{metric_key_prefix}_evidence_hit_rate']
            + 0.30 * numeric_component
            + 0.10 * numeric_precision_rate
            + 0.05 * reference_support_rate
        )
        return metrics

    def evaluation_loop(
        self,
        dataloader,
        description,
        prediction_loss_only=None,
        ignore_keys=None,
        metric_key_prefix='eval',
    ):
        output = super().evaluation_loop(
            dataloader,
            description,
            prediction_loss_only=prediction_loss_only,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )

        generation_metrics = self._compute_generation_metrics(metric_key_prefix=metric_key_prefix)
        if generation_metrics:
            output.metrics.update(generation_metrics)

        return output

In [ ]:
stage2_args = TrainingArguments(
    output_dir=cfg.stage2_output_dir,
    num_train_epochs=cfg.stage2_epochs,
    learning_rate=cfg.stage2_learning_rate,
    per_device_train_batch_size=cfg.stage2_train_batch_size,
    per_device_eval_batch_size=cfg.stage2_eval_batch_size,
    gradient_accumulation_steps=cfg.stage2_grad_accum_steps,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_grounded_score',
    greater_is_better=True,
    lr_scheduler_type='cosine',
    bf16=bf16,
    fp16=fp16,
    gradient_checkpointing=cfg.gradient_checkpointing,
    gradient_checkpointing_kwargs=checkpointing_kwargs(),
    report_to=['wandb'] if cfg.use_wandb else ['none'],
    run_name=f'{cfg.wandb_run_name}-stage2' if cfg.use_wandb else None,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    label_names=['labels'],
)

stage2_model = prepare_peft_model_for_training(stage2_model, 'stage2 model before trainer init')

stage2_callbacks = []
if cfg.early_stopping_patience > 0:
    stage2_callbacks.append(EarlyStoppingCallback(early_stopping_patience=cfg.early_stopping_patience))

stage2_trainer = GroundedQATrainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=stage2_train_tok,
    eval_dataset=stage2_dev_tok,
    generation_eval_dataset=stage2_dev_raw_ds,
    generation_max_examples=cfg.generation_eval_examples,
    data_collator=default_data_collator,
    callbacks=stage2_callbacks,
)

stage2_trainer.train()
final_eval_metrics = stage2_trainer.evaluate()
final_eval_metrics

In [ ]:
def generate_answer(prompt_text: str, max_new_tokens: int = None) -> str:
    model = stage2_trainer.model
    model_device = next(model.parameters()).device
    was_training = model.training
    model.eval()

    model_inputs = tokenizer(
        prompt_text,
        return_tensors='pt',
        add_special_tokens=False,
    ).to(model_device)

    with torch.no_grad():
        generation_kwargs = {
            'max_new_tokens': max_new_tokens or cfg.generation_max_new_tokens,
            'do_sample': False,
            'renormalize_logits': True,
            'pad_token_id': tokenizer.pad_token_id,
            'eos_token_id': tokenizer.eos_token_id,
        }
        if cfg.generation_repetition_penalty and cfg.generation_repetition_penalty != 1.0:
            generation_kwargs['repetition_penalty'] = cfg.generation_repetition_penalty
        if cfg.generation_no_repeat_ngram_size and cfg.generation_no_repeat_ngram_size > 0:
            generation_kwargs['no_repeat_ngram_size'] = cfg.generation_no_repeat_ngram_size
        generated = model.generate(
            **model_inputs,
            **generation_kwargs,
        )

    prompt_len = model_inputs['input_ids'].shape[1]
    prediction = tokenizer.decode(generated[0][prompt_len:], skip_special_tokens=True).strip()

    if was_training:
        model.train()

    return prediction


def print_dev_predictions(rows: List[Dict], limit: int = 10, numeric_only: bool = False) -> None:
    shown = 0
    for row in rows:
        if numeric_only and not (required_numeric_tokens(row.get('answer', ''), row.get('question', '')) or NUMERIC_SENSITIVE_QUESTION_RE.search(row.get('question', '') or '')):
            continue
        pred = generate_answer(row['prompt_text'])
        numeric_match = numeric_preservation_match(pred, row.get('answer', ''), row.get('question', ''))
        strict_numeric_match = numeric_strict_match(pred, row.get('answer', ''), row.get('question', ''))
        unexpected_numbers = unexpected_numeric_tokens(pred, row.get('answer', ''), row.get('question', ''))
        allowed_text = '\n'.join([str(row.get('question', '')), str(row.get('answer', '')), str(row.get('evidence', '')), str(row.get('context', ''))])
        unsupported_hits = unsupported_reference_hits(pred, allowed_text)
        print('QUESTION:', row.get('question', ''))
        print('TARGET:', row.get('answer', ''))
        print('EVIDENCE:', row.get('evidence', ''))
        print('TARGET NUMBERS:', extract_numeric_tokens(row.get('answer', '')))
        print('QUESTION NUMBERS:', extract_numeric_tokens(row.get('question', '')))
        print('REQUIRED TARGET NUMBERS:', required_numeric_tokens(row.get('answer', ''), row.get('question', '')))
        print('REQUIRED TARGET NUMBERS IN CONTEXT:', numeric_context_support_diagnostic(row))
        print('PRED NUMBERS:', extract_numeric_tokens(pred))
        print('NUMERIC RECALL MATCH:', numeric_match)
        print('STRICT NUMERIC MATCH:', strict_numeric_match)
        print('UNEXPECTED PRED NUMBERS:', unexpected_numbers)
        print('UNSUPPORTED REFERENCES:', unsupported_hits)
        print('PRED:', pred)
        print('-' * 80)
        shown += 1
        if shown >= limit:
            break
    if shown == 0:
        print('No matching dev rows found.')


if evaluation_sanity_records:
    print('Evaluation sanity predictions:')
    print_dev_predictions(evaluation_sanity_records, limit=min(cfg.evaluation_sanity_examples, len(evaluation_sanity_records)), numeric_only=False)

print('General dev predictions:')
print_dev_predictions(stage2_dev_records, limit=10, numeric_only=False)

print('Numeric-focused dev predictions:')
print_dev_predictions(stage2_dev_records, limit=10, numeric_only=True)


In [ ]:
from huggingface_hub import HfApi

apply_grounded_generation_defaults(stage2_trainer.model)
clear_sampling_generation_fields(stage2_trainer.model.generation_config)
stage2_trainer.save_model(cfg.stage2_output_dir)
apply_grounded_generation_defaults(stage2_trainer.model)

clean_adapter_tokenizer_dir = './_tmp_adapter_tokenizer'
os.makedirs(clean_adapter_tokenizer_dir, exist_ok=True)
tokenizer.save_pretrained(clean_adapter_tokenizer_dir)
sanitize_llama_tokenizer_export_dir(clean_adapter_tokenizer_dir)
clean_adapter_tokenizer = AutoTokenizer.from_pretrained(clean_adapter_tokenizer_dir, trust_remote_code=True)
clean_adapter_tokenizer.save_pretrained(cfg.stage2_output_dir)

prepare_endpoint_export_dir(cfg.stage2_output_dir)

if cfg.push_to_hub:
    HfApi().upload_folder(
        folder_path=cfg.stage2_output_dir,
        repo_id=cfg.push_repo_id_adapter,
        repo_type='model',
    )
    print(f'Pushed adapter: https://huggingface.co/{cfg.push_repo_id_adapter}')
else:
    print('Skipping Hub push for adapter.')

In [ ]:
from huggingface_hub import HfApi
import shutil

merge_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

if cfg.push_to_hub:
    try:
        merge_model = AutoPeftModelForCausalLM.from_pretrained(
            cfg.push_repo_id_adapter,
            torch_dtype=merge_dtype,
            device_map=None,
            low_cpu_mem_usage=True,
        )
    except TypeError as exc:
        if "unhashable type: 'set'" not in str(exc):
            raise
        print('[WARN] device-map load path failed; retrying fully on CPU for merge.')
        merge_model = AutoPeftModelForCausalLM.from_pretrained(
            cfg.push_repo_id_adapter,
            torch_dtype=torch.float32,
            device_map=None,
            low_cpu_mem_usage=False,
        )

    merge_tokenizer = AutoTokenizer.from_pretrained(cfg.push_repo_id_adapter, trust_remote_code=True)
    tmp_merged_tokenizer_dir = './_tmp_merged_tokenizer'
    os.makedirs(tmp_merged_tokenizer_dir, exist_ok=True)
    merge_tokenizer.save_pretrained(tmp_merged_tokenizer_dir)
    sanitize_llama_tokenizer_export_dir(tmp_merged_tokenizer_dir)
    clean_merged_tokenizer = AutoTokenizer.from_pretrained(tmp_merged_tokenizer_dir, trust_remote_code=True)

    merged_export_dir = './_tmp_merged_export'
    if os.path.exists(merged_export_dir):
        shutil.rmtree(merged_export_dir)
    os.makedirs(merged_export_dir, exist_ok=True)

    merged_model = merge_model.merge_and_unload()
    apply_grounded_generation_defaults(merged_model)
    clear_sampling_generation_fields(merged_model.generation_config)
    merged_model.save_pretrained(merged_export_dir, safe_serialization=True)
    clean_merged_tokenizer.save_pretrained(merged_export_dir)
    prepare_endpoint_export_dir(merged_export_dir)

    HfApi().upload_folder(
        folder_path=merged_export_dir,
        repo_id=cfg.push_repo_id_merged,
        repo_type='model',
    )
    print(f'Pushed merged model: https://huggingface.co/{cfg.push_repo_id_merged}')
else:
    print('Skipping merged model export because push_to_hub=False.')

In [ ]:
import json
import os
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoConfig, AutoTokenizer


if 'read_json_file_strict' not in globals():
    def read_json_file_strict(path: str, label: str = 'JSON file') -> Dict:
        try:
            with open(path, 'r', encoding='utf-8') as file_obj:
                return json.load(file_obj)
        except json.JSONDecodeError as exc:
            with open(path, 'r', encoding='utf-8', errors='replace') as file_obj:
                preview = file_obj.read(500)
            raise ValueError(
                f'{label} is not valid JSON: {path} at line {exc.lineno}, column {exc.colno}. '
                f'First 500 characters: {preview!r}'
            ) from exc


if 'write_strict_json_file' not in globals():
    def write_strict_json_file(path: str, data: Dict) -> None:
        with open(path, 'w', encoding='utf-8') as file_obj:
            json.dump(data, file_obj, ensure_ascii=False, indent=2, allow_nan=False)
            file_obj.write('\n')


if 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token


def build_clean_merged_config_for_repair() -> Dict:
    local_config_path = os.path.join('./_tmp_merged_export', 'config.json')
    if os.path.exists(local_config_path):
        try:
            config_data = read_json_file_strict(local_config_path, 'local merged config.json')
            write_strict_json_file(local_config_path, config_data)
            print('Using local merged export config.json for Hub repair.')
            return config_data
        except Exception as exc:
            print('[WARN] Local merged config.json is not usable for repair:', repr(exc))

    print('Using base model AutoConfig for Hub repair config.json.')
    config_data = AutoConfig.from_pretrained(cfg.base_model, trust_remote_code=True).to_dict()
    config_data['architectures'] = ['LlamaForCausalLM']
    config_data['model_type'] = 'llama'
    config_data['pad_token_id'] = tokenizer.pad_token_id
    config_data['bos_token_id'] = tokenizer.bos_token_id
    config_data['eos_token_id'] = config_data.get('eos_token_id') or tokenizer.eos_token_id
    config_data['max_position_embeddings'] = max(
        int(config_data.get('max_position_embeddings') or 0),
        cfg.model_context_limit,
        cfg.max_seq_len,
    )
    config_data['use_cache'] = True
    config_data.pop('_name_or_path', None)
    return config_data


def repair_merged_hub_config_json() -> None:
    if not cfg.push_to_hub:
        print('Skipping merged Hub config repair because push_to_hub=False.')
        return

    repair_dir = './_tmp_merged_config_repair'
    os.makedirs(repair_dir, exist_ok=True)
    repair_config_path = os.path.join(repair_dir, 'config.json')

    try:
        remote_config_path = hf_hub_download(
            repo_id=cfg.push_repo_id_merged,
            filename='config.json',
            repo_type='model',
            force_download=True,
        )
        read_json_file_strict(remote_config_path, 'remote merged config.json')
        print('Remote merged config.json already parses as valid JSON; rewriting/uploading a normalized copy anyway.')
    except Exception as exc:
        print('[WARN] Remote merged config.json is invalid or unavailable:', repr(exc))

    clean_config = build_clean_merged_config_for_repair()
    write_strict_json_file(repair_config_path, clean_config)
    read_json_file_strict(repair_config_path, 'repair config.json')

    HfApi().upload_file(
        path_or_fileobj=repair_config_path,
        path_in_repo='config.json',
        repo_id=cfg.push_repo_id_merged,
        repo_type='model',
        commit_message='Fix merged model config.json JSON formatting',
    )
    print(f'Repaired merged model config.json: https://huggingface.co/{cfg.push_repo_id_merged}/blob/main/config.json')


repair_merged_hub_config_json()


In [ ]:
import json
import os
import tempfile
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoConfig, AutoTokenizer


def write_json_strict(path: str, data: dict) -> None:
    with open(path, 'w', encoding='utf-8') as file_obj:
        json.dump(data, file_obj, ensure_ascii=False, indent=2, allow_nan=False)
        file_obj.write('\n')


def read_json_strict(path: str, label: str) -> dict:
    with open(path, 'r', encoding='utf-8') as file_obj:
        text = file_obj.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        print(f'First 1200 chars of bad {label}:')
        print(text[:1200])
        raise ValueError(f'{label} is invalid JSON at line {exc.lineno}, column {exc.colno}: {exc.msg}') from exc


def build_forced_clean_llama_config() -> dict:
    tok = globals().get('tokenizer')
    if tok is None:
        tok = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True, trust_remote_code=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

    config_data = AutoConfig.from_pretrained(cfg.base_model, trust_remote_code=True).to_dict()
    config_data['architectures'] = ['LlamaForCausalLM']
    config_data['model_type'] = 'llama'
    config_data['pad_token_id'] = tok.pad_token_id
    config_data['bos_token_id'] = tok.bos_token_id
    config_data['eos_token_id'] = config_data.get('eos_token_id') or tok.eos_token_id
    config_data['max_position_embeddings'] = max(
        int(config_data.get('max_position_embeddings') or 0),
        int(getattr(cfg, 'model_context_limit', 0) or 0),
        int(getattr(cfg, 'max_seq_len', 0) or 0),
    )
    config_data['use_cache'] = True
    config_data.pop('_name_or_path', None)
    return config_data


def force_repair_merged_config_json() -> None:
    repo_id = cfg.push_repo_id_merged
    api = HfApi()

    with tempfile.TemporaryDirectory() as tmp_dir:
        repair_config_path = os.path.join(tmp_dir, 'config.json')
        clean_config = build_forced_clean_llama_config()
        write_json_strict(repair_config_path, clean_config)
        read_json_strict(repair_config_path, 'local forced repair config.json')

        with open(repair_config_path, 'r', encoding='utf-8') as file_obj:
            local_text = file_obj.read()
        print('Local forced config length:', len(local_text))
        print('Local forced config tail:', repr(local_text[-120:]))

        commit_info = api.upload_file(
            path_or_fileobj=repair_config_path,
            path_in_repo='config.json',
            repo_id=repo_id,
            repo_type='model',
            commit_message='Force replace malformed config.json',
        )
        print('Uploaded forced config commit:', commit_info)

    # Verify after upload from the latest main revision. This must parse before redeploying.
    info = api.model_info(repo_id=repo_id, repo_type='model')
    print('Latest repo sha after upload:', info.sha)

    remote_config_path = hf_hub_download(
        repo_id=repo_id,
        filename='config.json',
        repo_type='model',
        revision=info.sha,
        force_download=True,
    )
    remote_config = read_json_strict(remote_config_path, 'remote forced repair config.json')
    print('Remote config parses OK. Keys:', sorted(remote_config.keys())[:20])
    print('Remote config file:', remote_config_path)
    print(f'Now restart/redeploy the endpoint so it pulls repo revision {info.sha}.')


force_repair_merged_config_json()


In [ ]:
if cfg.push_to_hub and cfg.export_4bit:
    quant_tokenizer = AutoTokenizer.from_pretrained(cfg.push_repo_id_merged)
    quant_bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )

    quant_model = AutoModelForCausalLM.from_pretrained(
        cfg.push_repo_id_merged,
        device_map='auto',
        quantization_config=quant_bnb,
    )
    quant_model.push_to_hub(cfg.push_repo_id_merged_4bit)
    quant_tokenizer.push_to_hub(cfg.push_repo_id_merged_4bit)
    print(f'Pushed 4-bit model: https://huggingface.co/{cfg.push_repo_id_merged_4bit}')
else:
    print('Skipping 4-bit export. Evaluate adapter and merged fp16/bf16 first.')

## Notes

- Stage 1 is intentionally light and mixed with retention conversations to reduce instruct drift.
- Stage 2 uses assistant-only loss on grounded QA records from your JSONL files.
- Stage 2 checkpoint selection uses a simple grounded QA score on the dev set, not plain LM loss.
- Evaluate the base model, the adapter-loaded model, and the merged fp16 or bf16 model before judging the effect of quantization.
- If faithfulness is still fragile, the next step should be adding explicit abstain-heavy examples and comparing against a RAG baseline.

### 2026-05-21 update

- Updated Stage 2 SFT to better preserve baseline Llama 3.1 long-prompt behavior by increasing `max_seq_len` to 4096 and lowering the Stage 2 learning rate to `1e-6`.
- Rebuilt Stage 2 examples in the production endpoint format: `Instruction`, `Context`, and `Question`, so training matches inference more closely.
- Strengthened generation/export defaults with `repetition_penalty=1.15`, `no_repeat_ngram_size=4`, and `renormalize_logits=True` to reduce repeated-token degeneration.
- Preserved Llama 3.1 long-context config during export by using `model_context_limit=131072` instead of forcing `max_position_embeddings` down to 8192.
- Updated the endpoint handler defaults and structured payload support so production should call the model with `inputs.system`, `inputs.context`, and `inputs.question`.

### 2026-05-21 numeric fidelity update

- Shortened default generation to `generation_max_new_tokens=64` so factual extraction answers stay concise.
- Added numeric-faithfulness instructions to the production SFT prompt and endpoint fallback instruction: preserve numbers, ranges, units, p-values, confidence intervals, dates, distances, comparison groups, and categories exactly.
- Added numeric extraction helpers and `eval_numeric_match_rate` / `eval_numeric_rows` to the Stage 2 generation metrics.
- Added a post-training inspection cell after `final_eval_metrics` that prints both general dev predictions and numeric-focused dev predictions, including target/predicted numeric tokens and a numeric match flag.

### 2026-05-21 extraction evaluation update

- Improved numeric parsing so comma-separated values such as `1,722` are treated as one number and normalized to `1722`.
- Numeric preservation checks now ignore numbers already present in the question, reducing false failures from comparison groups like `<5 GCSEs`.
- Added unsupported-reference detection for generated mentions like `Table 2`, `Fig 3`, `page 12`, or `row 4` when those references are not present in the allowed question/answer/evidence/context text.
- Tightened production and endpoint instructions to answer only the requested value, relationship, or finding and avoid adjacent table values or extra categories unless asked.
- Updated the post-training inspection cell to print question numbers, required target numbers, and unsupported reference hits.

### 2026-05-21 numeric oversampling update

- Added `numeric_sensitive_oversample_copies=2` and oversampling for Stage 2 training rows whose answers contain numeric tokens or whose questions ask for numeric/coded values such as odds ratios, percentages, cutoffs, coefficients, sample sizes, means, BMI values, dates, or codes.
- Oversampling is applied only to the training split; dev records remain unmodified for cleaner evaluation.


### 2026-05-21 response-rule and checkpoint update

- Added a short `Response rule` immediately after each question in Stage 2 training prompts so the extraction constraint sits at the end of the prompt, close to the assistant generation point.
- Updated the endpoint handler to append the same response rule for structured payloads and for message-list payloads that contain `Question:` or `Researcher question:`.
- Updated `eval_grounded_score` so checkpoint selection now rewards numeric preservation and supported references, rather than relying only on answer F1, evidence hit rate, and abstain matching.
- Broadened numeric-sensitive detection for counts, values, dates, classified/coded variables, units, and proportions; the numeric-focused debug print now uses required target numbers plus numeric-sensitive question wording.


### 2026-05-21 dependency repair update

- Replaced the broad `transformers>=4.44.0` setup install with a one-time forced reinstall of a coherent Hugging Face stack pinned around `transformers==4.51.3` and `tokenizers==0.21.1`.
- The setup cell now restarts the Colab runtime after the clean reinstall; this is required because the previous `ImportError: cannot import name interval` comes from a mixed in-memory/on-disk `transformers` installation.
- Added an early Llama configuration import check in the imports cell so package inconsistency is caught before model loading or training starts.
- Exported endpoint `requirements.txt` now pins the same core HF package versions, and the merge cell uses `torch_dtype` for compatibility with the pinned stack.


### 2026-05-21 torchvision conflict update

- Updated the setup cell to uninstall optional text-only conflict packages `torchvision` and `torchao` even when the Hugging Face dependency sentinel already exists.
- This addresses Colab runtime failures such as `operator torchvision::nms does not exist`, which come from a `torchvision` build that does not match the active `torch` build.
- Added `TRANSFORMERS_NO_TORCHVISION=1` before importing `transformers`; uninstalling `torchvision` is still the primary fix for this text-only notebook.


### 2026-05-21 bitsandbytes CUDA update

- Pinned `bitsandbytes==0.46.1` instead of `bitsandbytes>=0.46.1`; the open-ended spec can pull newer CUDA 13 builds that fail on Colab with `libnvJitLink.so.13` errors.
- Changed the dependency sentinel name so the setup cell forces a fresh reinstall on the next run rather than trusting the previous setup marker.
- Added a `bitsandbytes` version print and a small import sanity check before model loading, so CUDA/native-library issues fail with a clearer message before QLoRA setup starts.


### 2026-05-21 CUDA 13 bitsandbytes fallback update

- Updated the setup cell from `bitsandbytes==0.46.1` to `bitsandbytes==0.49.2`, because the runtime now reports PyTorch CUDA 13.0 and older BNB wheels do not ship `libbitsandbytes_cuda130.so`.
- Added CUDA 13 runtime support wheels `nvidia-nvjitlink==13.0.88` and `nvidia-cuda-runtime==13.0.96`, plus automatic `LD_LIBRARY_PATH` setup for Python-installed NVIDIA library directories.
- Added `allow_4bit_fallback=True` and a small NF4 CUDA probe before model loading. If BNB still cannot run in the Colab image, the notebook now falls back to non-4-bit LoRA loading instead of failing inside checkpoint loading.


### 2026-05-21 PEFT gradient checkpointing update

- Added explicit `label_names=['labels']` and `remove_unused_columns=False` to both Stage 1 and Stage 2 `TrainingArguments`, matching the Transformers `Trainer` requirement when model wrappers hide the base model forward signature.
- Added `gradient_checkpointing_use_reentrant=False` and passed `gradient_checkpointing_kwargs` through PEFT preparation and both Trainer argument blocks; this avoids the common frozen-input checkpointing path that produces `None of the inputs have requires_grad=True`.
- Added `enable_input_grads_for_checkpointing()` and `prepare_peft_model_for_training()` so embedding outputs require gradients when checkpointing is active, `use_cache=False` is restored before training, and adapter trainability is checked before Trainer starts.
- Added trainable-parameter preflight prints for the initial PEFT model and the Stage 2 model so frozen adapters are caught before a backward pass.


### 2026-05-21 copy-friendly generation update

- Changed default extraction decoding to `repetition_penalty=1.0` and `no_repeat_ngram_size=0`; the previous anti-loop settings can penalize copying numbers and phrases from the prompt, which is harmful for grounded numeric QA.
- Updated generation calls so penalty kwargs are omitted when disabled, keeping local eval, sanity checks, and endpoint defaults aligned.
- Expanded the pre-training sanity check to print required target numbers, predicted numbers, numeric match, unsupported references, and a reminder that the check is before Stage 2 SFT.


### 2026-05-22 strict numeric evaluation update

- Added parsing for middle-dot decimals such as `−0·20` and for common written percentages such as `seventeen point two percent`, normalizing them to numeric tokens for evaluation.
- Added `unexpected_numeric_tokens()` and `numeric_strict_match()` so predictions that include the correct number plus extra unrequested numbers are flagged instead of passing numeric recall.
- Updated generation metrics with strict numeric match, unexpected numeric rate, and numeric precision rate; checkpoint scoring now uses strict numeric match and numeric precision.
- Updated sanity and post-training prediction prints to show numeric recall match, strict numeric match, and unexpected predicted numbers.


### 2026-05-22 strict numeric metric loop fix

- Fixed the `GroundedQATrainer` generation metric loop so it actually calls `numeric_strict_match()` and `unexpected_numeric_tokens()` while evaluating predictions.
- The previous code defined the strict numeric metrics but did not increment `strict_numeric_rows`, causing `eval_strict_numeric_rows=0` and making checkpoint scoring fall back to answer F1 for the numeric component.


### 2026-05-22 deterministic generation config export fix

- Added `clear_sampling_generation_fields()` to remove stale sampling-only fields such as `temperature`, `top_p`, and `top_k` from inherited model `generation_config` objects when `do_sample=False`.
- Applied deterministic generation defaults immediately before adapter and merged-model saves so newer Transformers versions do not reject invalid generation configs during `save_pretrained()`.
- The exported `generation_config.json` remains deterministic and copy-friendly, with sampling fields omitted rather than set inconsistently.

### 2026-05-22 targeted extraction augmentation update

- Added train-only targeted Stage 2 augmentation for numeric/table extraction failures observed in dev inspection.
- The augmentation repeats inferred table headers beside evidence rows, adds examples where the requested answer is the second or later value in a row, and teaches the model to return only requested values rather than adjacent categories, p-values, confidence intervals, or unrelated row values.
- Added synthetic extraction drill rows for second/third table values, category-code-only answers, multi-value comparison-group answers, BMI cutoff extraction, percentage-plus-count answers, and mean extraction without adjacent p-values.
- Dev records remain unaugmented so evaluation still measures generalization rather than memorization of augmented examples.

### 2026-05-22 hard negative extraction update

- Added harder synthetic Stage 2 drills for the remaining observed failures: selecting the second `More than RWG` odds-ratio column instead of the first `Less than RWG` odds-ratio column, returning both percentages in comparison wording, answering binary variable codes without listing source-category codes, and summarising relationship questions without coefficient/CI spillover.
- Split response-rule handling so relationship/finding questions with numeric timepoints ask for a directional prose answer, while direct numeric questions still ask for exact value extraction from headers/labels.
- Mirrored the same relationship-aware response rule in the endpoint handler so production prompts match the new SFT format.

### 2026-05-22 hosted base-model sanity check update

- Added an optional hosted Hugging Face API sanity check for `meta-llama/Llama-3.1-8B-Instruct` on the same Stage 2 dev examples used by the local empty-adapter sanity check.
- The check uses `huggingface_hub.InferenceClient(...).chat.completions.create(...)` with the same rendered user messages, prints the same numeric/strict/reference diagnostics, and catches API errors so training does not fail if the hosted model is unavailable.
- This gives a three-way comparison: hosted base instruct model, local base model with empty LoRA adapter, and the final trained adapter.

### 2026-05-22 Stage 2 row schema fallback update

- Made Stage 2 row construction tolerant to common JSONL aliases for answers, questions, and contexts, including `target`, `gold_answer`, `query`, `retrieved_context`, `chunk_text`, `passage`, and related fields.
- If a row has no full context but does have `evidence`, the builder now uses `evidence` as last-resort context instead of dropping the row.
- Added skipped-row diagnostics that print example keys and previews for missing answer/question/context fields, making split-file schema problems easier to identify.

### 2026-05-22 self-contained merged config repair update

- Made the merged Hub `config.json` repair cell self-contained by defining `read_json_file_strict()` and `write_strict_json_file()` locally if the export-helper cell has not been run in the current runtime.
- The repair cell now also reloads the base tokenizer if needed, so it can run after the config/login cells without requiring the full training/export helper state.

### 2026-05-22 forced merged config replacement update

- Added a stronger forced repair cell for deployment failures where `/repository/config.json` raises `JSONDecodeError: Extra data`.
- The forced repair cell never reuses the local merged config; it builds a clean config from the base Llama config, uploads it as `config.json`, downloads the exact latest Hub revision, validates that remote JSON parses, and prints the repo SHA that the endpoint should redeploy against.

